# V11 suite v13 — thêm dữ liệu (category, tên entity, lịch sử dài), multi-interest, baseline rút gọn còn Fastformer (1 lần Run All, ước ~7,5–8 giờ; `V11_FOCUS=1` ước ~2–2,5 giờ)

Mọi biến thể được đo trên **đúng giao thức V10** (validation theo ngày, `MINDsmall_dev` chỉ dùng 1 lần làm test), có **CI 95% ghép cặp** so với control,
tách **tin mới (cold) / tin cũ (warm)**, hiệu chỉnh **Bonferroni**, và chọn đại diện từng hướng **trên validation** (không chọn trên test).
Toàn bộ stage cũ của v7 (H1–H10, quét encoder) giữ nguyên, và `E0` vẫn là `title. abstract` nên `frozen` vẫn phải khớp V10 (AUC 0.6241 / nDCG@10 0.3909).

**Vì sao có bản v13.** Ở v7, vector đóng băng + pooling kiểu late-interaction (`pool_lse`) ngang baseline huấn luyện đúng cách, còn các kỹ thuật phức tạp hơn (RAG, GraphRAG, Semantic ID, LoRA, LLM judge) cho ≤ 0,005 hoặc âm.
v13 hỏi hai câu còn bỏ ngỏ, **từng kỹ thuật riêng lẻ** (chưa ghép thành mô hình lớn): (1) phần dữ liệu chưa từng đưa vào encoder có giúp không? (2) nhiều interest có hơn một vector người dùng không?

| | Nội dung | Stage |
|---|---|---|
| **F** | **Thêm dữ liệu.** *MIND-small không có body của bài* (`news.tsv` chỉ có category, subcategory, title, abstract, URL, entity); abstract đã nằm trong `E0` từ v1. Chưa bao giờ đưa vào encoder: **category/subcategory**, **tên entity**, **hơn 50 click gần nhất của người đọc**. Các hàng: `txt_t` (chỉ title ⇒ phần abstract đóng góp bao nhiêu), `txt_cta` (+ category), `txt_ctae` (+ tên entity), `txt_views` (title / abstract / category mã hoá riêng rồi ghép có trọng số, kiểu NAML nhưng vector đóng băng), mỗi biến thể với mean-pool (so với `frozen`) và late-interaction `*_lse` (so với `pool_lse`); `hist100_*`, `hist200_*` (đọc 100 / 200 click thay vì 50). Biến thể thắng trên validation thành view `text` cho các reader học được. | `moredata` |
| **H11** | **Multi-interest.** Không huấn luyện: `pool_km{2,3,5}` (gom click của người đọc thành K interest bằng k-means; K=1 chính là mean-pool, K ≥ số click chính là `pool_lse`, nên nằm giữa hai đầu đã đo). Học được: `um_mi1_*` (một vector, **đối chứng**), `um_mi4_*` (4 interest, điểm = soft-max qua interest), `um_mi4d_*` (thêm phạt "bất đồng" giữa các interest (bình phương cosine)); log in độ giống nhau giữa các interest để biết có bị "sập" về một vector không. | `pooling`, `usermodel` |
| **R** | **Baseline huấn luyện chỉ còn Fastformer** theo công thức chuẩn: `fastformer_ref` (**trung bình 2 seed**, + `+pop`) và `fastformer_ref_refit` (huấn luyện lại trên train_core + validation với số epoch trung bình đã chọn, số so được với bài báo). NRMS / NAML / thang ablation **tắt mặc định** (`V11_REF_ALL=1` bật lại); số đo v7 của chúng ở `RESULTS_v7.md`. | `ref_ff`, `ref_ff_refit` |
| H1–H10 | như v7 (history-as-text, LoRA/DoRA, RAG, GraphRAG, tower LoRA, KAR, Semantic ID, pooling, LLM judge, user model) | như v7 |

**Lưu ý khi so với baseline.** Fastformer ở đây chỉ đọc **title** (cài đặt của `refmodels.py`, cùng đầu vào với NRMS; chưa đối chiếu với bài báo gốc). Vì NAML, mô hình duy nhất ở v7 dùng cả abstract và category, đã bị bỏ, phép so cùng đầu vào là `txt_t` / `txt_t_lse` (vector chỉ title) với `fastformer_ref`; các hàng dùng abstract/category thì nhìn thấy nhiều dữ liệu hơn baseline.

**Cách chạy:** (1) Settings → Accelerator **GPU (T4/P100)**, Internet **On**. (2) *Add Input* → dataset MIND có **cả** `MINDsmall_train` và `MINDsmall_dev`
(ví dụ `thinhhuynh3108/mindsmall`). (3) **Run All**. Lần đầu nên đặt `QUICK = True` để bắt lỗi môi trường rồi chạy đầy đủ; chỉ thử BGE: `V11_RUN_B=0`; bỏ phần baseline: `V11_REF=0`;
**chỉ phần mới** (moredata + multi-interest + Fastformer, bỏ run B và các stage cũ): `V11_FOCUS=1`.
GloVe-300 tự tải từ Hugging Face (~480 MB); nếu bạn đã Add Input một dataset GloVe hoặc muốn chỉ đường dẫn: `V11_GLOVE_PATH=/kaggle/input/.../glove.6B.300d.txt`. Nếu mọi nguồn GloVe đều hỏng,
baseline vẫn chạy với vector từ ngẫu nhiên và các hàng được gắn nhãn `_noglove` (đừng so chúng với số công bố).

**Thời gian.** *Đã đo* ở v7 trên Tesla T4 (cả notebook 7,52 giờ): A-core 18,5 phút · baseline (8 mô hình) 114 phút, trong đó NRMS 16, NAML 18, **Fastformer 29** (chậm nhất), refit NRMS 15 · run B (Qwen3) 3,6 giờ, trong đó tower LoRA 2,5 giờ · A-phần còn lại 1,5 giờ (LoRA/DoRA 62 phút).
*Ước (chưa đo)* cho phần mới: `moredata` ≈ 4 phút (BGE) + ≈ 14 phút (Qwen3, 2 biến thể text); `pool_km` ≈ 3 phút mỗi run; user model thêm ≈ 25 phút (BGE) + ≈ 12 phút (Qwen3) do thêm view `text`, reader multi-interest và 3 seed trên `frozen`/`text`;
Fastformer 2 seed + refit ≈ 85 phút thay cho 114 phút. Tổng ≈ 7,5–8 giờ; `V11_FOCUS=1` ≈ 2–2,5 giờ. Thứ tự chạy: **A-core → baseline → B (Qwen3) → A-phần còn lại**.
`BUDGET_H` (mặc định 9.5) là ngân sách đồng hồ chung: mọi vòng lặp dài tự dừng, stage nào không còn thời gian thì bị bỏ qua (kết quả luôn được ghi sau từng stage; chạy lại cell sẽ **resume** các stage đã xong).

**Kết quả:** `results.md` của từng run, **DECISION SUMMARY** (đại diện từng hướng, chọn theo validation, Bonferroni; baseline *không* tham gia như một "hướng"), **HEAD-TO-HEAD**, **LEADERBOARD** chung (có cả baseline) + `forest.png`,
`direct_comparisons.md`, `test_arrays.npz` (metric theo từng impression), và **một file gộp `SUMMARY_v13.md`** — gửi file đó (cùng log nếu có lỗi) để đọc kết quả.

Mọi biến thể zero-shot và `fastformer_ref` đều có **bản sinh đôi `+pop`** (gộp z-score với popularity online của V10, công thức đúng của `bge_zs_pop`; với các hàng `*_lse` thì đối chứng là `pool_lse+pop`).
**Lưu ý quan trọng:** `pop` đếm click của các impression *trước đó trong chính ngày test* (phản hồi streaming) nên các hàng `+pop` **không so được** với số offline trong bài báo; bảng *content only* mới so được.

Dòng đầu của kết quả là **reproduction check**: `frozen` phải khớp V10 `bge_zeroshot` (AUC 0.6241, nDCG@10 0.3909). Nếu in `MISMATCH` thì dừng và sửa trước khi tin bất kỳ dòng nào. Tương tự, `popularity` phải khớp (AUC 0.6533, nDCG@10 0.4026) và `frozen+pop` khớp `bge_zs_pop` (0.6769 / 0.4330).

### 0. Phụ thuộc và GPU

In [ ]:
%pip install -q "transformers>=4.51" peft sentence-transformers
import subprocess
print(subprocess.run("nvidia-smi -L", shell=True, capture_output=True, text=True).stdout or "no GPU visible")


### 1. Modules (ghi ra thư mục làm việc)

In [ ]:
%%writefile data.py
"""Self-contained MIND-small data reader with chronological validation split."""
from __future__ import annotations

import math
import random
from collections import Counter
from itertools import groupby
from dataclasses import dataclass, field
from datetime import datetime

import numpy as np
import torch

PAD = 0


@dataclass
class Batch:
    hist_title: torch.Tensor
    hist_cat: torch.Tensor
    hist_idx: torch.Tensor
    hist_mask: torch.Tensor
    cand_title: torch.Tensor
    cand_cat: torch.Tensor
    cand_idx: torch.Tensor
    cand_mask: torch.Tensor
    user_idx: torch.Tensor
    labels: torch.Tensor
    cand_pop: torch.Tensor

    def to(self, device):
        for name in self.__dataclass_fields__:
            setattr(self, name, getattr(self, name).to(device))
        return self


@dataclass
class MindData:
    news_title: np.ndarray
    news_cat: np.ndarray
    n_news: int
    n_users: int
    vocab_size: int
    n_cat: int
    title_len: int
    max_hist: int
    llm_emb: np.ndarray
    train_core: list = field(default_factory=list)
    validation: list = field(default_factory=list)
    test: list = field(default_factory=list)
    train_core_impressions: int = 0
    validation_impressions: int = 0
    test_impressions: int = 0
    train_days: list[str] = field(default_factory=list)
    validation_days: list[str] = field(default_factory=list)
    test_days: list[str] = field(default_factory=list)
    split_info: dict = field(default_factory=dict)
    _edges: list = field(default_factory=list)

    def _pad_hist(self, hist):
        hist = list(hist)[-self.max_hist:]
        mask = [True] * len(hist)
        while len(hist) < self.max_hist:
            hist.append(PAD)
            mask.append(False)
        return hist, mask

    def _gather(self, idxs):
        return self.news_title[idxs], self.news_cat[idxs]

    def collate_train(self, rows):
        H, L = self.max_hist, self.title_len
        C, B = 1 + len(rows[0][3]), len(rows)
        ht = np.zeros((B, H, L), np.int64)
        hc = np.zeros((B, H), np.int64)
        hi = np.zeros((B, H), np.int64)
        hm = np.zeros((B, H), bool)
        ct = np.zeros((B, C, L), np.int64)
        cc = np.zeros((B, C), np.int64)
        ci = np.zeros((B, C), np.int64)
        ui = np.zeros(B, np.int64)
        for b, (user, hist, pos, negs) in enumerate(rows):
            h, m = self._pad_hist(hist)
            hi[b], hm[b] = h, m
            ht[b], hc[b] = self._gather(h)
            cands = [pos] + list(negs)
            ci[b], ct[b], cc[b] = cands, *self._gather(cands)
            ui[b] = user
        return _to_batch(ht, hc, hi, hm, ct, cc, ci,
                         np.ones((B, C), bool), ui,
                         np.pad(np.ones((B, 1), np.float32), ((0, 0), (0, C - 1))),
                         np.zeros((B, C), np.float32))  # ponytail: no popularity for train rows, add when a trained model consumes cand_pop

    def collate_eval(self, rows):
        H, L = self.max_hist, self.title_len
        C, B = max(len(_eval_parts(r)[3]) for r in rows), len(rows)
        ht = np.zeros((B, H, L), np.int64)
        hc = np.zeros((B, H), np.int64)
        hi = np.zeros((B, H), np.int64)
        hm = np.zeros((B, H), bool)
        ct = np.zeros((B, C, L), np.int64)
        cc = np.zeros((B, C), np.int64)
        ci = np.zeros((B, C), np.int64)
        cm = np.zeros((B, C), bool)
        labels = np.zeros((B, C), np.float32)
        cp = np.zeros((B, C), np.float32)
        ui = np.zeros(B, np.int64)
        for b, row in enumerate(rows):
            _, user, hist, cands, labs, pop = _eval_parts(row)
            h, m = self._pad_hist(hist)
            hi[b], hm[b] = h, m
            ht[b], hc[b] = self._gather(h)
            n = len(cands)
            ci[b, :n], cm[b, :n], labels[b, :n], cp[b, :n] = cands, True, labs, pop
            if n:
                ct[b, :n], cc[b, :n] = self._gather(cands)
            ui[b] = user
        return _to_batch(ht, hc, hi, hm, ct, cc, ci, cm, ui, labels, cp)

    def build_adjacency(self):
        U, N = self.n_users, self.n_news
        rows, cols = [], []
        for u, n in self._edges:
            rows += [u, U + n]
            cols += [U + n, u]
        if not rows:
            rows, cols = [0], [0]
        idx = np.array([rows, cols], dtype=np.int64)
        vals = np.ones(idx.shape[1], np.float32)
        size = U + N
        deg = np.zeros(size, np.float64)
        np.add.at(deg, idx[0], vals)
        inv = np.zeros(size, np.float64)
        inv[deg > 0] = deg[deg > 0] ** -0.5
        norm = (inv[idx[0]] * inv[idx[1]]).astype(np.float32)
        return torch.sparse_coo_tensor(torch.from_numpy(idx), torch.from_numpy(norm), (size, size)).coalesce()

    @classmethod
    def from_mind(cls, train_dir, dev_dir, title_len=20, max_hist=50,
                  n_neg=4, min_word_freq=3, llm_dim=64,
                  llm_embeddings=None, dev_ratio=0.1, seed=0):
        import os
        from collections import Counter

        rng = np.random.default_rng(seed)

        def read_news(path):
            out = {}
            with open(path, encoding="utf-8") as f:
                for line in f:
                    p = line.rstrip("\n").split("\t")
                    if len(p) >= 5:
                        out[p[0]] = (p[1], p[3], p[4])
            return out

        news_rows = read_news(os.path.join(train_dir, "news.tsv"))
        if dev_dir and os.path.exists(os.path.join(dev_dir, "news.tsv")):
            news_rows.update(read_news(os.path.join(dev_dir, "news.tsv")))

        wf = Counter()
        for _, title, abstract in news_rows.values():
            wf.update((title + " " + abstract).lower().split())
        vocab = {"<pad>": PAD}
        for word, count in wf.items():
            if count >= min_word_freq:
                vocab[word] = len(vocab)
        cats = sorted({cat for cat, _, _ in news_rows.values()})
        cat_map = {cat: i + 1 for i, cat in enumerate(cats)}
        nid_map = {"<pad>": PAD}
        for nid in news_rows:
            nid_map[nid] = len(nid_map)
        n_news = len(nid_map)
        news_title = np.zeros((n_news, title_len), np.int64)
        news_cat = np.zeros(n_news, np.int64)
        for nid, (cat, title, abstract) in news_rows.items():
            words = (title + " " + abstract).lower().split()[:title_len]
            gi = nid_map[nid]
            news_title[gi, :len(words)] = [vocab.get(w, PAD) for w in words]
            news_cat[gi] = cat_map[cat]

        if llm_embeddings is None:
            llm_emb = np.zeros((n_news, llm_dim), np.float32)
        else:
            if not llm_embeddings:
                raise ValueError("empty BGE embedding map")
            dims = {np.asarray(v).reshape(-1).size for v in llm_embeddings.values()}
            if len(dims) != 1:
                raise ValueError("inconsistent embedding dimensions")
            dim = dims.pop()
            llm_emb = np.zeros((n_news, dim), np.float32)
            for nid, gi in nid_map.items():
                if nid != "<pad>" and nid in llm_embeddings:
                    llm_emb[gi] = np.asarray(llm_embeddings[nid], np.float32).reshape(-1)
            coverage = float((np.linalg.norm(llm_emb[1:], axis=1) > 0).mean())
            if coverage < 0.999 or not np.isfinite(llm_emb).all():
                raise ValueError(f"invalid BGE coverage={coverage:.4%}")
            print(f"  embedding coverage: {coverage:.3%} (PAD excluded)")
        llm_emb = (llm_emb / (np.linalg.norm(llm_emb, axis=1, keepdims=True) + 1e-8)).astype(np.float32)

        uid_map = {}
        def uidx(uid):
            if uid not in uid_map:
                uid_map[uid] = len(uid_map)
            return uid_map[uid]

        def time_of(value):
            value = value.strip()
            for fmt in ("%m/%d/%Y %H:%M:%S", "%m/%d/%Y %I:%M:%S %p", "%Y-%m-%d %H:%M:%S"):
                try:
                    return datetime.strptime(value, fmt)
                except ValueError:
                    pass
            raise ValueError(f"unparseable impression time: {value!r}")

        def read_behaviors(path):
            out = []
            with open(path, encoding="utf-8") as f:
                for line in f:
                    p = line.rstrip("\n").split("\t")
                    if len(p) < 5:
                        continue
                    impr_id, user_raw, timestamp, history_raw, impressions = p[:5]
                    hist = [nid_map[n] for n in history_raw.split() if n in nid_map] if history_raw else []
                    cands, labs = [], []
                    for item in impressions.split():
                        nid, lab = item.rsplit("-", 1)
                        if nid in nid_map:
                            cands.append(nid_map[nid])
                            labs.append(int(lab))
                    t = time_of(timestamp)
                    out.append((str(impr_id), uidx(user_raw), hist, cands, labs, t.date().isoformat(), t))
            return out

        train_raw = read_behaviors(os.path.join(train_dir, "behaviors.tsv"))
        if not dev_dir or os.path.abspath(dev_dir) == os.path.abspath(train_dir):
            raise ValueError("V10 requires separate MINDsmall_train and MINDsmall_dev")
        test_raw = read_behaviors(os.path.join(dev_dir, "behaviors.tsv"))
        if not train_raw or not test_raw:
            raise ValueError("MIND behaviors files are empty")
        pop = _online_log_ctr(train_raw + test_raw)
        train_raw = [r + (p,) for r, p in zip(train_raw, pop)]
        test_raw = [r + (p,) for r, p in zip(test_raw, pop[len(train_raw):])]

        days = sorted({row[5] for row in train_raw})
        target = max(1, int(np.ceil(len(train_raw) * dev_ratio)))
        val_days, count = [], 0
        for day in reversed(days):
            val_days.append(day)
            count += sum(row[5] == day for row in train_raw)
            if count >= target:
                break
        val_days = sorted(val_days)
        train_days = [d for d in days if d not in set(val_days)]
        if not train_days:
            raise ValueError("day split leaves no train_core day")
        train_raw_core = [r for r in train_raw if r[5] not in set(val_days)]
        val_raw = [r for r in train_raw if r[5] in set(val_days)]
        train_core_beh = [(u, h, c, labs) for _, u, h, c, labs, *_ in train_raw_core]
        validation = [(i, u, h, c, labs, p) for i, u, h, c, labs, _, _, p in val_raw]
        test = [(i, u, h, c, labs, p) for i, u, h, c, labs, _, _, p in test_raw]
        if max(r[5] for r in train_raw_core) >= min(r[5] for r in val_raw):
            raise AssertionError("chronological train/validation ordering violated")

        train_core, edges = [], []
        for user, hist, cands, labs in train_core_beh:
            edges.extend((user, n) for n in hist)
            pos = [c for c, lab in zip(cands, labs) if lab == 1]
            neg = [c for c, lab in zip(cands, labs) if lab == 0]
            if not pos or not neg:
                continue
            for p in pos:
                sampled = list(rng.choice(neg, size=n_neg, replace=len(neg) < n_neg))
                train_core.append((user, hist, p, sampled))
                edges.append((user, p))
        split_info = {
            "train_days": train_days,
            "validation_days": val_days,
            "test_days": sorted({r[5] for r in test_raw}),
            "train_day_min": min(train_days), "train_day_max": max(train_days),
            "validation_day_min": min(val_days), "validation_day_max": max(val_days),
            "target_validation_ratio": float(dev_ratio),
            "actual_validation_ratio": len(validation) / len(train_raw),
            "raw_train_impressions": len(train_raw),
            "train_core_impressions": len(train_core_beh),
            "validation_impressions": len(validation),
            "test_impressions": len(test),
        }
        print(f"  chronological split: {len(train_core_beh):,} train_core / "
              f"{len(validation):,} validation / {len(test):,} official test impressions")
        return cls(news_title, news_cat, n_news, len(uid_map), len(vocab), len(cat_map) + 1,
                   title_len, max_hist, llm_emb, train_core, validation, test,
                   len(train_core_beh), len(validation), len(test), train_days, val_days,
                   split_info["test_days"], split_info, edges)


def _online_log_ctr(rows):
    """Smoothed log-CTR of every candidate, counted only from impressions strictly earlier in time.

    Impressions sharing a timestamp are scored before any of them is counted, so no label leaks.
    """
    clicks, shown, out = Counter(), Counter(), [None] * len(rows)
    order = sorted(range(len(rows)), key=lambda k: rows[k][6])
    for _, group in groupby(order, key=lambda k: rows[k][6]):
        group = list(group)
        for k in group:
            out[k] = [math.log((clicks[n] + 1) / (shown[n] + 20)) for n in rows[k][3]]
        for k in group:
            for n, y in zip(rows[k][3], rows[k][4]):
                shown[n] += 1
                clicks[n] += y
    return out


def _eval_parts(row):
    if len(row) == 6:
        return row
    user, hist, cands, labs = row
    return "", user, hist, cands, labs, [0.0] * len(cands)


def _to_batch(*arrays):
    return Batch(*(torch.from_numpy(x) for x in arrays))


def iter_batches(rows, batch_size, collate, shuffle=False, seed=0):
    order = list(range(len(rows)))
    if shuffle:
        random.Random(seed).shuffle(order)
    for start in range(0, len(order), batch_size):
        yield collate([rows[i] for i in order[start:start + batch_size]])


In [ ]:
%%writefile metrics.py
"""Ranking metrics for news recommendation, computed per-impression.

Pure NumPy so there is no dependency beyond numpy. Definitions match the
Microsoft Recommenders / MIND leaderboard conventions (MRR, nDCG@k) plus a
tie-aware AUC.
"""
from __future__ import annotations

import numpy as np


def auc_score(labels: np.ndarray, scores: np.ndarray) -> float | None:
    """Area under ROC curve for one impression (tie-aware, rank based).

    Returns None if the impression has no positive or no negative (undefined),
    so the caller can skip it in the mean.
    """
    labels = np.asarray(labels)
    scores = np.asarray(scores, dtype=np.float64)
    n_pos = int(labels.sum())
    n_neg = int(len(labels) - n_pos)
    if n_pos == 0 or n_neg == 0:
        return None
    # Average ranks (1..n), ties share the mean rank -> Mann-Whitney U form.
    _, inv, counts = np.unique(scores, return_inverse=True, return_counts=True)
    cum = np.cumsum(counts)
    start = cum - counts
    avg = (start + cum + 1) / 2.0  # average 1-based rank per distinct value
    ranks = avg[inv]
    sum_ranks_pos = ranks[labels == 1].sum()
    return (sum_ranks_pos - n_pos * (n_pos + 1) / 2.0) / (n_pos * n_neg)


def mrr_score(labels: np.ndarray, scores: np.ndarray) -> float:
    order = np.argsort(scores)[::-1]
    y = np.asarray(labels)[order]
    rr = y / (np.arange(len(y)) + 1)
    denom = y.sum()
    return float(rr.sum() / denom) if denom > 0 else 0.0


def _dcg(labels: np.ndarray, scores: np.ndarray, k: int) -> float:
    order = np.argsort(scores)[::-1][:k]
    gains = (2 ** np.asarray(labels)[order] - 1).astype(np.float64)
    discounts = np.log2(np.arange(len(gains)) + 2)
    return float((gains / discounts).sum())


def ndcg_score(labels: np.ndarray, scores: np.ndarray, k: int) -> float:
    labels = np.asarray(labels)
    ideal = _dcg(labels, labels, k)
    if ideal == 0:
        return 0.0
    return _dcg(labels, scores, k) / ideal


def aggregate(impressions: list[tuple[np.ndarray, np.ndarray]]) -> dict[str, float]:
    """Mean metrics over a list of (labels, scores) impressions."""
    aucs, mrrs, n5, n10 = [], [], [], []
    for labels, scores in impressions:
        a = auc_score(labels, scores)
        if a is not None:
            aucs.append(a)
        mrrs.append(mrr_score(labels, scores))
        n5.append(ndcg_score(labels, scores, 5))
        n10.append(ndcg_score(labels, scores, 10))
    return {
        "auc": float(np.mean(aucs)) if aucs else 0.0,
        "mrr": float(np.mean(mrrs)) if mrrs else 0.0,
        "ndcg@5": float(np.mean(n5)) if n5 else 0.0,
        "ndcg@10": float(np.mean(n10)) if n10 else 0.0,
    }


In [ ]:
%%writefile common.py
"""Helpers shared by the suite and its stages."""
from __future__ import annotations

import os
import time

import numpy as np
import torch

import rep_eval as R

TASK = "Given the headlines a reader recently clicked, retrieve the news article they would most likely read next"


def log(*a):
    print(*a, flush=True)


# ------------------------------------------------------------------ wall-clock budget
class Clock:
    """One wall-clock budget shared by every ``suite.main`` call of a notebook.  Long loops receive
    ``deadline`` (absolute ``time.time()``) and stop gracefully, so a slow GPU degrades the run instead of killing it."""

    def __init__(self):
        self.t0, self.budget = time.time(), None

    def start(self, hours):
        """The first call of a notebook fixes the budget; later ``suite.main`` calls share it."""
        if self.budget is None and hours:
            self.t0, self.budget = time.time(), float(hours) * 3600.0

    def reset(self):
        self.t0, self.budget = time.time(), None

    @property
    def deadline(self):
        return None if self.budget is None else self.t0 + self.budget

    def left(self):
        return float("inf") if self.budget is None else self.t0 + self.budget - time.time()

    def elapsed(self):
        return time.time() - self.t0


CLOCK = Clock()


def time_left():
    return CLOCK.left()


def out_of_time(margin=0.0):
    return CLOCK.left() < margin


# ------------------------------------------------------------------ prompts / pooling of results
def query_prefix(name, fmt="auto", instruct=True):
    low = name.lower()
    if fmt == "auto":
        fmt = "qwen3" if "qwen3" in low else "bge" if "bge" in low else "e5" if "e5" in low else "plain"
    if fmt == "qwen3":
        return f"Instruct: {TASK}\nQuery:" if instruct else ""
    if fmt == "bge":
        return "Represent this sentence for searching relevant passages: " if instruct else ""
    if fmt == "e5":
        return "query: "
    return ""


def avg_results(rs):
    """Average several seeds of the same variant (per-impression arrays and slice values)."""
    if len(rs) == 1:
        return rs[0]
    out = dict(rs[0])
    out["arr"] = {k: np.nanmean([r["arr"][k] for r in rs], axis=0) for k in R.METRICS}
    out["pos_auc"] = np.mean([r["pos_auc"] for r in rs], axis=0)
    m = {"auc": float(np.nanmean(out["arr"]["auc"])), **{k: float(np.mean(out["arr"][k])) for k in R.METRICS[1:]}}
    if "cold" in out:
        c = out["cold"]
        m["cold_auc"] = float(out["pos_auc"][c].mean()) if c.any() else float("nan")
        m["warm_auc"] = float(out["pos_auc"][~c].mean()) if (~c).any() else float("nan")
    out["mean"] = m
    return out


# ------------------------------------------------------------------ Hugging Face loading
def load_hf(auto_cls, name, dtype=None, trust_remote_code=False):
    """``from_pretrained`` that copes with the ``torch_dtype`` -> ``dtype`` rename between transformers versions
    and always ends up in the requested dtype."""
    kw = {"trust_remote_code": trust_remote_code}
    if dtype is None:
        return auto_cls.from_pretrained(name, **kw)
    try:
        model = auto_cls.from_pretrained(name, dtype=dtype, **kw)
    except TypeError:
        model = auto_cls.from_pretrained(name, torch_dtype=dtype, **kw)
    return model if model.dtype == dtype else model.to(dtype)


def fix_peft_torchao():
    """Kaggle images ship torchao 0.10 while recent peft raises ``ImportError`` for torchao < 0.16 the moment a LoRA adapter is injected
    (``dispatch_torchao`` -> ``is_torchao_available``).  We never use torchao, so make peft treat it as absent.  Returns True if patched."""
    try:
        import peft.import_utils as iu
        iu.is_torchao_available()
        return False                                          # absent or recent enough: nothing to do
    except ImportError:
        pass
    except Exception:
        return False
    import importlib
    off = lambda *a, **k: False
    iu.is_torchao_available = off
    if hasattr(iu, "is_torchao_ge_v0_18_0"):
        iu.is_torchao_ge_v0_18_0 = off
    for mod in ("peft.tuners.lora.torchao", "peft.utils.quantization_utils"):
        try:
            m = importlib.import_module(mod)
            for name in ("is_torchao_available", "is_torchao_ge_v0_18_0"):
                if hasattr(m, name):
                    setattr(m, name, off)
        except Exception:
            pass
    log("      [peft] incompatible torchao detected -> treating it as absent (we do not use it)")
    return True


def decoder_only(name):
    """Last-token-pooled embedding models are decoder-only LLMs (Qwen3-Embedding, e5-mistral, ...)."""
    import adapt
    return adapt.guess_pooling(name) == "last"


# ------------------------------------------------------------------ evaluation subsets
def subset_evalset(rows, ids, device="cpu"):
    """EvalSet over ``rows[ids]`` that remembers the original positions (``.ids``) so that results over
    different subsets of the same split can be paired impression by impression."""
    ids = np.asarray(ids, np.int64)
    es = R.EvalSet([rows[i] for i in ids], device=device)
    es.ids = ids
    return es


def random_subset(n_total, n, seed=0):
    """Nested random subsets: ``random_subset(N, a) ⊂ random_subset(N, b)`` for a <= b (same seed)."""
    n = min(int(n), n_total)
    return np.sort(np.random.default_rng(seed).permutation(n_total)[:n])


# ------------------------------------------------------------------ score fusion and caching
def zscore(x, mask):
    """Standardise scores within each impression over its real candidates (same as V10 ``_znorm``)."""
    m = mask.float()
    n = m.sum(-1, keepdim=True).clamp(min=1)
    mu = (x * m).sum(-1, keepdim=True) / n
    sd = (((x - mu) * m) ** 2).sum(-1, keepdim=True).div(n).sqrt()
    return (x - mu) / (sd + 1e-6)


def fuse(parts):
    """Weighted sum of per-impression z-scored scorers: ``parts = [(scorer, weight), ...]``."""
    def f(b):
        return sum(w * zscore(s(b).float(), b["cand_mask"]) for s, w in parts)
    return f


def pop_scorer(b):
    """V10's online smoothed log-CTR of every candidate (counted only from strictly earlier impressions)."""
    return b["pop"]


class ScoreCache:
    """Raw per-candidate scores of an expensive scorer (SID language model, LLM tower, LLM judge), computed once
    per EvalSet so that fusion-weight grids are free."""

    def __init__(self, evalset, score_fn):
        self.n, self.rows = evalset.n, [None] * evalset.n
        for b in evalset:
            s = score_fn(b).float().cpu().numpy()
            cm = b["cand_mask"].cpu().numpy()
            for k, r in enumerate(b["rows"]):
                self.rows[r] = s[k, :int(cm[k].sum())].copy()

    def scorer(self, device, shift=None):
        """``shift``: optional callable ``(batch) -> [B,C]`` tensor subtracted from the cached scores."""
        def f(b):
            C = b["cand"].shape[1]
            out = np.zeros((len(b["rows"]), C), np.float32)
            for k, r in enumerate(b["rows"]):
                v = self.rows[r]
                out[k, :len(v)] = v
            out = torch.from_numpy(out).to(device)
            return out if shift is None else out - shift(b)
        return f


# ------------------------------------------------------------------ history-as-text queries
class HistQueries:
    """Unique last-K histories of validation/test rows, their query texts, and a scorer
    ``score = <query vector, candidate vector>`` (history-as-text user representation)."""

    def __init__(self, data, titles, K):
        self.titles, self.K = titles, K
        self.keys, self.rowq = {}, {}
        for sp, rows in (("val", data.validation), ("test", data.test)):
            rq = []
            for r in rows:
                h = tuple(r[2][-K:])
                rq.append(self.keys.setdefault(h, len(self.keys)) if h else -1)
            self.rowq[sp] = np.array(rq, np.int64)
        self.uniq = sorted(self.keys, key=self.keys.get)

    def needed(self, split, ids=None):
        """Indices of the unique queries used by the rows ``ids`` (default: every row) of a split."""
        rq = self.rowq[split] if ids is None else self.rowq[split][np.asarray(ids)]
        return np.unique(rq[rq >= 0])

    def texts(self, qidx=None):
        qidx = range(len(self.uniq)) if qidx is None else qidx
        return ["; ".join(self.titles[i] for i in self.uniq[q]) for q in qidx]

    def alloc(self, d):
        return np.zeros((max(len(self.uniq), 1), d), np.float32)

    def scorer(self, Q, E, split, device, ids=None):
        """``Q``: [n_unique, d] numpy; ``E``: candidate matrix tensor on ``device``;
        ``ids``: original row ids when scoring a subset EvalSet."""
        Qt = torch.from_numpy(np.concatenate([np.asarray(Q, np.float32), np.zeros((1, Q.shape[1]), np.float32)])).to(device)
        rq = self.rowq[split].copy()
        rq[rq < 0] = len(Q)                              # empty history -> zero query (ties like the mean-pool control)
        rq_t = torch.from_numpy(rq).to(device)
        ids_t = None if ids is None else torch.from_numpy(np.asarray(ids)).to(device)

        def f(b):
            r = torch.as_tensor(b["rows"], device=device)
            r = r if ids_t is None else ids_t[r]
            return torch.einsum("bd,bcd->bc", Qt[rq_t[r]], E[b["cand"]])
        return f


def atomic_write(path, data, mode="w"):
    tmp = path + ".tmp"
    with open(tmp, mode, **({} if "b" in mode else {"encoding": "utf-8"})) as f:
        f.write(data)
    os.replace(tmp, path)


In [ ]:
%%writefile rep_eval.py
"""Representation-level evaluation on the V10 protocol.

Zero-shot scorers (no learned user model) isolate the quality of the *item
embeddings*: user = mean of the clicked-history vectors, score = dot product with
the candidate vector.  Everything is vectorised in torch, so a full 73k-impression
pass takes seconds; the metric definitions are identical to ``metrics.py``
(tie-aware AUC, MRR over clicked items, nDCG@k) -- ``selftest.py`` asserts this.

Besides the usual means we keep:
* per-impression arrays -> paired confidence intervals between two variants;
* per-*positive* AUC contributions -> cold/warm slices (is the clicked article one
  the training data never saw?), the quantity LLM embeddings are meant to help.
"""
from __future__ import annotations

import math

import numpy as np
import torch

import metrics as ref_metrics  # vendored V10 metrics.py (tie handling reference)

METRICS = ("auc", "mrr", "ndcg@5", "ndcg@10")


class EvalSet:
    """Padded, slate-size-sorted view of validation/test rows
    ``(impr_id, user, hist, cands, labels, pop)``."""

    def __init__(self, rows, max_hist=50, device="cpu", max_elems=3_000_000, max_batch=2048):
        self.rows, self.n = rows, len(rows)
        self.max_hist, self.device = max_hist, device
        self.max_elems, self.max_batch = max_elems, max_batch
        self.lens = np.array([len(r[3]) for r in rows], dtype=np.int64)
        self.order = np.argsort(self.lens, kind="stable")

    def __iter__(self):
        o, L, i = self.order, self.lens, 0
        while i < self.n:
            b = 1
            while (i + b < self.n and b < self.max_batch
                   and (b + 1) * max(int(L[o[i + b]]), 1) ** 2 <= self.max_elems):
                b += 1
            idx = o[i:i + b]
            i += b
            yield self._pack(idx)

    def _pack(self, idx):
        B, H = len(idx), self.max_hist
        C = max(int(self.lens[idx].max()), 1)
        hist = np.zeros((B, H), np.int64)
        hm = np.zeros((B, H), bool)
        cand = np.zeros((B, C), np.int64)
        cm = np.zeros((B, C), bool)
        lab = np.zeros((B, C), np.float32)
        pop = np.zeros((B, C), np.float32)
        user = np.zeros(B, np.int64)
        for k, r in enumerate(idx):
            _, u, h, c, y, pp = self.rows[r]
            h = list(h)[-H:]
            hist[k, :len(h)], hm[k, :len(h)] = h, True
            cand[k, :len(c)], cm[k, :len(c)], lab[k, :len(c)], pop[k, :len(c)] = c, True, y, pp
            user[k] = u
        t = lambda x: torch.from_numpy(x).to(self.device)
        return {"rows": idx, "hist": t(hist), "hist_mask": t(hm), "cand": t(cand),
                "cand_mask": t(cm), "labels": t(lab), "user": t(user), "pop": t(pop)}


def batch_metrics(S, Y, M):
    """Per-impression metrics for padded scores ``S`` [B,C], labels ``Y``, mask ``M``."""
    B, C = S.shape
    S = torch.nan_to_num(S.float(), nan=0.0)
    Sm = S.masked_fill(~M, float("-inf"))
    pos = (Y > 0.5) & M
    neg = (Y <= 0.5) & M
    # exact tie-aware AUC through pairwise comparison (B*C*C is bounded by EvalSet)
    win = (Sm[:, :, None] > Sm[:, None, :]).float() + 0.5 * (Sm[:, :, None] == Sm[:, None, :]).float()
    n_neg = neg.sum(1).float()
    eq = (Sm[:, :, None] == Sm[:, None, :]) & M[:, :, None] & M[:, None, :]
    tie = (eq.sum((1, 2)) > M.sum(1)).cpu()  # more equalities than the diagonal alone
    pos_auc = (win * neg[:, None, :].float()).sum(2) / n_neg.clamp(min=1)[:, None]
    P = pos.sum(1).float()
    auc = (pos_auc * pos.float()).sum(1) / P.clamp(min=1)
    auc = torch.where((P > 0) & (n_neg > 0), auc, torch.full_like(auc, float("nan")))
    # ranking metrics
    order = Sm.argsort(dim=1, descending=True)
    ys = pos.float().gather(1, order)
    ranks = torch.arange(1, C + 1, device=S.device).float()[None, :]
    mrr = (ys / ranks).sum(1) / P.clamp(min=1)
    disc = 1.0 / torch.log2(ranks + 1.0)

    def ndcg(k):
        dcg = (ys[:, :k] * disc[:, :k]).sum(1)
        cum = torch.cumsum(disc[0, :min(k, C)], 0)
        kk = torch.clamp(P, max=min(k, C)).long()
        ideal = torch.where(kk > 0, cum[(kk - 1).clamp(min=0)], torch.zeros_like(dcg))
        return torch.where(ideal > 0, dcg / ideal.clamp(min=1e-12), torch.zeros_like(dcg))

    return {"auc": auc, "mrr": mrr, "ndcg@5": ndcg(5), "ndcg@10": ndcg(10),
            "pos_auc": pos_auc, "pos_mask": pos, "tie": tie, "scores": Sm}


@torch.no_grad()
def evaluate(score_fn, evalset, seen=None):
    """Run ``score_fn(batch) -> [B,C]`` over an EvalSet.

    ``seen`` (bool array over news ids): news the training data observed.  Used for
    the cold/warm slices, which are over *clicked* articles only.
    """
    arr = {k: np.full(evalset.n, np.nan, np.float64) for k in METRICS}
    pnews, pauc, prow = [], [], []
    for b in evalset:
        m = batch_metrics(score_fn(b), b["labels"], b["cand_mask"])
        r = b["rows"]
        for k in METRICS:
            arr[k][r] = m[k].cpu().numpy()
        if m["tie"].any():  # exact ties (e.g. users with empty history): defer to V10 metrics.py
            n_valid = b["cand_mask"].sum(1).cpu().numpy()
            sc = m["scores"].cpu().numpy()
            lb = b["labels"].cpu().numpy()
            for k in np.nonzero(m["tie"].numpy())[0]:
                n = int(n_valid[k])
                s_k, y_k = sc[k, :n].astype(np.float32), lb[k, :n]
                arr["mrr"][r[k]] = ref_metrics.mrr_score(y_k, s_k)
                arr["ndcg@5"][r[k]] = ref_metrics.ndcg_score(y_k, s_k, 5)
                arr["ndcg@10"][r[k]] = ref_metrics.ndcg_score(y_k, s_k, 10)
            evalset.n_tied = getattr(evalset, "n_tied", 0) + int(m["tie"].sum())
        pm = m["pos_mask"].cpu().numpy()
        pnews.append(b["cand"].cpu().numpy()[pm])
        pauc.append(m["pos_auc"].cpu().numpy()[pm])
        prow.append(np.repeat(r, pm.sum(1)))
    out = {"arr": arr, "ids": np.asarray(getattr(evalset, "ids", np.arange(evalset.n))),
           "pos_news": np.concatenate(pnews), "pos_auc": np.concatenate(pauc),
           "pos_row": np.concatenate(prow)}
    out["mean"] = {"auc": float(np.nanmean(arr["auc"])),
                   **{k: float(np.mean(arr[k])) for k in METRICS[1:]}}
    if seen is not None:
        cold = ~seen[out["pos_news"]]
        out["cold"] = cold
        out["cold_frac"] = float(cold.mean()) if len(cold) else float("nan")
        out["mean"]["cold_auc"] = float(out["pos_auc"][cold].mean()) if cold.any() else float("nan")
        out["mean"]["warm_auc"] = float(out["pos_auc"][~cold].mean()) if (~cold).any() else float("nan")
    return out


# ------------------------------------------------------------------ scorers
def user_vectors(E, b):
    m = b["hist_mask"].unsqueeze(-1).to(E.dtype)
    return (E[b["hist"]] * m).sum(1) / m.sum(1).clamp(min=1)


def mean_pool_scorer(E):
    """User = mean clicked-history vector; score = dot with the candidate (== cosine
    ranking for L2-normalised E).  Same scorer as the V10 ``bge_zeroshot`` baseline."""
    def f(b):
        return torch.einsum("bd,bcd->bc", user_vectors(E, b), E[b["cand"]])
    return f


# --------------------------------------------------------------- statistics
def ratio_ci(d, groups=None, z=1.96):
    """CI for the mean of ``d`` with (optional) clustered observations: positives of the
    same impression are correlated, so the SE uses per-impression sums (ratio estimator)."""
    d = np.asarray(d, np.float64)
    ok = ~np.isnan(d)
    d = d[ok]
    if len(d) < 2:
        return float("nan"), float("nan"), float("nan")
    if groups is None:
        m = d.mean()
        se = d.std(ddof=1) / math.sqrt(len(d))
        return float(m), float(m - z * se), float(m + z * se)
    g = np.asarray(groups)[ok]
    _, inv = np.unique(g, return_inverse=True)
    G = inv.max() + 1
    s = np.bincount(inv, weights=d, minlength=G)
    n = np.bincount(inv, minlength=G).astype(np.float64)
    m = s.sum() / n.sum()
    resid = s - m * n
    var = (G / max(G - 1, 1)) * (resid ** 2).sum() / (n.sum() ** 2)
    se = math.sqrt(var)
    return float(m), float(m - z * se), float(m + z * se)


def _align(res, ref):
    """Indices that pair up the impressions two results have in common (full set vs a random subset)."""
    a, b = res["ids"], ref["ids"]
    if len(a) == len(b) and np.array_equal(a, b):
        return slice(None), slice(None)
    _, ia, ib = np.intersect1d(a, b, return_indices=True)
    return ia, ib


def compare(res, ref, key="ndcg@10"):
    """Paired difference ``res - ref`` of an impression-level metric with a 95% CI (common impressions only)."""
    ia, ib = _align(res, ref)
    return ratio_ci(res["arr"][key][ia] - ref["arr"][key][ib])


def compare_slice(res, ref, which="cold"):
    """Paired difference of the positive-level AUC on cold (or warm) clicked articles.  Clicked articles
    are matched on (original impression id, article id), so results over different EvalSets compare."""
    big = int(max(res["pos_news"].max(initial=0), ref["pos_news"].max(initial=0))) + 1
    ka = res["ids"][res["pos_row"]].astype(np.int64) * big + res["pos_news"]
    kb = ref["ids"][ref["pos_row"]].astype(np.int64) * big + ref["pos_news"]
    _, pa, pb = np.intersect1d(ka, kb, return_indices=True)
    sel = res["cold"][pa] if which == "cold" else ~res["cold"][pa]
    if not sel.any():
        return float("nan"), float("nan"), float("nan")
    d = (res["pos_auc"][pa] - ref["pos_auc"][pb])[sel]
    return ratio_ci(d, groups=(ka[pa] // big)[sel])


def verdict(lo, hi):
    if np.isnan(lo):
        return "n/a"
    if abs(lo) < 1e-7 and abs(hi) < 1e-7:                      # the variant reproduces the reference exactly (e.g. an adapter that kept epoch 0)
        return "identical"
    return "BETTER" if lo > 0 else ("WORSE" if hi < 0 else "no sig. difference")


In [ ]:
%%writefile semgraph.py
"""Semantic-graph / retrieval-augmented representations ("GraphRAG-lite").

Why this exists: V10 showed the *click* graph (LightGCN) is noise on MIND-small -- a
fresh article has no click edge.  Here every variant is **content-only** (article
embeddings + the Wikidata entities MIND annotates on titles/abstracts, never click
labels), so it can still say something about *cold* articles:

* ``entity_view``   entity-grounded retrieval: an article is enriched with the context
                    vector of its entities (centroid of older articles mentioning them).
* ``knn_view``      dense retrieval: enrich with its k nearest older articles.
* ``communities``   GraphRAG-style global view: Louvain communities on the article kNN
                    graph; a user's history is summarised at community level.
* ``user_rag``      collaborative retrieval: add the clicks of similar *other* users.

Causality: the memory (what may be retrieved) is restricted to articles observed before
the evaluation period -- validation uses train_core only, test uses train_core+validation.
This is retrieval-augmented *representation* (no text generation); an LLM that reads the
retrieved context is the natural next step and is left out on purpose.
"""
from __future__ import annotations

import json
import os

import numpy as np
import scipy.sparse as sp
import torch
import torch.nn.functional as F


# --------------------------------------------------------------------- loading
def load_news_meta(train_dir, dev_dir):
    """Same article order as ``MindData.from_mind`` (global id = position + 1)."""
    by_id, n_cols = {}, 0
    for d in (train_dir, dev_dir):
        path = os.path.join(d, "news.tsv") if d else None
        if not path or not os.path.exists(path):
            continue
        with open(path, encoding="utf-8") as f:
            for line in f:
                p = line.rstrip("\n").split("\t")
                if len(p) >= 5:
                    n_cols = max(n_cols, len(p))
                    by_id[p[0]] = (p[1], p[3], p[4], p[6] if len(p) > 6 else "", p[7] if len(p) > 7 else "", p[2])
    ids = list(by_id)
    titles = [by_id[i][1] for i in ids]
    abstracts = [by_id[i][2] for i in ids]

    def wikidata(*cols):
        out = set()
        for c in cols:
            try:
                for e in json.loads(c) if c else []:
                    if e.get("WikidataId"):
                        out.add(e["WikidataId"])
            except (ValueError, TypeError, AttributeError):
                pass
        return sorted(out)

    def labels(*cols):
        """Entity names in order of first appearance (title entities first), duplicates removed."""
        out = []
        for c in cols:
            try:
                for e in json.loads(c) if c else []:
                    lab = (e.get("Label") or "").strip()
                    if lab and lab not in out:
                        out.append(lab)
            except (ValueError, TypeError, AttributeError):
                pass
        return out

    ents = [wikidata(by_id[i][3], by_id[i][4]) for i in ids]
    return {"ids": ids, "titles": titles, "abstracts": abstracts, "cats": [by_id[i][0] for i in ids],
            "subcats": [by_id[i][5] for i in ids],
            "text": [(t + ". " + a).strip() for t, a in zip(titles, abstracts)], "ents": ents,
            "ent_labels": [labels(by_id[i][3], by_id[i][4]) for i in ids], "n_cols": n_cols}


def observed_masks(data):
    """Articles the *training* rows observed (history / clicked / sampled negative) and the
    articles seen in validation slates.  Index 0 (PAD) is always False."""
    n = data.n_news
    tr = np.zeros(n, bool)
    for _user, hist, pos, negs in data.train_core:
        tr[np.asarray(hist, dtype=np.int64)] = True
        tr[pos] = True
        tr[np.asarray(negs, dtype=np.int64)] = True
    va = np.zeros(n, bool)
    for _i, _u, hist, cands, _labs, _pop in data.validation:
        va[np.asarray(hist, dtype=np.int64)] = True
        va[np.asarray(cands, dtype=np.int64)] = True
    tr[0] = va[0] = False
    return tr, va


def entity_matrix(ents, n_news):
    """Binary article x entity incidence (row 0 = PAD stays empty)."""
    vocab, rows, cols = {}, [], []
    for i, es in enumerate(ents):
        for e in es:
            rows.append(i + 1)
            cols.append(vocab.setdefault(e, len(vocab)))
    A = sp.csr_matrix((np.ones(len(rows), np.float32), (rows, cols)), shape=(n_news, max(len(vocab), 1)))
    return A


def _norm(x, eps=1e-8):
    return x / (np.linalg.norm(x, axis=1, keepdims=True) + eps)


# ---------------------------------------------------------------------- memory
class Memory:
    """Retrieval memory over a boolean mask of admissible (older) articles."""

    def __init__(self, E, A, mem, device="cpu"):
        self.E = np.ascontiguousarray(E, dtype=np.float32)
        self.A, self.mem, self.device = A, np.asarray(mem, bool), device
        self.mem_ids = np.nonzero(self.mem)[0]
        self.n = len(E)

    # --- entity-grounded retrieval
    def entity_view(self, beta):
        Am = sp.diags(self.mem.astype(np.float32)) @ self.A
        df = np.asarray(Am.sum(0)).ravel()
        ctx = _norm(np.asarray(Am.T @ self.E, np.float32))               # entity context vectors
        idf = np.log((len(self.mem_ids) + 1.0) / (df + 1.0)) * (df > 0)  # no memory -> no context
        R = np.asarray((self.A @ sp.diags(idf.astype(np.float32))) @ ctx, np.float32)
        has = np.linalg.norm(R, axis=1) > 0
        out = self.E.copy()
        out[has] = _norm(self.E[has] + beta * _norm(R[has]))
        return out, float(has[1:].mean())

    # --- dense retrieval
    def neighbours(self, rows, k, chunk=2048):
        """Top-k admissible neighbours (self excluded) for ``rows`` -> (idx [r,k], sim [r,k])."""
        dev = self.device
        Em = torch.from_numpy(self.E[self.mem_ids]).to(dev)
        pos_in_mem = np.full(self.n, -1, np.int64)
        pos_in_mem[self.mem_ids] = np.arange(len(self.mem_ids))
        rows = np.asarray(rows)
        nbr, sim = [], []
        for s in range(0, len(rows), chunk):
            r = rows[s:s + chunk]
            sc = torch.from_numpy(self.E[r]).to(dev) @ Em.T
            pm = pos_in_mem[r]
            hit = np.nonzero(pm >= 0)[0]
            if len(hit):
                sc[torch.from_numpy(hit).to(dev), torch.from_numpy(pm[hit]).to(dev)] = float("-inf")
            v, i = sc.topk(min(k, sc.shape[1]), dim=1)
            nbr.append(self.mem_ids[i.cpu().numpy()])
            sim.append(v.cpu().numpy())
        return np.concatenate(nbr), np.concatenate(sim)

    def knn_view(self, beta, k, rows=None):
        rows = np.arange(1, self.n) if rows is None else np.asarray(rows)
        nbr, sim = self.neighbours(rows, k)
        w = np.clip(sim, 0, None)
        w = w / (w.sum(1, keepdims=True) + 1e-8)
        agg = (self.E[nbr] * w[:, :, None]).sum(1)
        out = self.E.copy()
        out[rows] = _norm(self.E[rows] + beta * _norm(agg))
        return out

    # --- GraphRAG-style communities
    def communities(self, k=10, resolution=1.0, seed=0, max_nodes=80_000):
        """Louvain communities on the memory kNN graph; every other article joins its nearest
        community centroid.  Returns (labels [n], centroids [K,d])."""
        ids = self.mem_ids
        try:
            if len(ids) > max_nodes:
                raise RuntimeError("memory too large for networkx Louvain")
            import networkx as nx
            nbr, sim = self.neighbours(ids, k)
            G = nx.Graph()
            G.add_nodes_from(ids.tolist())
            for a, row, sw in zip(ids, nbr, sim):
                for b, w in zip(row, sw):
                    if w > 0 and a != b:
                        G.add_edge(int(a), int(b), weight=float(w))
            comms = nx.community.louvain_communities(G, weight="weight", resolution=resolution, seed=seed)
            lab_mem = np.zeros(len(ids), np.int64)
            where = {int(n): i for i, n in enumerate(ids)}
            for ci, c in enumerate(comms):
                for n in c:
                    lab_mem[where[n]] = ci
            method = "louvain"
        except Exception as exc:  # fall back to k-means centroids on the same embeddings
            from sklearn.cluster import MiniBatchKMeans
            K = int(max(8, min(500, len(ids) // 100)))
            lab_mem = MiniBatchKMeans(K, random_state=seed, n_init=3).fit_predict(self.E[ids])
            method = f"kmeans({exc.__class__.__name__})"
        K = int(lab_mem.max()) + 1
        cent = np.zeros((K, self.E.shape[1]), np.float32)
        np.add.at(cent, lab_mem, self.E[ids])
        cent = _norm(cent)
        labels = np.full(self.n, -1, np.int64)
        labels[ids] = lab_mem
        rest = np.nonzero(labels < 0)[0]
        rest = rest[rest > 0]
        if len(rest):
            labels[rest] = (self.E[rest] @ cent.T).argmax(1)
        return labels, cent, method


# --------------------------------------------------------------------- scorers
def community_scorer(E, labels, cent, lam, device="cpu"):
    """score = cos(user, cand) + lam * community-level affinity of the user's history."""
    Et = torch.from_numpy(np.ascontiguousarray(E, np.float32)).to(device)
    lab = torch.from_numpy(labels).to(device)
    C = torch.from_numpy(np.ascontiguousarray(cent, np.float32)).to(device)
    Cs, K = C @ C.T, len(cent)                                             # [K,K] community affinity

    def f(b):
        m = b["hist_mask"].unsqueeze(-1).float()
        u = (Et[b["hist"]] * m).sum(1) / m.sum(1).clamp(min=1)
        base = torch.einsum("bd,bcd->bc", u, Et[b["cand"]])
        hl = lab[b["hist"]].clamp(min=0)
        P = torch.zeros(hl.shape[0], K, device=Et.device).scatter_add_(1, hl, b["hist_mask"].float())
        P = P / P.sum(1, keepdim=True).clamp(min=1)                        # user's community histogram
        aff = P @ Cs                                                       # [B,K] affinity to each community
        return base + lam * aff.gather(1, lab[b["cand"]].clamp(min=0))
    return f


class UserMemory:
    """Profiles of users observed in training rows (history + clicks) and their click centroids."""

    def __init__(self, data, E, device="cpu"):
        U, N = data.n_users, data.n_news
        pu, pn, cu, cn = [], [], [], []
        for user, hist, pos, _negs in data.train_core:
            pu.extend([user] * len(hist)); pn.extend(hist)
            pu.append(user); pn.append(pos)
            cu.append(user); cn.append(pos)
        prof = sp.csr_matrix((np.ones(len(pu), np.float32), (pu, pn)), shape=(U, N)).sign()   # unique pairs
        clk = sp.csr_matrix((np.ones(len(cu), np.float32), (cu, cn)), shape=(U, N)).sign()
        self.profile = torch.from_numpy(_norm(np.asarray(prof @ E, np.float32))).to(device)
        self.clicks = torch.from_numpy(_norm(np.asarray(clk @ E, np.float32))).to(device)
        self.valid = (self.profile.norm(dim=1) > 0)
        self.device = device


def user_rag_scorer(E, um, gamma, device="cpu", k=20, chunk=256):
    """u' = u + gamma * mean(click centroids of the k most similar OTHER users)."""
    Et = torch.from_numpy(np.ascontiguousarray(E, np.float32)).to(device)
    prof, clicks, valid = um.profile.to(device), um.clicks.to(device), um.valid.to(device)

    def f(b):
        m = b["hist_mask"].unsqueeze(-1).float()
        u = (Et[b["hist"]] * m).sum(1) / m.sum(1).clamp(min=1)
        nb = torch.zeros_like(u)
        for s in range(0, u.shape[0], chunk):                              # bound the [chunk, n_users] similarity
            sim = F.normalize(u[s:s + chunk], dim=-1) @ prof.T
            sim = sim.masked_fill(~valid[None, :], float("-inf"))
            uid = b["user"][s:s + chunk].clamp(max=sim.shape[1] - 1).unsqueeze(1)
            sim.scatter_(1, uid, float("-inf"))                            # never retrieve the user itself
            top = sim.topk(min(k, sim.shape[1]), dim=1).indices
            nb[s:s + chunk] = clicks[top].mean(1)
        u2 = u + gamma * nb * (m.sum(1) > 0).float()                       # cold users keep their (empty) profile
        return torch.einsum("bd,bcd->bc", u2, Et[b["cand"]])
    return f


In [ ]:
%%writefile adapt.py
"""Contrastive adaptation of text embeddings to news recommendation.

Hypothesis (LLM2Rec, KDD'25 / LC-Rec / CoLLM line): a frozen text embedding knows what an
article is *about*, not which articles readers consume together.  We adapt it with a
co-click objective and ask whether the gain **transfers to articles never seen in
training** -- the case that matters for news.

Training signal (V10 ``train_core`` only, so nothing from validation/test leaks):
  anchor   = one of the reader's last ``k_anchor`` history articles
  positive = an article the reader then clicked
  hard neg = articles shown to the same reader in that impression but skipped
             (impression-aware negatives; MIND provides them for free)
  + in-batch negatives, with false-negative masking (same article id).

Adapters (all start as the identity => step 0 == the frozen control by construction):
  lin / mlp     residual head on precomputed vectors            (TAM-style, no encoder pass)
  lora / dora   low-rank adaptation of the encoder, in the loop (LoRA; DoRA = weight-decomposed)
Options: ``mrl`` Matryoshka loss over nested prefixes (MRL, NeurIPS'22; used in recsys by
fMRLRec/SMEC) and ``hardneg`` impression negatives.
"""
from __future__ import annotations

import math
import time

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

from common import fix_peft_torchao, load_hf


# ------------------------------------------------------------------- pairs
def build_pairs(data, k_anchor=3, max_pairs=200_000, seed=0):
    """(anchor, positive, hard-negatives) from ``data.train_core``."""
    rng = np.random.default_rng(seed)
    A, P, N = [], [], []
    seen = set()
    for _user, hist, pos, negs in data.train_core:
        if not len(hist):
            continue
        for a in hist[-k_anchor:]:
            key = (a, pos)
            if key in seen or a == pos:
                continue
            seen.add(key)
            A.append(a); P.append(pos); N.append(negs)
    A, P, N = np.array(A, np.int64), np.array(P, np.int64), np.array(N, np.int64)
    if len(A) > max_pairs:
        keep = rng.choice(len(A), max_pairs, replace=False)
        A, P, N = A[keep], P[keep], N[keep]
    return A, P, N


# -------------------------------------------------------------------- loss
def mrl_dims(d, scales=(64, 128, 256)):
    return [m for m in scales if m < d] + [d]


def contrastive_loss(a, p, p_idx, negs=None, tau=0.05, dims=None):
    """InfoNCE(anchor -> positive) with in-batch negatives (+ optional hard negatives),
    averaged over Matryoshka prefixes.  ``p_idx`` masks in-batch false negatives."""
    dims = dims or [a.shape[-1]]
    same = p_idx[:, None] == p_idx[None, :]
    same.fill_diagonal_(False)
    tgt = torch.arange(len(a), device=a.device)
    total = 0.0
    for m in dims:
        am, pm = F.normalize(a[:, :m], dim=-1), F.normalize(p[:, :m], dim=-1)
        logits = (am @ pm.T / tau).masked_fill(same, float("-inf"))
        if negs is not None:
            nm = F.normalize(negs[:, :, :m], dim=-1)
            logits = torch.cat([logits, torch.einsum("bd,bkd->bk", am, nm) / tau], dim=1)
        total = total + F.cross_entropy(logits, tgt)
    return total / len(dims)


# ------------------------------------------------------------------- heads
class ResLinear(nn.Module):
    def __init__(self, d):
        super().__init__()
        self.w = nn.Linear(d, d, bias=False)
        nn.init.zeros_(self.w.weight)

    def forward(self, x):
        return F.normalize(x + self.w(x), dim=-1)


class ResMLP(nn.Module):
    def __init__(self, d, h=None):
        super().__init__()
        self.a, self.b = nn.Linear(d, h or d), nn.Linear(h or d, d)
        nn.init.zeros_(self.b.weight)
        nn.init.zeros_(self.b.bias)

    def forward(self, x):
        return F.normalize(x + self.b(F.gelu(self.a(x))), dim=-1)


HEADS = {"lin": ResLinear, "mlp": ResMLP}


def train_head(E0, pairs, kind="lin", mrl=False, hardneg=False, epochs=3, bs=1024, lr=1e-3,
               tau=0.05, seed=0, device="cpu", val_fn=None, deadline=None, log=print):
    """Train a residual head on precomputed vectors.  ``val_fn(E) -> float`` (higher is
    better) selects the epoch; epoch 0 (= frozen control) competes, so the returned
    embedding is never worse than frozen on validation."""
    torch.manual_seed(seed)
    rng = np.random.default_rng(seed)
    E = torch.from_numpy(E0).to(device)
    head = HEADS[kind](E.shape[1]).to(device)
    opt = torch.optim.AdamW(head.parameters(), lr=lr, weight_decay=0.0)
    A, P, N = pairs
    dims = mrl_dims(E.shape[1]) if mrl else None
    best = (val_fn(E0) if val_fn else -math.inf, E0.copy(), 0)
    for ep in range(1, epochs + 1):
        if deadline is not None and time.time() > deadline and ep > 1:
            break
        perm = rng.permutation(len(A))
        head.train()
        tot, nb = 0.0, 0
        for s in range(0, len(A), bs):
            ix = perm[s:s + bs]
            a, p = head(E[A[ix]]), head(E[P[ix]])
            n = head(E[N[ix]]) if hardneg else None
            loss = contrastive_loss(a, p, torch.from_numpy(P[ix]).to(device), n, tau, dims)
            opt.zero_grad(); loss.backward(); opt.step()
            tot += loss.item(); nb += 1
        head.eval()
        with torch.no_grad():
            Ea = head(E).cpu().numpy()
        score = val_fn(Ea) if val_fn else float(ep)          # no validation signal: keep the last epoch
        log(f"      [{kind}{'+mrl' if mrl else ''}{'+hn' if hardneg else ''}] epoch {ep} loss={tot / max(nb, 1):.4f} val={score:.4f}")
        if score > best[0]:
            best = (score, Ea, ep)
    return best[1], {"best_epoch": best[2], "val": float(best[0])}


# ----------------------------------------------------------- encoder in loop
POOLING_BY_NAME = (("qwen3-embedding", "last"), ("bge", "cls"), ("e5", "mean"), ("gte", "mean"), ("minilm", "mean"))


def guess_pooling(name):
    low = name.lower()
    for key, mode in POOLING_BY_NAME:
        if key in low:
            return mode
    return "mean"


class HFEncoder:
    """Plain Hugging Face encoder with explicit pooling (so LoRA/DoRA training and
    inference share one code path and no ST-version drift)."""

    def __init__(self, name, pooling=None, max_len=128, device="cpu", trust_remote_code=False, prefix="",
                 dtype=torch.float32):
        from transformers import AutoModel, AutoTokenizer
        self.name, self.max_len, self.device, self.prefix = name, max_len, device, prefix
        self.pooling = pooling or guess_pooling(name)
        self.tok = AutoTokenizer.from_pretrained(name, trust_remote_code=trust_remote_code)
        if self.pooling == "last":
            self.tok.padding_side = "left"
        self.model = load_hf(AutoModel, name, dtype, trust_remote_code).to(device)
        self.amp = device == "cuda" and dtype == torch.float32        # fp32 master weights + fp16 autocast

    def _pool(self, out, attn):
        h = out.last_hidden_state
        if self.pooling == "cls":
            return h[:, 0]
        if self.pooling == "last":
            return h[:, -1]
        m = attn.unsqueeze(-1).to(h.dtype)
        return (h * m).sum(1) / m.sum(1).clamp(min=1)

    def embed(self, texts):
        batch = self.tok([self.prefix + t for t in texts], padding=True, truncation=True,
                         max_length=self.max_len, return_tensors="pt").to(self.device)
        kw = {k: v for k, v in batch.items() if k in ("input_ids", "attention_mask", "token_type_ids")}
        if self.pooling == "last":
            kw["use_cache"] = False                                  # decoder-only embedders: no KV cache needed
        with torch.autocast("cuda", dtype=torch.float16, enabled=self.amp):
            out = self.model(**kw)
        return self._pool(out, batch["attention_mask"]).float()

    @torch.no_grad()
    def encode(self, texts, bs=128):
        was = self.model.training
        self.model.eval()
        order = np.argsort([len(t) for t in texts])
        vecs = np.zeros((len(texts), self.model.config.hidden_size), np.float32)
        for s in range(0, len(texts), bs):
            ix = order[s:s + bs]
            vecs[ix] = F.normalize(self.embed([texts[i] for i in ix]), dim=-1).cpu().numpy()
        self.model.train(was)
        return vecs

    def attach_peft(self, mode="lora", r=16, dropout=0.05):
        from peft import LoraConfig, get_peft_model
        fix_peft_torchao()
        names = {n.split(".")[-1] for n, _ in self.model.named_modules()}
        targets = ["q_proj", "v_proj"] if "q_proj" in names else ["query", "value"]
        cfg = LoraConfig(r=r, lora_alpha=2 * r, lora_dropout=dropout, target_modules=targets,
                         use_dora=(mode == "dora"), bias="none")
        self.model = get_peft_model(self.model, cfg)
        try:                                                  # activations of 192 texts x 256 tokens do not fit a 16 GB card otherwise
            self.model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})
        except Exception:
            pass
        return sum(p.numel() for p in self.model.parameters() if p.requires_grad)


def check_equivalence(enc, st_encode, texts, tol=0.99):
    """Guard against pooling/prefix mistakes: HFEncoder must agree with the reference
    (Sentence-Transformers) embeddings of the same model."""
    a, b = enc.encode(texts), st_encode(texts)
    cos = (a * b).sum(1) / (np.linalg.norm(a, axis=1) * np.linalg.norm(b, axis=1) + 1e-8)
    return float(cos.min()), float(cos.mean()), bool(cos.min() >= tol)


def fit_encoder_batch(enc, texts, pairs, bs, hardneg, params, log=print):
    """Dry-run one forward+backward on the pairs with the longest texts; halve the batch while the GPU is out of memory."""
    A, P, N = pairs
    L = np.array([len(texts[a]) + len(texts[p]) for a, p in zip(A, P)])
    while bs >= 4:
        ix = np.argsort(-L)[:bs]
        ids = np.concatenate([A[ix], P[ix]] + ([N[ix].ravel()] if hardneg else []))
        uniq = np.unique(ids)
        try:
            enc.model.train()
            z = enc.embed([texts[i] for i in uniq])
            z.pow(2).mean().backward()
            for p in params:
                p.grad = None
            return bs
        except RuntimeError as e:
            if "out of memory" not in str(e).lower():
                raise
            for p in params:
                p.grad = None
            if enc.device == "cuda":
                torch.cuda.empty_cache()
            bs //= 2
            log(f"      [encoder] out of memory on the longest pairs -> batch size {bs}")
    raise RuntimeError("cannot fit a batch of 4 pairs on the device")


def train_encoder(enc, texts, pairs, E0, mode="lora", mrl=False, hardneg=True, epochs=1, bs=32,
                  lr=2e-4, tau=0.05, seed=0, r=16, val_ids=None, val_fn=None, max_steps=None, deadline=None, log=print):
    """Contrastive LoRA/DoRA training with the encoder in the loop.

    After every epoch only the articles needed for validation are re-encoded (``val_ids``);
    epoch 0 (= the frozen embedding) competes, and the adapter weights of the best epoch are
    restored before one final pass over *all* articles.  Returns ``(E_final, info)``."""
    torch.manual_seed(seed)
    rng = np.random.default_rng(seed)
    n_train = enc.attach_peft(mode, r)
    log(f"      [{mode} r={r}] trainable params: {n_train:,}")
    named = [(n, p) for n, p in enc.model.named_parameters() if p.requires_grad]
    params = [p for _, p in named]
    opt = torch.optim.AdamW(params, lr=lr, weight_decay=0.0)
    scaler = torch.amp.GradScaler("cuda", enabled=enc.amp)
    A, P, N = pairs
    bs = fit_encoder_batch(enc, texts, pairs, bs, hardneg, params, log)
    dims = mrl_dims(E0.shape[1]) if mrl else None
    best_score, best_ep, best_state = (val_fn(E0) if val_fn else -math.inf), 0, None
    step, t0 = 0, time.time()
    for ep in range(1, epochs + 1):
        enc.model.train()
        perm = rng.permutation(len(A))
        tot, nb = 0.0, 0
        for s in range(0, len(A), bs):
            ix = perm[s:s + bs]
            ids = np.concatenate([A[ix], P[ix]] + ([N[ix].ravel()] if hardneg else []))
            uniq, inv = np.unique(ids, return_inverse=True)
            z = enc.embed([texts[i] for i in uniq])
            inv = torch.from_numpy(inv).to(z.device)
            b = len(ix)
            a, p = z[inv[:b]], z[inv[b:2 * b]]
            n = z[inv[2 * b:]].view(b, -1, z.shape[-1]) if hardneg else None
            loss = contrastive_loss(a, p, torch.from_numpy(P[ix]).to(z.device), n, tau, dims)
            opt.zero_grad()
            scaler.scale(loss).backward()
            scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(params, 1.0)
            scaler.step(opt)
            scaler.update()
            tot += loss.item(); nb += 1; step += 1
            if step % 100 == 0:
                log(f"        step {step} loss={tot / nb:.4f} ({time.time() - t0:.0f}s)")
            if max_steps and step >= max_steps:
                break
            if deadline is not None and time.time() > deadline:
                log(f"      [{mode}] deadline reached at step {step}")
                break
        if val_fn is not None and val_ids is not None:
            Ea = E0.copy()
            Ea[val_ids] = enc.encode([texts[i] for i in val_ids])
            score = val_fn(Ea)
        else:
            score = float(ep)                                              # no validation signal: keep the last epoch
        log(f"      [{mode}] epoch {ep} loss={tot / max(nb, 1):.4f} val={score:.4f}")
        if score > best_score:
            best_score, best_ep = score, ep
            best_state = {n: p.detach().clone() for n, p in named}
        if (max_steps and step >= max_steps) or (deadline is not None and time.time() > deadline):
            break
    info = {"best_epoch": best_ep, "val": float(best_score), "steps": step, "trainable": n_train}
    if best_ep == 0:
        return E0.copy(), info                                             # adaptation did not beat frozen on validation
    with torch.no_grad():
        for n, p in named:
            p.copy_(best_state[n])
    E = E0.copy()
    E[1:] = enc.encode(texts[1:])
    return E, info


In [ ]:
%%writefile fields.py
"""More of the data for the frozen-vector techniques (stage ``moredata``).

What MIND-small's ``news.tsv`` holds per article: category, subcategory, title, abstract, URL, and the Wikidata entities found in the
title and in the abstract.  The release has **no article body** (the URL points to a page the benchmark does not ship), so the
abstract is the longest text there is.  The suite has always embedded ``title. abstract`` (``meta['text']``); what it never fed to an
encoder is the category / subcategory phrase, the entity names, or more than a reader's last 50 clicks.  This module builds those
variants so that each one can be measured on its own against the suite's text:

  t      title only                                    (the difference to ``ta`` is what the abstract adds)
  ta     title. abstract                               (the suite's own text: the control)
  cta    Category: c. Subcategory: s. title. abstract
  ctae   cta + Entities: e1; e2; ...
  views  title, abstract and category phrase embedded *separately* and concatenated with weights (NAML-style view fusion with
         frozen vectors): <u, v> = sum_v w_v <u_v, v_v>, so mean-pooling and late interaction both work on the fused table.
"""
from __future__ import annotations

import hashlib
import os
import re
import time

import numpy as np
import torch

import adapt

MODES = ("t", "ta", "cta", "ctae")
# (title, abstract, category) weights of the view fusion; they sum to 1, so the fused vector has unit norm
VIEW_WEIGHTS = ((0.5, 0.5, 0.0), (0.4, 0.4, 0.2), (0.3, 0.5, 0.2), (0.5, 0.3, 0.2), (0.35, 0.35, 0.3))
MAX_ENTITIES = 8


# ------------------------------------------------------------------------------------------- text variants
def phrase(s):
    """'football_nfl' -> 'football nfl' (MIND subcategories are snake/kebab-case; some, like 'newsworld', are not separated at all)."""
    return re.sub(r"[_\-]+", " ", (s or "").strip()).strip()


def category_phrase(cat, sub):
    cat, sub = phrase(cat), phrase(sub)
    if not cat:
        return ""
    return f"Category: {cat}. Subcategory: {sub}." if sub and sub != cat else f"Category: {cat}."


def entity_phrase(labels, top=MAX_ENTITIES):
    return f"Entities: {'; '.join(labels[:top])}." if labels else ""


def _join(*parts):
    return " ".join(p for p in parts if p).strip()


def build_texts(meta, mode):
    """One string per article (same order as ``meta['ids']``)."""
    if mode == "t":
        return [t.strip() for t in meta["titles"]]
    if mode == "ta":
        return list(meta["text"])
    cat = [category_phrase(c, s) for c, s in zip(meta["cats"], meta["subcats"])]
    cta = [_join(c, text) for c, text in zip(cat, meta["text"])]
    if mode == "cta":
        return cta
    if mode == "ctae":
        return [_join(x, entity_phrase(ls)) for x, ls in zip(cta, meta["ent_labels"])]
    raise ValueError(f"unknown text mode {mode!r}; choose from {MODES + ('views',)}")


def view_texts(meta):
    """The three views of ``views``: title, abstract (the title again when the abstract is empty, so no vector is junk) and category phrase."""
    t = [x.strip() for x in meta["titles"]]
    a = [(x.strip() or tt) for x, tt in zip(meta["abstracts"], t)]
    c = [category_phrase(cc, s) or "Category: unknown." for cc, s in zip(meta["cats"], meta["subcats"])]
    return {"t": t, "a": a, "c": c}


def combine(parts, w):
    """Concatenate unit-norm view tables [n, d] scaled by sqrt(weight): the dot product of two fused vectors is the weighted sum of the
    per-view dot products.  Works on numpy arrays and torch tensors."""
    sq = [float(np.sqrt(x)) for x in w]
    if isinstance(parts[0], torch.Tensor):
        return torch.cat([s * p for s, p in zip(sq, parts)], dim=1)
    return np.concatenate([s * p for s, p in zip(sq, parts)], axis=1).astype(np.float32)


def audit(meta, data):
    """Facts about what the fields hold (logged by the stage, so a surprising result can be traced to the data)."""
    A = meta["abstracts"]
    hist_len = np.array([len(r[2]) for r in data.test])
    return {"articles": len(A), "news_columns": int(meta.get("n_cols", 0)),
            "abstract_empty": float(np.mean([not a.strip() for a in A])),
            "title_words": float(np.mean([len(t.split()) for t in meta["titles"]])),
            "abstract_words": float(np.mean([len(a.split()) for a in A])),
            "with_entity_labels": float(np.mean([bool(x) for x in meta["ent_labels"]])),
            "entities_per_article": float(np.mean([len(x) for x in meta["ent_labels"]])),
            "categories": len(set(meta["cats"])), "subcategories": len(set(meta["subcats"])),
            "test_history_mean": float(hist_len.mean()), "test_history_over_50": float((hist_len > 50).mean()),
            "test_history_max": int(hist_len.max())}


# ------------------------------------------------------------------------------------------- embedding with a content-keyed cache
def digest(texts):
    h = hashlib.sha1()
    for t in texts:
        h.update(t.encode("utf-8", "ignore"))
        h.update(b"\x1f")
    return h.hexdigest()[:16]


def embed_texts(texts, name, max_len, prefix, device, bs, cache_dir, dtype=torch.float32):
    """Frozen vectors ``[len(texts), d]``, cached on disk per (model, max_len, prefix, **the texts themselves**): a different text variant
    can never be served from another variant's cache.  Returns ``(vecs, seconds)`` (0 if cached)."""
    key = f"{name}|{max_len}|{prefix}|{len(texts)}|{digest(texts)}"
    path = os.path.join(cache_dir, "emb_cache_" + hashlib.sha1(key.encode()).hexdigest()[:12] + ".npz")
    if os.path.exists(path):
        z = np.load(path)
        if str(z["key"]) == key:
            return z["vecs"], 0.0
    enc = adapt.HFEncoder(name, max_len=max_len, device=device, prefix=prefix, dtype=dtype)
    t0 = time.time()
    vecs = enc.encode(texts, bs=bs)
    secs = time.time() - t0
    del enc
    if device == "cuda":
        torch.cuda.empty_cache()
    if not np.isfinite(vecs).all():
        raise RuntimeError(f"{name}: non-finite embeddings (fp16 overflow?)")
    os.makedirs(cache_dir, exist_ok=True)
    np.savez(path, key=np.array(key), vecs=vecs)
    return vecs, secs


def embed_unique(embed, texts):
    """Embed each distinct string once (the category phrases are ~270 distinct strings for 65k articles) and broadcast."""
    uniq = sorted(set(texts))
    vecs = embed(uniq)
    index = {t: i for i, t in enumerate(uniq)}
    return vecs[[index[t] for t in texts]]


In [ ]:
%%writefile pooling.py
"""H8: user pooling over frozen item vectors -- single mean vector vs late interaction / recency.

The V10 control collapses the whole history into ONE mean vector.  Multi-interest and late-interaction
(ColBERT-style) matching instead score a candidate against *individual* history items, which should help
readers with several interests.  Everything here is zero-shot (no parameters), so the comparison with the
mean-pool control isolates the aggregation rule.
"""
from __future__ import annotations

import torch


def _sims(E, b):
    """cosine/dot similarity between every candidate and every history item -> [B, C, H]."""
    return torch.einsum("bcd,bhd->bch", E[b["cand"]], E[b["hist"]])


def recency_scorer(E, lam):
    """Mean of the history with weights exp(-lam * age); age 0 = most recent click (history is chronological)."""
    def f(b):
        mask = b["hist_mask"].float()
        n = mask.sum(1, keepdim=True)
        age = (n - 1 - torch.arange(mask.shape[1], device=mask.device)[None, :]).clamp(min=0)
        w = torch.exp(-lam * age) * mask
        u = (E[b["hist"]] * w.unsqueeze(-1)).sum(1) / w.sum(1, keepdim=True).clamp(min=1e-8)
        return torch.einsum("bd,bcd->bc", u, E[b["cand"]])
    return f


def interest_means(E, b, K, iters=3):
    """Group every reader's clicked items into at most ``K`` interests with spherical k-means (cosine assignment), started
    deterministically from the most recent click and then, repeatedly, the click least similar to the centres chosen so far.
    Returns ``(means [B,K,d], share [B,K])``: the plain mean vector of each group and the share of the reader's clicks in it
    (0 for an empty group).  ``K=1`` gives exactly the mean-pool user vector."""
    h = E[b["hist"]]                                                       # [B,H,d]
    mask = b["hist_mask"]
    B, H, d = h.shape
    ar = torch.arange(B, device=h.device)
    n = mask.sum(1).clamp(min=1).to(h.dtype)
    last = (mask.long() * torch.arange(1, H + 1, device=h.device)).argmax(1)       # history is chronological and left-aligned
    cen = h.new_zeros(B, K, d)
    cen[:, 0] = h[ar, last]
    nearest = torch.einsum("bhd,bd->bh", h, cen[:, 0])                     # similarity of every click to its closest centre so far (max over centres)
    for k in range(1, K):
        far = nearest.masked_fill(~mask, float("inf")).argmin(1)
        cen[:, k] = h[ar, far]
        nearest = torch.maximum(nearest, torch.einsum("bhd,bd->bh", h, cen[:, k]))
    for _ in range(iters):
        asg = torch.einsum("bhd,bkd->bhk", h, cen).argmax(-1)
        one = torch.nn.functional.one_hot(asg, K).to(h.dtype) * mask.unsqueeze(-1).to(h.dtype)
        sums = torch.einsum("bhk,bhd->bkd", one, h)
        cen = torch.where((one.sum(1) > 0).unsqueeze(-1), torch.nn.functional.normalize(sums, dim=-1), cen)
    asg = torch.einsum("bhd,bkd->bhk", h, cen).argmax(-1)
    one = torch.nn.functional.one_hot(asg, K).to(h.dtype) * mask.unsqueeze(-1).to(h.dtype)
    cnt = one.sum(1)
    return torch.einsum("bhk,bhd->bkd", one, h) / cnt.clamp(min=1).unsqueeze(-1), cnt / n.unsqueeze(-1)


def cluster_scorer(E, K, tau, iters=3):
    """Zero-shot multi-interest: score(c) = tau * logsumexp_k( <c, mean of interest k> / tau + log share_k ).  ``K=1`` is the mean-pool
    control (exactly); ``K >= the number of clicks`` (every click its own interest) is the ``lse`` late interaction of ``late_scorer`` with
    the same ``tau`` -- so the family interpolates between the two ends that are already measured.  Readers without history score 0."""
    def f(b):
        has = b["hist_mask"].any(1)[:, None]
        mu, share = interest_means(E, b, K, iters)
        sim = torch.einsum("bcd,bkd->bck", E[b["cand"]], mu)              # [B,C,K]
        s = tau * torch.logsumexp(sim / tau + torch.log(share).unsqueeze(1), -1)
        return torch.where(has, s, torch.zeros_like(s))
    return f


def late_scorer(E, kind, param=None):
    """Late interaction: ``max`` (MaxSim), ``topk`` (mean of the k most similar history items) or
    ``lse`` (soft maximum with temperature ``param``).  Users without history score 0 (ties, like the control)."""
    def f(b):
        mask = b["hist_mask"]
        has = mask.any(1)[:, None]
        sim = _sims(E, b)
        n = mask.sum(1).clamp(min=1).float()[:, None]
        if kind == "max":
            s = sim.masked_fill(~mask[:, None, :], float("-inf")).max(-1).values
        elif kind == "topk":
            k = int(param)
            vals = sim.masked_fill(~mask[:, None, :], -1e4).topk(min(k, sim.shape[-1]), -1).values      # [B,C,k]
            cnt = n.clamp(max=k)                                                                         # items actually available
            keep = (torch.arange(vals.shape[-1], device=vals.device)[None, None, :] < cnt[:, :, None]).float()
            s = (vals * keep).sum(-1) / cnt
        elif kind == "lse":
            t = float(param)
            s = t * (torch.logsumexp(sim.masked_fill(~mask[:, None, :], float("-inf")) / t, -1) - torch.log(n))
        else:
            raise ValueError(kind)
        return torch.where(has, s, torch.zeros_like(s))
    return f


In [ ]:
%%writefile sid.py
"""H7: Semantic IDs -- a zero-shot SID profile scorer and a TIGER-style generative model used as a slate ranker.

Why a *ranker*: MIND asks us to order a **given** slate, so generative retrieval (decode the next ID) does not
apply.  The SID language model instead scores every candidate by the likelihood of its code sequence given the
reading history, ``log P(codes(c) | history)``, optionally minus the history-free ``log P(codes(c))`` (a PMI
score that removes the "popular semantic region" prior).  This is the generative-recommendation paradigm
(TIGER, NeurIPS'23 -> LETTER, SIGIR'25 -> LSIG, WWW'26) adapted to slate ranking.

Tokenizer: RQ-KMeans (residual k-means), the simple tokenizer used by recent industrial generative recommenders.
"""
from __future__ import annotations

import copy
import math
import time

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F


# --------------------------------------------------------------- tokenizer
def _assign(X, C, chunk=65536):
    c2 = (C * C).sum(1)
    out = []
    for s in range(0, X.shape[0], chunk):
        x = X[s:s + chunk]
        out.append(((x * x).sum(1, keepdim=True) - 2 * x @ C.T + c2[None, :]).argmin(1))
    return torch.cat(out)


def kmeans(X, K, iters=25, seed=0):
    """Lloyd's algorithm on a tensor; empty clusters are re-seeded from random points."""
    g = torch.Generator().manual_seed(seed)
    n = X.shape[0]
    K = min(K, n)
    C = X[torch.randperm(n, generator=g)[:K].to(X.device)].clone()
    for _ in range(iters):
        a = _assign(X, C)
        S = torch.zeros_like(C).index_add_(0, a, X)
        cnt = torch.bincount(a, minlength=K).float()
        C = S / cnt.clamp(min=1)[:, None]
        empty = cnt == 0
        if empty.any():
            C[empty] = X[torch.randint(0, n, (int(empty.sum()),), generator=g).to(X.device)]
    return C


def rq_kmeans(E, fit_mask, L=3, K=256, seed=0, device="cpu"):
    """Residual k-means: level l quantises what levels < l left over.  Codebooks are fitted on ``fit_mask`` rows
    (content-only, causal); every article gets codes by nearest-centroid.  Returns int64 [n, L]."""
    X = torch.from_numpy(np.ascontiguousarray(E, np.float32)).to(device)
    fit = torch.from_numpy(np.nonzero(fit_mask)[0]).to(device)
    R = X.clone()
    codes = torch.zeros(len(E), L, dtype=torch.long, device=device)
    for l in range(L):
        C = kmeans(R[fit], K, seed=seed + l)
        a = _assign(R, C)
        codes[:, l] = a
        R = R - C[a]
    return codes.cpu().numpy().astype(np.int64)


def prefix_ids(codes, K):
    """Integer id of the length-l code prefix for l = 1..L -> int64 [L, n]."""
    out, p = [], np.zeros(len(codes), np.int64)
    for l in range(codes.shape[1]):
        p = p * K + codes[:, l]
        out.append(p.copy())
    return np.stack(out)


def sid_profile_scorer(codes, K, levels, device="cpu"):
    """Zero-shot: score = sum over the chosen prefix levels of the share of the history that falls in the
    candidate's code prefix (coarse levels generalise to cold articles, fine levels are specific)."""
    pref = torch.from_numpy(prefix_ids(codes, K)).to(device)

    def f(b):
        mask = b["hist_mask"]
        n = mask.sum(1).clamp(min=1).float()[:, None]
        s = 0.0
        for l in levels:
            ph, pc = pref[l - 1][b["hist"]], pref[l - 1][b["cand"]]
            s = s + ((pc[:, :, None] == ph[:, None, :]) & mask[:, None, :]).float().sum(-1) / n
        return s
    return f


# ------------------------------------------------------------------- model
class _Block(nn.Module):
    def __init__(self, d, heads, drop):
        super().__init__()
        self.ln1, self.ln2 = nn.LayerNorm(d), nn.LayerNorm(d)
        self.qkv, self.proj = nn.Linear(d, 3 * d), nn.Linear(d, d)
        self.mlp = nn.Sequential(nn.Linear(d, 4 * d), nn.GELU(), nn.Linear(4 * d, d))
        self.heads, self.drop = heads, drop

    def forward(self, x, allow):
        B, T, D = x.shape
        q, k, v = self.qkv(self.ln1(x)).view(B, T, 3, self.heads, D // self.heads).permute(2, 0, 3, 1, 4)
        a = F.scaled_dot_product_attention(q, k, v, attn_mask=allow, dropout_p=self.drop if self.training else 0.0)
        x = x + self.proj(a.transpose(1, 2).reshape(B, T, D))
        return x + self.mlp(self.ln2(x))


class SIDGPT(nn.Module):
    """Causal transformer over [BOS] item_1 codes ... item_n codes (left padded).  Positions are counted over
    valid tokens only, so the output does not depend on how much padding precedes the sequence."""

    def __init__(self, K, L, d=256, layers=4, heads=4, max_items=32, drop=0.1):
        super().__init__()
        self.K, self.L = K, L
        self.PAD, self.BOS = L * K, L * K + 1
        self.tok = nn.Embedding(L * K + 2, d)
        self.pos = nn.Embedding(1 + max_items * L, d)
        self.lvl = nn.Embedding(L + 1, d)
        self.blocks = nn.ModuleList(_Block(d, heads, drop) for _ in range(layers))
        self.ln, self.head = nn.LayerNorm(d), nn.Linear(d, L * K, bias=False)

    def forward(self, tokens, last_n=None):
        valid = tokens != self.PAD
        pos = (valid.long().cumsum(1) - 1).clamp(min=0)
        lvl = torch.where(tokens < self.L * self.K, tokens // self.K, torch.full_like(tokens, self.L))
        x = self.tok(tokens) + self.pos(pos) + self.lvl(lvl)
        T = tokens.shape[1]
        allow = torch.ones(T, T, dtype=torch.bool, device=tokens.device).tril()[None, None] & valid[:, None, None, :]
        allow = allow | torch.eye(T, dtype=torch.bool, device=tokens.device)[None, None]   # no fully-masked rows
        for blk in self.blocks:
            x = blk(x, allow)
        x = self.ln(x if last_n is None else x[:, -last_n:])
        return self.head(x)                                     # [B, T or last_n, L*K] logits over code tokens

    def code_logprob(self, logits, targets):
        """log P(target token) with the softmax restricted to the target's own level (constrained decoding).
        ``targets``: code tokens (level*K + code), same leading shape as ``logits[..., 0]``."""
        lvl, code = targets // self.K, targets % self.K
        lp = F.log_softmax(logits.view(*logits.shape[:-1], self.L, self.K), dim=-1)          # [..., L, K]
        sel = lp.gather(-2, lvl[..., None, None].expand(*lvl.shape, 1, self.K)).squeeze(-2)  # [..., K]
        return sel.gather(-1, code[..., None]).squeeze(-1)


# ---------------------------------------------------------------- sequences
def right_align(hist, mask, n_max):
    """Keep the last ``n_max`` valid history items, right aligned (padding on the left)."""
    n = mask.sum(1)
    take = n.clamp(max=n_max)
    j = torch.arange(n_max, device=hist.device)[None, :]
    valid = j >= (n_max - take)[:, None]
    src = ((n - take)[:, None] + (j - (n_max - take)[:, None])).clamp(0, hist.shape[1] - 1)
    return hist.gather(1, src) * valid, valid


def item_tokens(codes_t, items, valid, K, L):
    """[..., N] article ids -> [..., N*L] code tokens (PAD where the item is not valid)."""
    off = torch.arange(L, device=items.device) * K
    tok = codes_t[items] + off                                    # [..., N, L]
    tok = torch.where(valid[..., None], tok, torch.full_like(tok, L * K))
    return tok.flatten(-2)


def with_bos(tok, valid_items, L, bos, pad):
    """[..., N*L] code tokens (PAD where an item is invalid, invalid items on the left) -> [..., 1+N*L] where BOS sits
    immediately before the first valid token: [PAD..., BOS, item_1 codes, ...].  Putting BOS anywhere else would make
    the position that predicts the first code a padding position."""
    pad_len = (~valid_items).sum(-1) * L
    full = torch.cat([torch.full_like(tok[..., :1], pad), tok], -1)
    return full.scatter(-1, pad_len[..., None], torch.full_like(pad_len[..., None], bos))


def make_training_arrays(rows, n_max):
    """(history items right aligned, target) from train_core rows ``(user, hist, pos, negs)``."""
    H = np.zeros((len(rows), n_max), np.int64)
    M = np.zeros((len(rows), n_max), bool)
    Y = np.zeros(len(rows), np.int64)
    for i, (_u, hist, pos, _n) in enumerate(rows):
        h = list(hist)[-n_max:]
        if h:
            H[i, n_max - len(h):], M[i, n_max - len(h):] = h, True
        Y[i] = pos
    keep = M.any(1)
    return H[keep], M[keep], Y[keep]


def train_sid_gpt(codes, K, rows, n_max=30, epochs=3, bs=128, lr=1e-3, hist_w=0.5, seed=0, device="cpu",
                  val_fn=None, d=256, layers=4, heads=4, max_steps=None, deadline=None, log=print):
    """Next-code-token language modelling over reading sequences (history items weighted ``hist_w``, the
    clicked target 1.0).  Validation (``val_fn(model) -> float``, higher is better) picks the epoch."""
    torch.manual_seed(seed)
    rng = np.random.default_rng(seed)
    L = codes.shape[1]
    codes_t = torch.from_numpy(codes).to(device)
    model = SIDGPT(K, L, d, layers, heads, max_items=n_max + 2).to(device)
    H, M, Y = make_training_arrays(rows, n_max)
    total = max(1, epochs * math.ceil(len(H) / bs))
    if max_steps:
        total = min(total, max_steps)
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=0.01)
    sched = torch.optim.lr_scheduler.LambdaLR(opt, lambda s: min(1.0, (s + 1) / 100) * 0.5 * (1 + math.cos(math.pi * min(s, total) / total)))
    amp = device == "cuda"
    scaler = torch.amp.GradScaler("cuda", enabled=amp)
    best_score, best_state, best_ep, step = (val_fn(model) if val_fn else -math.inf), None, 0, 0
    w_item = torch.cat([torch.full((n_max * L,), hist_w), torch.ones(L)]).to(device)            # weight per code token
    for ep in range(1, epochs + 1):
        model.train()
        perm = rng.permutation(len(H))
        tot, nb = 0.0, 0
        for s in range(0, len(H), bs):
            ix = perm[s:s + bs]
            h, m = torch.from_numpy(H[ix]).to(device), torch.from_numpy(M[ix]).to(device)
            y = torch.from_numpy(Y[ix]).to(device)
            items = torch.cat([h, y[:, None]], 1)
            valid = torch.cat([m, torch.ones(len(ix), 1, dtype=torch.bool, device=device)], 1)
            tok = item_tokens(codes_t, items, valid, K, L)                                          # [B, (n+1)*L]
            seq = with_bos(tok, valid, L, model.BOS, model.PAD)
            with torch.autocast("cuda", dtype=torch.float16, enabled=amp):
                logits = model(seq[:, :-1])
            tgt = seq[:, 1:]
            ok = tgt != model.PAD
            lp = model.code_logprob(logits.float(), tgt.clamp(max=L * K - 1))
            w = w_item[None, :] * ok
            loss = -(lp * w).sum() / w.sum().clamp(min=1)
            opt.zero_grad()
            scaler.scale(loss).backward()
            scaler.unscale_(opt)
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(opt); scaler.update(); sched.step()
            tot += loss.item(); nb += 1; step += 1
            if step % 200 == 0:
                log(f"        sid-gpt step {step}/{total} loss={tot / nb:.4f}")
            if (max_steps and step >= max_steps) or (deadline is not None and time.time() > deadline):
                break
        score = val_fn(model) if val_fn else float(ep)
        log(f"      [sid-gpt] epoch {ep} loss={tot / max(nb, 1):.4f} val={score:.4f}")
        if score > best_score:
            best_score, best_ep, best_state = score, ep, copy.deepcopy(model.state_dict())
        if (max_steps and step >= max_steps) or (deadline is not None and time.time() > deadline):
            break
    if best_state is not None:
        model.load_state_dict(best_state)
    model.eval()
    return model, {"best_epoch": best_ep, "val": float(best_score), "steps": step}


# ------------------------------------------------------------------ scoring
@torch.no_grad()
def history_free_logprob(model, codes_t, n_items, bs=4096):
    """log P(codes(article) | empty history) for every article -> [n]."""
    L, K, dev = model.L, model.K, codes_t.device
    out = torch.zeros(n_items, device=dev)
    for s in range(0, n_items, bs):
        c = codes_t[s:s + bs]
        seq = torch.cat([torch.full((len(c), 1), model.BOS, device=dev), c + torch.arange(L, device=dev) * K], 1)
        lp = model.code_logprob(model(seq[:, :-1]).float(), seq[:, 1:])
        out[s:s + bs] = lp.sum(1)
    return out


@torch.no_grad()
def sid_scorer(model, codes, n_max=30, pmi=False, rows_budget=2048, device="cpu"):
    """Slate scorer: ``log P(codes(c) | history)`` (minus the history-free log-prob when ``pmi``)."""
    L, K = model.L, model.K
    codes_t = torch.from_numpy(codes).to(device)
    base = history_free_logprob(model, codes_t, len(codes)) if pmi else None
    amp = device == "cuda"

    @torch.no_grad()
    def f(b):
        model.eval()
        B, C = b["cand"].shape
        items, valid = right_align(b["hist"], b["hist_mask"], n_max)
        out = torch.empty(B, C, device=device)
        step = max(1, rows_budget // C)
        for s in range(0, B, step):
            sl = slice(s, s + step)
            n = items[sl].shape[0]
            ht = with_bos(item_tokens(codes_t, items[sl], valid[sl], K, L), valid[sl], L, model.BOS, model.PAD)  # [n, 1+n_max*L]
            ct = codes_t[b["cand"][sl]] + torch.arange(L, device=device) * K         # [n, C, L]
            seq = torch.cat([ht[:, None, :].expand(n, C, ht.shape[1]), ct], -1).view(n * C, -1)
            with torch.autocast("cuda", dtype=torch.float16, enabled=amp):
                logits = model(seq[:, :-1], last_n=L)
            lp = model.code_logprob(logits.float(), seq[:, -L:]).sum(-1).view(n, C)
            out[sl] = lp
        return out - base[b["cand"]] if pmi else out
    return f


In [ ]:
%%writefile llm_user.py
"""H5: a fine-tuned LLM as the *user encoder* (history-as-text), with causal / bidirectional / soft attention masks.

The reader's last K headlines become one text; a (decoder-only) LLM with LoRA/DoRA maps it to a vector that is
trained, contrastively, to be close to the embedding of the article the reader clicked next (frozen document
tower, in-batch + impression hard negatives).  Compared with H1 (the same model, zero-shot) this isolates what
*learning to read a history* adds.

Attention-mask study (arXiv 2602.10622, "How Do Decoder-Only LLMs Perceive Users?", Ant Group, 2/2026): a decoder-only LLM
reads the history causally, so early headlines never see later ones.  Three variants are trained with the same recipe:

  causal  the model's native mask                                     (last-token pooling)
  bidir   full bidirectional attention                                (LLM2Vec-style; 4D mask built here)
  soft    future positions are down-weighted by an additive ``log(lambda)`` bias, lambda ramped 0 -> 1 during
          the first half of training (linear causal -> bidirectional scheduler); evaluated at lambda = 1.

``soft`` is only the *scheduler* ingredient of that paper.  Its Gradient-Guided Soft Masking (a gradient-based
pre-warmup computed from a frozen left-tower encoder) and its hybrid mask are NOT implemented here.

Pooling is the last token in all three, so the mask is the only thing that differs.  Whether the loaded model
honours a custom 4D mask is checked at run time (``probe_masks``); a variant whose probe fails is skipped.
"""
from __future__ import annotations

import math
import time

import numpy as np
import torch
import torch.nn.functional as F

import adapt
from common import fix_peft_torchao, load_hf, log

MODES = ("causal", "bidir", "soft")


def build_mask(attn, mode, lam, dtype):
    """Additive 4D attention mask ``[B,1,T,T]`` for left-padded batches (``attn``: [B,T] 1 = real token).

    Padding keys are always masked; padding *queries* may attend to themselves so that no row is fully masked
    (their outputs are never used).  ``soft``: future keys stay visible but get ``log(lam)`` added."""
    B, T = attn.shape
    dev = attn.device
    valid = attn.bool()[:, None, None, :]                                      # [B,1,1,T] key is a real token
    tril = torch.ones(T, T, dtype=torch.bool, device=dev).tril()[None, None]   # key j <= query i
    eye = torch.eye(T, dtype=torch.bool, device=dev)[None, None]
    if mode == "causal":
        allow = (tril & valid) | eye
    else:
        allow = valid.expand(B, 1, T, T) | eye
    bias = torch.zeros(B, 1, T, T, dtype=torch.float32, device=dev)
    if mode == "soft" and lam <= 0.0:                                           # lambda = 0 is exactly causal
        allow = (tril & valid) | eye
    elif mode == "soft" and lam < 1.0:
        bias = bias + torch.where(tril, 0.0, math.log(max(lam, 1e-9)))
    neg = torch.finfo(dtype).min
    return torch.where(allow, bias, torch.full_like(bias, neg)).to(dtype)


def peft_targets(model):
    names = {n.split(".")[-1] for n, _ in model.named_modules()}
    if "q_proj" in names:
        return [t for t in ("q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj") if t in names]
    return [t for t in ("query", "key", "value") if t in names]


class UserTower:
    """LoRA/DoRA-adapted language model that maps a history text to one vector."""

    def __init__(self, name, mode="causal", prefix="", max_len=256, device="cpu", dtype=torch.float32,
                 trust_remote_code=False, grad_ckpt=True):
        from transformers import AutoModel, AutoTokenizer
        self.name, self.prefix, self.max_len, self.device = name, prefix, max_len, device
        self.pooling = adapt.guess_pooling(name)
        self.decoder = self.pooling == "last"
        self.mode = mode if self.decoder else "native"
        self.tok = AutoTokenizer.from_pretrained(name, trust_remote_code=trust_remote_code)
        if self.decoder:
            self.tok.padding_side = "left"
        self.model = load_hf(AutoModel, name, dtype, trust_remote_code).to(device)
        self.amp = device == "cuda" and dtype == torch.float32          # fp32 master weights + fp16 autocast
        # a float attention mask must match the dtype of q/k/v: fp16 under autocast, otherwise the weights' dtype
        self.mask_dtype = torch.float16 if self.amp else self.model.dtype
        self.grad_ckpt = grad_ckpt
        self.lam = 1.0
        self.peft = False

    # -- adaptation
    def attach_peft(self, kind="lora", r=16, dropout=0.05):
        from peft import LoraConfig, get_peft_model
        fix_peft_torchao()
        cfg = LoraConfig(r=r, lora_alpha=2 * r, lora_dropout=dropout, target_modules=peft_targets(self.model),
                         use_dora=(kind == "dora"), bias="none")
        self.model = get_peft_model(self.model, cfg)
        self.peft = True
        if self.grad_ckpt:
            try:
                self.model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})
            except Exception:                                     # some wrappers expose it on the base model only
                self.model.base_model.model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})
        return sum(p.numel() for p in self.model.parameters() if p.requires_grad)

    def trainable(self):
        return [(n, p) for n, p in self.model.named_parameters() if p.requires_grad]

    # -- forward
    def _tokenize(self, texts):
        return self.tok([self.prefix + t for t in texts], padding=True, truncation=True, max_length=self.max_len,
                        return_tensors="pt").to(self.device)

    def _hidden(self, batch, mode, lam):
        ids, attn = batch["input_ids"], batch["attention_mask"]
        kw = {"input_ids": ids}
        if "token_type_ids" in batch:
            kw["token_type_ids"] = batch["token_type_ids"]
        if self.decoder and mode != "causal":
            kw["attention_mask"] = build_mask(attn, mode, lam, self.mask_dtype)
        else:
            kw["attention_mask"] = attn
        if self.decoder:
            kw["use_cache"] = False
        return self.model(**kw).last_hidden_state

    def embed(self, texts, mode=None, lam=None):
        batch = self._tokenize(texts)
        mode = mode or self.mode
        lam = self.lam if lam is None else lam
        with torch.autocast("cuda", dtype=torch.float16, enabled=self.amp):
            h = self._hidden(batch, mode, lam)
        if self.pooling == "cls":
            z = h[:, 0]
        elif self.pooling == "last":
            z = h[:, -1]
        else:
            m = batch["attention_mask"].unsqueeze(-1).to(h.dtype)
            z = (h * m).sum(1) / m.sum(1).clamp(min=1)
        return z.float()

    @torch.no_grad()
    def encode(self, texts, bs=64, mode=None, lam=1.0, deadline=None):
        """Normalised vectors, sorted by length to keep padding small.  Un-encoded rows (deadline hit) stay zero."""
        was = self.model.training
        self.model.eval()
        order = np.argsort([len(t) for t in texts])
        out = None
        for s in range(0, len(texts), bs):
            if deadline is not None and time.time() > deadline:
                log("      [tower] deadline reached while encoding - remaining queries left at zero")
                break
            ix = order[s:s + bs]
            z = F.normalize(self.embed([texts[i] for i in ix], mode, lam), dim=-1).cpu().numpy()
            if out is None:
                out = np.zeros((len(texts), z.shape[1]), np.float32)
            out[ix] = z
        self.model.train(was)
        return out if out is not None else np.zeros((len(texts), 1), np.float32)


@torch.no_grad()
def probe_masks(tower, texts, tol=5e-3):
    """Does the loaded model honour a custom 4D attention mask?  Returns ``(ok, report)``.

    1. our 4D causal mask reproduces the model's native 2D-mask path (so building the mask by hand is correct);
    2. the bidirectional mask changes the output (it is not silently ignored);
    3. ``soft`` at lambda=1 equals ``bidir`` and at lambda->0 equals ``causal``."""
    was = tower.model.training
    tower.model.eval()
    batch = tower._tokenize(texts)

    def pooled(mode, lam=1.0, custom_causal=False):
        with torch.autocast("cuda", dtype=torch.float16, enabled=tower.amp):
            if custom_causal:
                ids, attn = batch["input_ids"], batch["attention_mask"]
                h = tower.model(input_ids=ids, attention_mask=build_mask(attn, "causal", 1.0, tower.mask_dtype),
                                use_cache=False).last_hidden_state
            else:
                h = tower._hidden(batch, mode, lam)
        return F.normalize(h[:, -1].float(), dim=-1)

    rep = {}
    try:
        native = pooled("causal")
        rep["causal_4d_vs_native"] = float((pooled("causal", custom_causal=True) - native).abs().max())
        bidir = pooled("bidir")
        rep["bidir_vs_causal"] = float((bidir - native).abs().max())
        rep["soft1_vs_bidir"] = float((pooled("soft", 1.0) - bidir).abs().max())
        rep["soft0_vs_causal"] = float((pooled("soft", 1e-6) - native).abs().max())
    except Exception as e:                                          # the model rejects 4D masks altogether
        tower.model.train(was)
        return False, {"error": repr(e)[:200]}
    tower.model.train(was)
    return probe_ok(rep, tol), rep


def probe_ok(rep, tol=5e-3):
    """Pass/fail of a probe report.  ``soft0_vs_causal`` (lambda = 1e-6) is informational only: real LLMs put huge attention logits on a
    few special tokens (e.g. the final EOS), so a -13.8 bias does not make future tokens negligible even though the mask code is right
    (the first Kaggle run measured 0.099 there while causal-4D == native and soft(1) == bidir were exact)."""
    return bool(rep["causal_4d_vs_native"] < tol and rep["bidir_vs_causal"] > 1e-6 and rep["soft1_vs_bidir"] < tol)


def shared_negative_loss(z, E_doc, pos, negs, tau=0.05):
    """InfoNCE of each query against ALL positives and ALL impression negatives of the batch (``B + B*K`` candidates) instead of
    only its own K negatives.  The document tower is frozen, so the extra negatives cost nothing.  A candidate that is the same
    article as a row's positive (but sits in another column) is masked: it would be a false negative."""
    a = F.normalize(z, dim=-1)
    ids = torch.cat([pos, negs.reshape(-1)])
    C = F.normalize(E_doc[ids], dim=-1)
    logits = a @ C.T / tau
    B = len(pos)
    ar = torch.arange(B, device=z.device)
    same = ids[None, :] == pos[:, None]
    same[ar, ar] = False
    return F.cross_entropy(logits.masked_fill(same, float("-inf")), ar)


def fit_batch_size(tower, texts, bs, params, log=log):
    """Dry-run forward+backward on the longest queries; halve the batch while the GPU runs out of memory
    (a failure 30 minutes into training would cost the whole variant)."""
    longest = sorted(texts, key=len)[-bs:]
    while bs >= 2:
        try:
            tower.model.train()
            z = tower.embed(longest[:bs])
            z.float().pow(2).mean().backward()
            for p in params:
                p.grad = None
            return bs
        except RuntimeError as e:
            if "out of memory" not in str(e).lower():
                raise
            for p in params:
                p.grad = None
            if tower.device == "cuda":
                torch.cuda.empty_cache()
            bs //= 2
            log(f"      [tower] out of memory on the longest queries -> batch size {bs}")
    raise RuntimeError("cannot fit even a batch of 2 queries on the device")


def train_user_tower(tower, query_fn, rows, E_doc, kind="lora", steps=1000, bs=16, lr=1e-4, tau=0.05, r=16,
                     seed=0, eval_fn=None, eval_at=(0.33, 0.67, 1.0), ramp=0.5, warmup=50, shared_negs=True, deadline=None, log=log):
    """Contrastive fine-tuning of the user tower.

    ``query_fn(hist) -> str``: history ids -> query text;  ``rows``: train_core ``(user, hist, pos, negs)``;
    ``E_doc``: frozen document vectors [n_news, d] (torch, on the device).  ``eval_fn(tower) -> float`` (higher is better,
    typically validation nDCG@10) is called at the fractions ``eval_at`` of training and may leave anything it computed
    in ``tower.eval_payload``; the weights (and payload) of the best evaluation are restored/returned.  Returns ``info``."""
    torch.manual_seed(seed)
    rng = np.random.default_rng(seed)
    n_train = tower.attach_peft(kind, r)
    log(f"      [{tower.mode}/{kind} r={r}] trainable params: {n_train:,}")
    named = tower.trainable()
    params = [p for _, p in named]
    opt = torch.optim.AdamW(params, lr=lr, weight_decay=0.0)
    sched = torch.optim.lr_scheduler.LambdaLR(
        opt, lambda s: min(1.0, (s + 1) / max(warmup, 1)) * 0.5 * (1 + math.cos(math.pi * min(s, steps) / steps)))
    scaler = torch.amp.GradScaler("cuda", enabled=tower.amp)
    usable = [i for i, rw in enumerate(rows) if len(rw[1])]
    order = rng.permutation(usable)
    order = np.concatenate([order] * math.ceil(steps * bs / max(len(order), 1)))[:steps * bs]
    sample = [query_fn(rows[i][1]) for i in order[:min(len(order), 4000)]]
    bs = fit_batch_size(tower, sample, bs, params, log)
    order = np.concatenate([order] * math.ceil(steps * bs / max(len(order), 1)))[:steps * bs]
    marks = sorted({max(1, int(round(f * steps))) for f in eval_at})
    best = (-math.inf, None, 0, None)
    tot, nb, first, t0, done, losses, bad = 0.0, 0, None, time.time(), 0, [], 0
    dims = None
    tower.model.train()
    for step in range(1, steps + 1):
        if deadline is not None and time.time() > deadline:
            log(f"      [tower] deadline reached at step {step - 1}/{steps}")
            break
        tower.lam = min(1.0, (step - 1) / max(ramp * steps, 1)) if tower.mode == "soft" else 1.0     # step 1 is exactly causal
        ix = order[(step - 1) * bs:step * bs]
        sel = [rows[i] for i in ix]
        z = tower.embed([query_fn(rw[1]) for rw in sel])
        pos = torch.as_tensor([rw[2] for rw in sel], device=z.device)
        negs = torch.as_tensor(np.stack([np.asarray(rw[3], np.int64) for rw in sel]), device=z.device)
        loss = (shared_negative_loss(z, E_doc, pos, negs, tau) if shared_negs
                else adapt.contrastive_loss(z, E_doc[pos], pos, E_doc[negs], tau, dims))
        if not torch.isfinite(loss):                                   # fp16 overflow in the LLM: skip the batch, abort if persistent
            bad += 1
            if bad > 5:
                raise FloatingPointError(f"non-finite loss in {bad} steps (fp16 overflow?) - try a smaller learning rate or fp32")
            opt.zero_grad()
            continue
        opt.zero_grad()
        scaler.scale(loss).backward()
        scaler.unscale_(opt)
        torch.nn.utils.clip_grad_norm_(params, 1.0)
        scaler.step(opt); scaler.update(); sched.step()
        tot += loss.item(); nb += 1; done = step
        losses.append(loss.item())
        first = losses[0]
        if step % 50 == 0:
            log(f"        step {step}/{steps} loss={tot / nb:.4f} lam={tower.lam:.2f} ({time.time() - t0:.0f}s)")
            tot, nb = 0.0, 0
        if eval_fn is not None and step in marks:
            tower.lam = 1.0
            score = eval_fn(tower)
            log(f"      [{tower.mode}] step {step}: val={score:.4f}")
            if score > best[0]:
                best = (score, {n: p.detach().clone() for n, p in named}, step, getattr(tower, "eval_payload", None))
            tower.model.train()
    if eval_fn is not None and done not in marks:                  # stopped early by the deadline: evaluate what we have
        tower.lam = 1.0
        score = eval_fn(tower)
        log(f"      [{tower.mode}] step {done}: val={score:.4f}")
        if score > best[0]:
            best = (score, {n: p.detach().clone() for n, p in named}, done, getattr(tower, "eval_payload", None))
    if best[1] is not None:
        with torch.no_grad():
            for n, p in named:
                p.copy_(best[1][n])
    tower.lam = 1.0
    tower.model.eval()
    return {"best_step": best[2], "val": float(best[0]) if best[1] is not None else float("nan"), "steps": done, "trainable": n_train,
            "first_loss": first, "train_s": time.time() - t0, "losses": losses, "payload": best[3]}


In [ ]:
%%writefile llm_gen.py
"""Generative-LLM utilities for the retrieval-augmented / LLM-reranking stages (H4b, H6, H9).

* ``Generator.generate``      batched greedy generation (chat template, thinking off), JSONL-cached;
* ``Generator.yes_no``        pointwise judge: ``logit(Yes) - logit(No)`` of the first answer token
                              (the pointwise zero-shot ranking prompt of the LLM-as-reranker line, e.g. LLM4Rerank WWW'25).

Both honour an absolute ``deadline`` and report which items were not processed, so a slow GPU shrinks the
experiment (the stage records ``n``) instead of killing the notebook.
"""
from __future__ import annotations

import hashlib
import json
import os
import re
import time

import numpy as np
import torch

from common import load_hf, log

_THINK = re.compile(r"<think>.*?</think>", re.S)


def clean(text):
    text = _THINK.sub("", text)
    if "<think>" in text:                                   # truncated reasoning: nothing usable
        text = text.split("<think>")[0]
    return text.strip()


class Generator:
    def __init__(self, name, device="cpu", dtype=torch.float16, cache_path=None, trust_remote_code=False):
        from transformers import AutoModelForCausalLM, AutoTokenizer
        self.name, self.device = name, device
        self.tok = AutoTokenizer.from_pretrained(name, trust_remote_code=trust_remote_code)
        self.tok.padding_side = "left"
        if self.tok.pad_token is None:
            self.tok.pad_token = self.tok.eos_token
        self.model = load_hf(AutoModelForCausalLM, name, dtype, trust_remote_code).to(device).eval()
        self.cache_path = cache_path
        self.cache = {}
        if cache_path and os.path.exists(cache_path):
            with open(cache_path, encoding="utf-8") as f:
                for line in f:
                    try:
                        r = json.loads(line)
                        self.cache[r["k"]] = r["v"]
                    except (ValueError, KeyError):
                        pass
        self.rate = {}                                       # kind -> (items, seconds) for budget planning
        yes, no = self.tok.encode("Yes", add_special_tokens=False), self.tok.encode("No", add_special_tokens=False)
        self.yes_id, self.no_id = yes[0], no[0]

    # -- prompts
    def render(self, text, system=None):
        msgs = ([{"role": "system", "content": system}] if system else []) + [{"role": "user", "content": text}]
        if getattr(self.tok, "chat_template", None):
            try:
                return self.tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True, enable_thinking=False)
            except TypeError:
                return self.tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
        return text + "\n"

    def _key(self, kind, prompt, extra=""):
        return hashlib.sha1(f"{self.name}|{kind}|{extra}|{prompt}".encode("utf-8")).hexdigest()

    def _persist(self, items):
        if not self.cache_path:
            return
        with open(self.cache_path, "a", encoding="utf-8") as f:
            for k, v in items:
                f.write(json.dumps({"k": k, "v": v}) + "\n")

    def _tok(self, rendered):
        return self.tok(rendered, return_tensors="pt", padding=True, truncation=True, max_length=1536,
                        add_special_tokens=False).to(self.device)

    def _note(self, kind, n, secs):
        a, b = self.rate.get(kind, (0, 0.0))
        self.rate[kind] = (a + n, b + secs)

    def seconds_per_item(self, kind, default=1.0):
        n, s = self.rate.get(kind, (0, 0.0))
        return s / n if n else default

    # -- generation
    @torch.no_grad()
    def generate(self, prompts, max_new_tokens=64, bs=32, system=None, deadline=None, desc="generate"):
        """``prompts``: user messages.  Returns a list of strings (``None`` where the deadline cut generation short)."""
        rendered = [self.render(p, system) for p in prompts]
        keys = [self._key("gen", r, str(max_new_tokens)) for r in rendered]
        out = [self.cache.get(k) for k in keys]
        todo = [i for i, o in enumerate(out) if o is None]
        todo.sort(key=lambda i: len(rendered[i]))
        t0, done = time.time(), 0
        for s in range(0, len(todo), bs):
            if deadline is not None and time.time() > deadline:
                log(f"      [{desc}] deadline reached with {len(todo) - s} prompts left")
                break
            ix = todo[s:s + bs]
            t1 = time.time()
            batch = self._tok([rendered[i] for i in ix])
            gen = self.model.generate(**batch, max_new_tokens=max_new_tokens, do_sample=False, temperature=None,
                                      top_p=None, top_k=None, pad_token_id=self.tok.pad_token_id)
            texts = self.tok.batch_decode(gen[:, batch["input_ids"].shape[1]:], skip_special_tokens=True)
            new = []
            for i, t in zip(ix, texts):
                out[i] = clean(t)
                self.cache[keys[i]] = out[i]
                new.append((keys[i], out[i]))
            self._persist(new)
            self._note("gen", len(ix), time.time() - t1)
            done += len(ix)
            if (s // bs) % 20 == 19:
                log(f"      [{desc}] {done}/{len(todo)} generated ({time.time() - t0:.0f}s)")
        return out

    # -- pointwise judge
    @torch.no_grad()
    def yes_no(self, prompts, bs=32, system=None, deadline=None, desc="judge"):
        """``logit(Yes) - logit(No)`` for the first answer token of each prompt (NaN where not computed)."""
        rendered = [self.render(p, system) for p in prompts]
        keys = [self._key("yn", r) for r in rendered]
        out = np.array([self.cache.get(k, np.nan) for k in keys], np.float64)
        todo = [i for i in range(len(prompts)) if np.isnan(out[i])]
        todo.sort(key=lambda i: len(rendered[i]))
        t0, base = time.time(), getattr(self.model, "model", None)
        for s in range(0, len(todo), bs):
            if deadline is not None and time.time() > deadline:
                log(f"      [{desc}] deadline reached with {len(todo) - s} prompts left")
                break
            ix = todo[s:s + bs]
            t1 = time.time()
            batch = self._tok([rendered[i] for i in ix])
            h = base(input_ids=batch["input_ids"], attention_mask=batch["attention_mask"], use_cache=False).last_hidden_state[:, -1]
            logits = self.model.lm_head(h).float()
            lo = (logits[:, self.yes_id] - logits[:, self.no_id]).cpu().numpy()
            new = []
            for i, v in zip(ix, lo):
                out[i] = float(v)
                self.cache[keys[i]] = float(v)
                new.append((keys[i], float(v)))
            self._persist(new)
            self._note("yn", len(ix), time.time() - t1)
            if (s // bs) % 50 == 49:
                log(f"      [{desc}] {s + len(ix)}/{len(todo)} judged ({time.time() - t0:.0f}s)")
        return out

    def self_test(self):
        """Is the model usable in this dtype?  fp16 overflow shows up as non-finite logits or garbage text."""
        try:
            text = self.generate(["Name one common topic of news articles, in one short sentence."], max_new_tokens=16, bs=1)[0] or ""
            lo = self.yes_no(["Is the sky blue? Answer Yes or No."], bs=1)
            letters = sum(ch.isalpha() for ch in text)
            return bool(letters >= 3 and np.isfinite(lo).all())
        except Exception as e:
            log(f"      [generator] self-test raised {repr(e)[:100]}")
            return False

    def free(self):
        del self.model
        if self.device == "cuda":
            torch.cuda.empty_cache()


# ------------------------------------------------------------------ budget planning
def plan_prefix(item_lists, cap, per_impr=0.0):
    """Largest ``n`` such that ``|union(item_lists[:n])| + per_impr * n <= cap`` -> ``(n, n_items)``.
    Impressions are taken in a fixed random order, so a smaller ``n`` is a prefix (subset) of a larger one."""
    seen, best = set(), (0, 0)
    for n, items in enumerate(item_lists, 1):
        seen.update(items)
        if len(seen) + per_impr * n > cap:
            break
        best = (n, len(seen))
    return best


In [ ]:
%%writefile refdata.py
"""Reference-recipe data for the NRMS / NAML / Fastformer baselines on the *exact* V10 split.

Why this module exists.  V10's trained baselines are small and cheap on purpose: 64-d random word vectors, the first 20
whitespace tokens of title+abstract with rare words mapped to PAD, 4 negatives drawn once for all epochs, lr 1e-3.  They score
~0.04 below published MIND-small numbers on every metric, so "beats V10's Fastformer" says little.  Here the same model families
are trained with the usual published recipe (title-only regex tokens, GloVe-300 initialisation, negatives re-drawn every epoch,
lr 1e-4) on the same train_core / validation / MINDsmall_dev split and scored by the same evaluator.

Nothing here decides *which* impressions are used on its own: the training impressions are re-derived exactly as ``data.py`` does it
and asserted against the ``MindData`` object (same day split, same number of impressions, same number of training samples).
"""
from __future__ import annotations

import glob
import io
import json
import os
import re
import zipfile
from collections import Counter
from datetime import datetime

import numpy as np
import torch

PAD, UNK = 0, 1
_WORD = re.compile(r"[\w]+|[.,!?;|]")


# ------------------------------------------------------------------------------------------- text
def tokenize(s):
    """Microsoft Recommenders' ``word_tokenize``: lower-cased words, punctuation marks as tokens of their own."""
    return _WORD.findall(s.lower()) if isinstance(s, str) else []


def build_vocab(token_lists, min_freq=2, keep=None):
    """0 = PAD, 1 = UNK, then every word seen at least ``min_freq`` times or listed in ``keep`` (words with a pre-trained vector)."""
    cnt = Counter()
    for t in token_lists:
        cnt.update(t)
    keep = keep or ()
    words = sorted((w for w, c in cnt.items() if c >= min_freq or w in keep), key=lambda w: (-cnt[w], w))
    vocab = {"<pad>": PAD, "<unk>": UNK}
    for w in words:
        vocab[w] = len(vocab)
    return vocab


def encode_tokens(token_lists, vocab, length):
    """[n_articles + 1, length] int64; row 0 is the PAD article, tokens are left-aligned."""
    out = np.zeros((len(token_lists) + 1, length), np.int64)
    for i, toks in enumerate(token_lists, 1):
        ids = [vocab.get(w, UNK) for w in toks[:length]]
        out[i, :len(ids)] = ids
    return out


def label_ids(values):
    """Category-like strings -> ids (0 = PAD article), and the table size."""
    m = {v: i for i, v in enumerate(sorted(set(values)), 1)}
    return np.array([0] + [m[v] for v in values], np.int64), len(m) + 1


# ------------------------------------------------------------------------------------------- GloVe
def _from_txt(fh, dim, want):
    words, vecs = [], []
    for line in fh:
        if isinstance(line, bytes):
            line = line.decode("utf-8", "ignore")
        w, _, rest = line.partition(" ")
        if w not in want:
            continue
        v = rest.split()
        if len(v) == dim:                                           # multi-token entries of the 840B file are skipped
            words.append(w)
            vecs.append(np.asarray(v, np.float32))
    if not words:
        raise ValueError("no overlap with the corpus vocabulary")
    return words, np.stack(vecs)


def _glove_local(path, dim, want):
    if path.endswith(".zip"):
        with zipfile.ZipFile(path) as z:
            name = next(n for n in z.namelist() if n.endswith(f".{dim}d.txt"))
            with io.TextIOWrapper(z.open(name), encoding="utf-8") as fh:
                return _from_txt(fh, dim, want)
    with open(path, encoding="utf-8") as fh:
        return _from_txt(fh, dim, want)


def _from_st_dir(root, dim, want):
    """sentence-transformers ``WordEmbeddings`` folder: a JSON holding the vocabulary list and a torch file with the [V, dim] matrix."""
    folder = os.path.join(root, "0_WordEmbeddings")
    folder = folder if os.path.isdir(folder) else root
    vocab = None
    for fn in sorted(os.listdir(folder)):
        if fn.endswith(".json"):
            with open(os.path.join(folder, fn), encoding="utf-8") as f:
                j = json.load(f)
            if isinstance(j, dict) and isinstance(j.get("vocab"), list):
                vocab = j["vocab"]
                break
    mat = None
    for fn in sorted(os.listdir(folder)):
        p = os.path.join(folder, fn)
        if fn.endswith(".safetensors"):
            from safetensors.torch import load_file
            sd = load_file(p)
        elif fn.endswith((".bin", ".pt")):
            sd = torch.load(p, map_location="cpu")
        else:
            continue
        cands = [v for v in sd.values() if getattr(v, "ndim", 0) == 2 and v.shape[1] == dim]
        if cands:
            mat = max(cands, key=lambda v: v.shape[0]).float().numpy()
            break
    if vocab is None or mat is None or len(vocab) != mat.shape[0]:
        raise ValueError(f"unrecognised word-embedding folder {folder} (vocab={None if vocab is None else len(vocab)}, "
                         f"matrix={None if mat is None else mat.shape})")
    idx = {w: i for i, w in enumerate(vocab)}
    words = [w for w in want if w in idx]
    if not words:
        raise ValueError("no overlap with the corpus vocabulary")
    return words, mat[[idx[w] for w in words]].copy()


def _glove_hf(dim, want):
    from huggingface_hub import snapshot_download
    root = snapshot_download(f"sentence-transformers/average_word_embeddings_glove.6B.{dim}d", allow_patterns=["0_WordEmbeddings/*"])
    return _from_st_dir(root, dim, want)


def _glove_gensim(dim, want):
    import gensim.downloader as api
    kv = api.load(f"glove-wiki-gigaword-{dim}")
    words = [w for w in want if w in kv.key_to_index]
    if not words:
        raise ValueError("no overlap with the corpus vocabulary")
    return words, np.stack([np.asarray(kv[w], np.float32) for w in words])


def find_local_glove(dim):
    out = []
    for pat in (f"/kaggle/input/**/glove*{dim}d*.txt", f"/kaggle/input/**/glove*{dim}d*.zip", "/kaggle/input/**/glove*6B*.zip"):
        out += sorted(glob.glob(pat, recursive=True))
    return list(dict.fromkeys(out))


def load_glove(want, dim=300, source=None, log=print):
    """GloVe vectors restricted to ``want`` (the corpus words) -> ``(words, matrix, label)``; ``(None, None, '')`` if every source fails,
    in which case the models start from random vectors and say so.  Order: ``source`` / ``$V11_GLOVE`` (txt, gz-less zip), a GloVe file
    attached to the notebook, the Hugging Face copy of GloVe-6B, gensim's downloader."""
    attempts = []
    src = source or os.environ.get("V11_GLOVE")
    if src:
        attempts.append((f"file:{src}", lambda: _glove_local(src, dim, want)))
    for p in find_local_glove(dim):
        attempts.append((f"file:{p}", lambda p=p: _glove_local(p, dim, want)))
    attempts.append(("huggingface:sentence-transformers/average_word_embeddings_glove.6B", lambda: _glove_hf(dim, want)))
    attempts.append(("gensim:glove-wiki-gigaword", lambda: _glove_gensim(dim, want)))
    for label, fn in attempts:
        try:
            words, mat = fn()
            if mat.ndim != 2 or mat.shape[1] != dim or not np.isfinite(mat).all():
                raise ValueError("malformed matrix")
            log(f"  GloVe-{dim}d <- {label}: {len(words):,} of {len(want):,} corpus words have a vector")
            return words, mat.astype(np.float32), label
        except Exception as e:                                       # a missing source is normal; try the next one
            log(f"  GloVe source {label} unavailable: {repr(e)[:110]}")
    log("  !! no GloVe source worked -> word vectors start from random (the rows are labelled '_noglove')")
    return None, None, ""


class RefText:
    """Token tables of the reference recipe (row 0 = PAD article): title (NRMS / Fastformer / NAML), abstract (NAML), category ids."""

    def __init__(self, meta, title_len=30, body_len=50, min_freq=2, glove_dim=300, use_glove=True, glove_source=None, log=print):
        titles = [tokenize(t) for t in meta["titles"]]
        bodies = [tokenize(a) for a in meta["abstracts"]]
        want = set()
        for t in titles + bodies:
            want.update(t)
        self.glove_words = self.glove_mat = None
        self.glove_label = ""
        if use_glove:
            self.glove_words, self.glove_mat, self.glove_label = load_glove(want, glove_dim, glove_source, log)
        self.glove_dim = glove_dim
        self.vocab = build_vocab(titles + bodies, min_freq, keep=set(self.glove_words) if self.glove_words else None)
        self.title = encode_tokens(titles, self.vocab, title_len)
        self.body = encode_tokens(bodies, self.vocab, body_len)
        self.cat, self.n_cat = label_ids(meta["cats"])
        self.sub, self.n_sub = label_ids(meta.get("subcats") or ["-"] * len(meta["cats"]))
        gset = set(self.glove_words) if self.glove_words else set()
        real = self.title[1:][self.title[1:] != PAD]
        self.stats = {"vocab": len(self.vocab), "glove": self.glove_label or "none", "glove_vocab_hits": sum(1 for w in self.vocab if w in gset),
                      "title_tokens": int(real.size), "unk_title_share": float((real == UNK).mean()) if real.size else 0.0}

    @property
    def vocab_size(self):
        return len(self.vocab)

    def embedding(self, dim, seed=0, init="glove"):
        """Initial word-embedding matrix [V, dim]: GloVe rows where available, otherwise N(0, s) with s matched to GloVe (0.1 without it);
        the PAD row is zero.  ``init='random'`` ignores GloVe."""
        rng = np.random.default_rng(seed)
        use = init == "glove" and self.glove_mat is not None and dim == self.glove_mat.shape[1]
        std = float(self.glove_mat.std()) if use else 0.1
        M = rng.normal(0.0, std, (len(self.vocab), dim)).astype(np.float32)
        M[PAD] = 0.0
        if use:
            gi = {w: i for i, w in enumerate(self.glove_words)}
            for w, i in self.vocab.items():
                j = gi.get(w)
                if j is not None:
                    M[i] = self.glove_mat[j]
        return M

    def tokens(self, device):
        t = lambda a: torch.from_numpy(a).to(device)
        return {"title": t(self.title), "body": t(self.body), "cat": t(self.cat), "sub": t(self.sub)}


# ------------------------------------------------------------------------------------------- behaviours / split
_TIME_FORMATS = ("%m/%d/%Y %H:%M:%S", "%m/%d/%Y %I:%M:%S %p", "%Y-%m-%d %H:%M:%S")


def _time_of(value):
    value = value.strip()
    for fmt in _TIME_FORMATS:
        try:
            return datetime.strptime(value, fmt)
        except ValueError:
            pass
    raise ValueError(f"unparseable impression time: {value!r}")


def read_behaviors(path, nid_map):
    """Same filtering as ``MindData.from_mind``: ``(user, history, candidates, labels, day)`` per impression, global article ids."""
    out = []
    with open(path, encoding="utf-8") as f:
        for line in f:
            p = line.rstrip("\n").split("\t")
            if len(p) < 5:
                continue
            _impr, user_raw, timestamp, history_raw, impressions = p[:5]
            hist = [nid_map[n] for n in history_raw.split() if n in nid_map] if history_raw else []
            cands, labs = [], []
            for item in impressions.split():
                nid, lab = item.rsplit("-", 1)
                if nid in nid_map:
                    cands.append(nid_map[nid])
                    labs.append(int(lab))
            out.append((user_raw, hist, cands, labs, _time_of(timestamp).date().isoformat()))
    return out


def split_train_core(rows, dev_ratio=0.1):
    """The chronological split of ``MindData.from_mind``: the last days holding at least ``dev_ratio`` of the impressions are validation."""
    days = sorted({r[4] for r in rows})
    target = max(1, int(np.ceil(len(rows) * dev_ratio)))
    val_days, count = [], 0
    for day in reversed(days):
        val_days.append(day)
        count += sum(r[4] == day for r in rows)
        if count >= target:
            break
    vd = set(val_days)
    return [r for r in rows if r[4] not in vd], [r for r in rows if r[4] in vd], sorted(val_days)


def read_train_core(train_dir, ids, data, dev_ratio=0.1):
    """Training impressions (with every candidate and label) re-derived from ``behaviors.tsv`` and checked against ``MindData``."""
    nid_map = {nid: i + 1 for i, nid in enumerate(ids)}
    core, val, val_days = split_train_core(read_behaviors(os.path.join(train_dir, "behaviors.tsv"), nid_map), dev_ratio)
    if len(core) != data.train_core_impressions or len(val) != data.validation_impressions or val_days != list(data.validation_days):
        raise ValueError(f"re-derived split differs from MindData: {len(core)}/{len(val)} impressions, validation days {val_days} vs "
                         f"{data.train_core_impressions}/{data.validation_impressions}, {list(data.validation_days)}")
    return core


def check_train_alignment(ts, data):
    """The re-derived samples must be V10's: same clicked article and history in the same order, every negative V10 drew lies in the
    impression's pool (catches an article-id mapping that differs from ``MindData`` while the counts still agree)."""
    if ts.n_samples != len(data.train_core):
        raise ValueError(f"{ts.n_samples} re-derived samples vs {len(data.train_core)} in MindData.train_core")
    H = ts.hist.shape[1]
    for k, (_u, hist, pos, negs) in enumerate(data.train_core):
        i = int(ts.samp_imp[k])
        h = list(hist)[-H:]
        pool = ts.neg_flat[ts.neg_off[i]:ts.neg_off[i] + ts.neg_cnt[i]]
        if (pos != ts.samp_pos[k] or not np.array_equal(ts.hist[i, :len(h)], h) or ts.hist[i, len(h):].any()
                or not set(int(x) for x in negs) <= set(pool.tolist())):
            raise ValueError(f"re-derived training sample {k} differs from MindData.train_core[{k}] (article-id mapping mismatch?)")


def subsample(items, frac, seed=0):
    """Random subset (original order kept); ``frac >= 1`` keeps everything."""
    if frac is None or frac >= 1:
        return items
    n = max(1, int(round(len(items) * frac)))
    return [items[i] for i in np.sort(np.random.default_rng(seed).permutation(len(items))[:n])]


# ------------------------------------------------------------------------------------------- training samples
class _Samples:
    def _to(self, device):
        if getattr(self, "_dev", None) != device:
            self._hist_t = torch.from_numpy(self.hist).to(device)
            self._dev = device
        return self._hist_t


class TrainSet(_Samples):
    """V10's training samples (one per clicked article of an impression that also has a non-clicked one) but with the *whole* pool of
    impression negatives kept, so ``n_neg`` of them are re-drawn every epoch (V10 draws them once).  ``replace`` follows V10:
    sampling without replacement unless the pool is smaller than ``n_neg``."""

    dynamic = True

    def __init__(self, imps, max_hist=50, n_neg=4):
        keep = [r for r in imps if any(l == 1 for l in r[3]) and any(l == 0 for l in r[3])]
        self.n_neg, self.n_imp = n_neg, len(keep)
        self.hist = np.zeros((len(keep), max_hist), np.int32)
        neg_flat, samp_imp, samp_pos = [], [], []
        self.neg_off = np.zeros(len(keep), np.int64)
        self.neg_cnt = np.zeros(len(keep), np.int32)
        for i, (_u, h, c, y, _d) in enumerate(keep):
            h = list(h)[-max_hist:]
            self.hist[i, :len(h)] = h
            neg = [x for x, l in zip(c, y) if l == 0]
            self.neg_off[i], self.neg_cnt[i] = len(neg_flat), len(neg)
            neg_flat.extend(neg)
            for x, l in zip(c, y):
                if l == 1:
                    samp_imp.append(i)
                    samp_pos.append(x)
        self.neg_flat = np.asarray(neg_flat, np.int32)
        self.samp_imp = np.asarray(samp_imp, np.int64)
        self.samp_pos = np.asarray(samp_pos, np.int32)
        self.n_samples = len(self.samp_imp)

    def draw(self, rng):
        """[n_samples, n_neg] negatives for one epoch."""
        K = self.n_neg
        out = np.empty((self.n_samples, K), np.int32)
        for s in range(self.n_samples):
            i = self.samp_imp[s]
            o, c = self.neg_off[i], self.neg_cnt[i]
            out[s] = rng.choice(self.neg_flat[o:o + c], size=K, replace=bool(c < K))
        return out

    def batches(self, bs, epoch, seed, device):
        rng = np.random.default_rng([seed, epoch])
        negs = self.draw(rng)
        perm = rng.permutation(self.n_samples)
        hist_t = self._to(device)
        for s in range(0, self.n_samples, bs):
            ix = perm[s:s + bs]
            H = hist_t[torch.from_numpy(self.samp_imp[ix]).to(device)].long()
            C = torch.from_numpy(np.concatenate([self.samp_pos[ix, None], negs[ix]], 1).astype(np.int64)).to(device)
            yield H, H > 0, C


class StaticTrainSet(_Samples):
    """V10's own ``train_core`` rows ``(user, hist, pos, negs)``: the negatives drawn at build time stay fixed for every epoch."""

    dynamic = False

    def __init__(self, rows, max_hist=50):
        self.n_samples = self.n_imp = len(rows)
        self.hist = np.zeros((len(rows), max_hist), np.int32)
        self.cands = np.zeros((len(rows), 1 + len(rows[0][3])), np.int64)
        for i, (_u, h, pos, negs) in enumerate(rows):
            h = list(h)[-max_hist:]
            self.hist[i, :len(h)] = h
            self.cands[i] = [pos] + list(negs)

    def batches(self, bs, epoch, seed, device):
        rng = np.random.default_rng([seed, epoch])
        perm = rng.permutation(self.n_samples)
        hist_t = self._to(device)
        for s in range(0, self.n_samples, bs):
            ix = perm[s:s + bs]
            H = hist_t[torch.from_numpy(ix).to(device)].long()
            yield H, H > 0, torch.from_numpy(self.cands[ix]).to(device)


In [ ]:
%%writefile refmodels.py
"""Reference news-recommendation models (NRMS / NAML / Fastformer) and a small trainer, scored by the suite's evaluator.

One ``RefNet`` class covers the three families so that a V10-style configuration (64-d, no title padding mask, ``nn.MultiheadAttention``
layout) and the published recipe (300-d GloVe, 16 heads x 16, masked, lr 1e-4) differ only in ``RefCfg`` -- that is what lets the
ablation ladder attribute the gap between V10's baselines and the literature to individual choices.

News vectors depend only on the article's own tokens, so evaluation encodes every article once (``all_news_vecs``) and a slate is
scored with ``<user vector, candidate vector>`` -- the same interface as the representation-level scorers in ``rep_eval``.
The building blocks (``AddAttn``, ``SelfAttn``, ``FastEncoder``) are reused by ``usermodel`` for learned user encoders over frozen
LLM article vectors.
"""
from __future__ import annotations

import math
import time
from dataclasses import dataclass, replace

import torch
import torch.nn as nn
import torch.nn.functional as F

PAD = 0


# ------------------------------------------------------------------------------------------- building blocks
class AddAttn(nn.Module):
    """Additive attention pooling ``sum_i softmax(q . tanh(W x_i)) x_i`` over the second-to-last axis, padding masked."""

    def __init__(self, d, hidden=200):
        super().__init__()
        self.proj, self.q = nn.Linear(d, hidden), nn.Linear(hidden, 1, bias=False)

    def forward(self, x, mask=None):
        a = self.q(torch.tanh(self.proj(x))).squeeze(-1).float()
        if mask is not None:
            a = a.masked_fill(~mask, -1e9)
        return torch.einsum("...n,...nd->...d", torch.softmax(a, -1).to(x.dtype), x)


class SelfAttn(nn.Module):
    """Multi-head self-attention.  NRMS layout: Q/K/V maps only (``out_proj=False``); ``out_proj=True`` adds the output projection and is
    the layout of ``torch.nn.MultiheadAttention`` that V10 uses.  ``mask`` [B, L] bool (True = real token) must keep one True per row."""

    def __init__(self, d_in, heads, head_dim, out_proj=False, attn_drop=0.0):
        super().__init__()
        D = heads * head_dim
        self.h, self.dh, self.p = heads, head_dim, attn_drop
        self.q, self.k, self.v = nn.Linear(d_in, D), nn.Linear(d_in, D), nn.Linear(d_in, D)
        self.o = nn.Linear(D, D) if out_proj else nn.Identity()

    def forward(self, x, mask=None):
        B, L, _ = x.shape
        sp = lambda t: t.view(B, L, self.h, self.dh).transpose(1, 2)
        o = F.scaled_dot_product_attention(sp(self.q(x)), sp(self.k(x)), sp(self.v(x)),
                                           attn_mask=None if mask is None else mask[:, None, None, :],
                                           dropout_p=self.p if self.training else 0.0)
        return self.o(o.transpose(1, 2).reshape(B, L, self.h * self.dh))


class FastSelfAttn(nn.Module):
    """Fastformer additive attention (Wu et al. 2021), following the layout of the official code: a global query from per-head additive
    attention over the queries, an element-wise product with every key, a global key from a second additive attention, an element-wise
    product with the value (the query doubles as value), a linear transform and a residual connection to the query."""

    def __init__(self, d, heads):
        super().__init__()
        assert d % heads == 0
        self.h, self.dh = heads, d // heads
        self.q, self.k = nn.Linear(d, d), nn.Linear(d, d)
        self.q_att, self.k_att = nn.Linear(d, heads), nn.Linear(d, heads)
        self.transform = nn.Linear(d, d)

    def _pool(self, scores, x, mask):
        B, L, d = x.shape
        w = torch.softmax((scores.float() / math.sqrt(self.dh)).masked_fill(~mask[:, :, None], -1e9), 1)
        return (w.unsqueeze(-1).to(x.dtype) * x.view(B, L, self.h, self.dh)).sum(1)                       # [B, h, dh]

    def forward(self, x, mask):
        B, L, d = x.shape
        q, k = self.q(x), self.k(x)
        gq = self._pool(self.q_att(q), q, mask)
        p = (k.view(B, L, self.h, self.dh) * gq[:, None]).reshape(B, L, d)
        gk = self._pool(self.k_att(p), p, mask)
        u = (gk[:, None] * q.view(B, L, self.h, self.dh)).reshape(B, L, d)
        return self.transform(u) + q


class FastLayer(nn.Module):
    def __init__(self, d, heads, dropout, ff_mult=4):
        super().__init__()
        self.att, self.ln1, self.ln2, self.drop = FastSelfAttn(d, heads), nn.LayerNorm(d), nn.LayerNorm(d), nn.Dropout(dropout)
        self.ff = nn.Sequential(nn.Linear(d, ff_mult * d), nn.GELU(), nn.Dropout(dropout), nn.Linear(ff_mult * d, d))

    def forward(self, x, mask):
        x = self.ln1(x + self.drop(self.att(x, mask)))
        return self.ln2(x + self.drop(self.ff(x)))


class FastEncoder(nn.Module):
    """Input projection + learned positions + ``layers`` Fastformer blocks (token mixing in O(L)).  ``dropout`` acts on the input
    (callers that already drop the embeddings pass 0), ``inner_dropout`` (BERT's 0.1) inside the blocks: stacking 0.2 dropouts stalls
    training at small widths (measured on the toy fixture)."""

    def __init__(self, d_in, d, heads, layers, max_len, dropout=0.0, inner_dropout=0.1):
        super().__init__()
        self.proj = nn.Identity() if d_in == d else nn.Linear(d_in, d)
        self.pos, self.ln, self.drop = nn.Embedding(max_len, d), nn.LayerNorm(d), nn.Dropout(dropout)
        self.layers = nn.ModuleList(FastLayer(d, heads, inner_dropout) for _ in range(layers))

    def forward(self, x, mask):
        L = x.shape[1]
        x = self.drop(self.ln(self.proj(x) + self.pos(torch.arange(L, device=x.device))))
        for layer in self.layers:
            x = layer(x, mask)
        return x


def safe_mask(mask):
    """A row without any real token attends to itself instead of producing NaN (such rows are zeroed afterwards)."""
    return mask | ~mask.any(-1, keepdim=True)


# ------------------------------------------------------------------------------------------- configuration
@dataclass(frozen=True)
class RefCfg:
    kind: str = "nrms"              # nrms | naml | fastformer
    text: str = "ref"               # ref: regex tokens, title-only (abstract for NAML) | v10: the 20-token title+abstract ids of data.py
    emb_dim: int = 300
    heads: int = 16
    head_dim: int = 16
    att_hidden: int = 200
    dropout: float = 0.2
    attn_drop: float = 0.0
    out_proj: bool = False          # True: torch.nn.MultiheadAttention layout (V10)
    title_mask: bool = True         # False: padding tokens take part in attention and pooling (V10)
    init: str = "glove"             # glove | random (N(0, 0.1)) | torch (nn.Embedding default N(0, 1), V10)
    dyn_neg: bool = True            # re-draw the negatives every epoch (V10: once)
    lr: float = 1e-4
    wd: float = 0.0
    bs: int = 64
    max_epochs: int = 10
    min_epochs: int = 3
    patience: int = 2
    min_delta: float = 1e-3
    ff_layers: int = 1              # V10's Fastformer is one block too; a second layer doubles the cost (the news encoder dominates)
    naml_filters: int = 400
    cat_dim: int = 100
    clip: float = 5.0               # gradient-norm clip (0 = none, as in V10)


def preset(name):
    """``v10``: V10's NRMS recipe re-implemented (harness check).  ``data``: + title-only regex tokens, padding mask, negatives re-drawn each
    epoch.  ``recipe``: the published NRMS hyper-parameters with random word vectors.  ``nrms`` / ``naml`` / ``fastformer``: + GloVe."""
    if name == "v10":
        return RefCfg(kind="nrms", text="v10", emb_dim=64, heads=2, head_dim=32, dropout=0.2, attn_drop=0.2, out_proj=True, title_mask=False,
                      init="torch", dyn_neg=False, lr=1e-3, wd=1e-5, max_epochs=12, clip=0.0)
    if name == "data":
        return replace(preset("v10"), text="ref", title_mask=True, dyn_neg=True)
    if name == "recipe":
        return RefCfg(kind="nrms", init="random")
    if name in ("nrms", "naml", "fastformer"):
        return RefCfg(kind=name, init="glove")
    raise KeyError(name)


# ------------------------------------------------------------------------------------------- the network
class RefNet(nn.Module):
    def __init__(self, cfg, tokens, vocab_size, n_cat=0, n_sub=0, emb_init=None, max_hist=50):
        super().__init__()
        self.cfg = cfg
        for k, v in tokens.items():
            self.register_buffer(k, v, persistent=False)
        D = cfg.heads * cfg.head_dim
        if cfg.out_proj and cfg.emb_dim != D:
            raise ValueError("the nn.MultiheadAttention layout needs emb_dim == heads * head_dim")
        self.emb = nn.Embedding(vocab_size, cfg.emb_dim, padding_idx=PAD)
        if emb_init is not None:
            with torch.no_grad():
                self.emb.weight.copy_(torch.as_tensor(emb_init))
                self.emb.weight[PAD].zero_()
        self.drop = nn.Dropout(cfg.dropout)
        if cfg.kind == "nrms":
            self.news_att = SelfAttn(cfg.emb_dim, cfg.heads, cfg.head_dim, cfg.out_proj, cfg.attn_drop)
            self.news_pool = AddAttn(D, cfg.att_hidden)
            self.user_att = SelfAttn(D, cfg.heads, cfg.head_dim, cfg.out_proj, cfg.attn_drop)
            self.user_pool = AddAttn(D, cfg.att_hidden)
            self.out_dim = D
        elif cfg.kind == "fastformer":
            self.news_enc = FastEncoder(cfg.emb_dim, D, cfg.heads, cfg.ff_layers, tokens["title"].shape[1])
            self.news_pool = AddAttn(D, cfg.att_hidden)
            self.user_enc = FastEncoder(D, D, cfg.heads, cfg.ff_layers, max_hist)
            self.user_pool = AddAttn(D, cfg.att_hidden)
            self.out_dim = D
        elif cfg.kind == "naml":
            Fn = cfg.naml_filters
            self.conv_t, self.conv_b = nn.Conv1d(cfg.emb_dim, Fn, 3, padding=1), nn.Conv1d(cfg.emb_dim, Fn, 3, padding=1)
            self.pool_t, self.pool_b = AddAttn(Fn, cfg.att_hidden), AddAttn(Fn, cfg.att_hidden)
            self.cat_emb, self.sub_emb = nn.Embedding(n_cat, cfg.cat_dim, padding_idx=PAD), nn.Embedding(n_sub, cfg.cat_dim, padding_idx=PAD)
            self.cat_fc, self.sub_fc = nn.Linear(cfg.cat_dim, Fn), nn.Linear(cfg.cat_dim, Fn)
            self.view_pool, self.user_pool = AddAttn(Fn, cfg.att_hidden), AddAttn(Fn, cfg.att_hidden)
            self.out_dim = Fn
        else:
            raise ValueError(cfg.kind)

    # ---- articles
    def _mask(self, t, force=False):
        if not (self.cfg.title_mask or force):
            return None
        return safe_mask(t != PAD)

    def encode_news(self, ids):
        """[M] global article ids -> [M, out_dim]."""
        c, t = self.cfg, self.title[ids]
        if c.kind == "nrms":
            m = self._mask(t)
            return self.news_pool(self.drop(self.news_att(self.drop(self.emb(t)), m)), m)
        if c.kind == "fastformer":
            m = self._mask(t, force=True)
            return self.news_pool(self.news_enc(self.drop(self.emb(t)), m), m)
        b, mt, mb = self.body[ids], self._mask(t, True), self._mask(self.body[ids], True)
        tv = self.pool_t(self.drop(F.relu(self.conv_t(self.drop(self.emb(t)).transpose(1, 2)))).transpose(1, 2), mt)
        bv = self.pool_b(self.drop(F.relu(self.conv_b(self.drop(self.emb(b)).transpose(1, 2)))).transpose(1, 2), mb)
        cv = F.relu(self.cat_fc(self.cat_emb(self.cat[ids])))
        sv = F.relu(self.sub_fc(self.sub_emb(self.sub[ids])))
        return self.view_pool(torch.stack([tv, bv, cv, sv], 1))

    @torch.no_grad()
    def all_news_vecs(self, bs=4096, amp=False):
        """Eval-mode vectors of every article (row 0 = zeros) in float32."""
        was = self.training
        self.eval()
        N = self.title.shape[0]
        out = torch.zeros(N, self.out_dim, device=self.title.device)
        for s in range(0, N, bs):
            ids = torch.arange(s, min(s + bs, N), device=out.device)
            with torch.autocast("cuda", dtype=torch.float16, enabled=amp):
                out[s:s + len(ids)] = self.encode_news(ids).float()
        out[PAD] = 0
        self.train(was)
        return out

    # ---- readers
    def user_vec(self, hv, hmask):
        """[B, H, d] history vectors + [B, H] mask -> [B, d]; a reader without history gets the zero vector (all scores tie)."""
        c, safe = self.cfg, safe_mask(hmask)
        if c.kind == "nrms":
            u = self.user_pool(self.user_att(hv, safe), safe)
        elif c.kind == "fastformer":
            u = self.user_pool(self.user_enc(hv, safe), safe)
        else:
            u = self.user_pool(hv, safe)
        return u * hmask.any(-1, keepdim=True).to(u.dtype)

    def forward(self, hist, hmask, cand):
        """Training scores [B, C]: every distinct article of the batch is encoded once."""
        B, H = hist.shape
        C = cand.shape[1]
        uniq, inv = torch.unique(torch.cat([hist.reshape(-1), cand.reshape(-1)]), return_inverse=True)
        vecs = self.encode_news(uniq)
        hv, cv = vecs[inv[:B * H]].view(B, H, -1), vecs[inv[B * H:]].view(B, C, -1)
        return torch.einsum("bd,bcd->bc", self.user_vec(hv, hmask), cv)


def build_net(cfg, device, seed, text=None, data=None, max_hist=50):
    """``cfg.text == 'v10'`` reads V10's ``data.news_title`` (20 tokens, OOV -> PAD); otherwise the ``RefText`` tables."""
    torch.manual_seed(seed)
    if cfg.text == "v10":
        net = RefNet(cfg, {"title": torch.from_numpy(data.news_title)}, data.vocab_size, max_hist=max_hist)
    else:
        init = None if cfg.init == "torch" else text.embedding(cfg.emb_dim, seed, cfg.init)
        net = RefNet(cfg, text.tokens("cpu"), text.vocab_size, text.n_cat, text.n_sub, init, max_hist)
    return net.to(device)


def n_params(net):
    return int(sum(p.numel() for p in net.parameters()))


def make_scorer(net, amp=False):
    """Slate scorer for ``rep_eval.evaluate`` / ``Suite.run`` (encodes the catalogue once, in eval mode)."""
    V = net.all_news_vecs(amp=amp)

    def f(b):
        return torch.einsum("bd,bcd->bc", net.user_vec(V[b["hist"]], b["hist_mask"]), V[b["cand"]])
    return f


# ------------------------------------------------------------------------------------------- training
def train_ref(net, trainset, val_fn, cfg, seed=0, device="cpu", deadline=None, log=print, fixed_epochs=None):
    """Adam + cross-entropy over (1 positive, ``n_neg`` negatives).  Early stopping exactly as V10 / ``usermodel``: validation nDCG@10 per
    epoch, patience ``cfg.patience`` after ``cfg.min_epochs``, improvement must exceed ``cfg.min_delta``; the best epoch is restored.
    ``val_fn(net) -> float``.  Stops (after finishing a validation pass) once ``deadline`` has passed.
    ``fixed_epochs=k``: no validation and no early stopping -- exactly ``k`` epochs (or fewer if the deadline passes) and the final weights are
    kept; used to refit on train+validation for the epoch count a validated run selected (``val_fn`` is not called and may be None)."""
    amp = device == "cuda"
    torch.manual_seed(seed)
    opt = torch.optim.Adam(net.parameters(), lr=cfg.lr, weight_decay=cfg.wd)
    scaler = torch.amp.GradScaler("cuda", enabled=amp)
    best, best_state, best_epoch, stale, hist, steps, t0 = None, None, 0, 0, [], 0, time.time()
    over = lambda: deadline is not None and time.time() > deadline
    for epoch in range(1, (fixed_epochs or cfg.max_epochs) + 1):
        if best_state is not None and over():
            log(f"      [ref] deadline reached after epoch {epoch - 1}")
            break
        net.train()
        te, tot, nb, cut = time.time(), torch.zeros((), device=device), 0, False
        for H, M, C in trainset.batches(cfg.bs, epoch, seed, device):
            with torch.autocast("cuda", dtype=torch.float16, enabled=amp):
                logits = net(H, M, C)
            loss = F.cross_entropy(logits.float(), torch.zeros(len(H), dtype=torch.long, device=device))
            opt.zero_grad(set_to_none=True)
            scaler.scale(loss).backward()
            if cfg.clip:
                scaler.unscale_(opt)
                torch.nn.utils.clip_grad_norm_(net.parameters(), cfg.clip)
            scaler.step(opt)
            scaler.update()
            tot += loss.detach()
            nb += 1
            steps += 1
            if nb % 100 == 0:
                if not torch.isfinite(tot):
                    raise FloatingPointError(f"non-finite loss in epoch {epoch}, step {nb}")
                if over() and best_state is not None:
                    cut = True
                    break
        train_s = time.time() - te
        if not torch.isfinite(tot):
            raise FloatingPointError(f"non-finite loss in epoch {epoch}")
        net.eval()
        loss_m = float(tot) / max(nb, 1)
        if fixed_epochs:
            best, best_epoch = 0.0, epoch
            best_state = {k: v.detach().clone() for k, v in net.state_dict().items()}
            hist.append({"epoch": epoch, "loss": loss_m, "val": float("nan"), "train_s": train_s, "partial": cut})
            log(f"      [ref] refit epoch {epoch:02d}/{fixed_epochs} loss={loss_m:.4f} ({train_s:.0f}s train, {nb} steps){' PARTIAL' if cut else ''}")
            if cut:
                break
            continue
        val = float(val_fn(net))
        hist.append({"epoch": epoch, "loss": loss_m, "val": val, "train_s": train_s, "partial": cut})
        improved = best is None or val > best + cfg.min_delta
        if improved:
            best, best_epoch, stale = val, epoch, 0
            best_state = {k: v.detach().clone() for k, v in net.state_dict().items()}
        elif epoch >= cfg.min_epochs:
            stale += 1
        log(f"      [ref] epoch {epoch:02d} loss={loss_m:.4f} val nDCG@10={val:.4f} ({train_s:.0f}s train, {nb} steps){' PARTIAL' if cut else ''}")
        if cut or (epoch >= cfg.min_epochs and stale >= cfg.patience):
            break
    net.load_state_dict(best_state)
    net.eval()
    return net, {"best_val": float(best), "best_epoch": best_epoch, "epochs": len(hist), "steps": steps, "train_s": time.time() - t0,
                 "history": hist, "params": n_params(net)}


In [ ]:
%%writefile usermodel.py
"""Downstream consistency check: does a better article representation survive a *learned* user model?

``LLMEncCA`` is a line-by-line clone of the V10 model of the same name (candidate-aware attention over the
history, on top of a frozen article embedding): 2-layer head to ``dim``, candidate projection, scaled dot-product
attention of the candidate over the clicked history, score = <attended user, candidate>.  Training follows the V10
protocol (Adam 1e-3, wd 1e-5, batch 64, 1 positive + 4 sampled negatives, cross-entropy, early stopping on
validation nDCG@10 with patience 2 / min_delta 0.001 / at least 3 epochs, at most 12).

Only the embedding table changes between runs, so differences are attributable to the representation.
"""
from __future__ import annotations

import math
import re
import time

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F


class LLMEncCA(nn.Module):
    def __init__(self, emb, dim=64, dropout=0.2, device="cpu"):
        super().__init__()
        self.set_emb(emb, device)
        self.head = nn.Sequential(nn.Linear(self.emb.shape[1], dim), nn.GELU(), nn.LayerNorm(dim), nn.Dropout(dropout),
                                  nn.Linear(dim, dim))
        self.cand_proj = nn.Linear(dim, dim, bias=False)
        self.scale = dim ** 0.5
        self.to(device)

    def set_emb(self, emb, device=None):
        self.emb = torch.as_tensor(np.ascontiguousarray(emb, np.float32)).to(device or self.emb.device)

    def score(self, hist, hist_mask, cand):
        h, c = self.head(self.emb[hist]), self.head(self.emb[cand])
        att = torch.einsum("bcd,bhd->bch", self.cand_proj(c), h) / self.scale
        att = torch.softmax(att.masked_fill(~hist_mask[:, None, :], -1e9), -1)
        return (torch.einsum("bch,bhd->bcd", att, h) * c).sum(-1)


class LLMEncUser(nn.Module):
    """Same frozen article table and 2-layer head as ``LLMEncCA``, but the reader is encoded *without* looking at the candidate:
    ``add`` additive attention pooling (NAML-style), ``nrms`` multi-head self-attention + additive pooling, ``ff`` Fastformer block +
    additive pooling.  Only the user encoder differs from the V10 clone, so the comparison isolates the attention module."""

    def __init__(self, emb, kind="ff", dim=64, heads=2, dropout=0.2, device="cpu"):
        super().__init__()
        from refmodels import AddAttn, FastEncoder, SelfAttn
        self.set_emb(emb, device)
        self.kind = kind
        self.head = nn.Sequential(nn.Linear(self.emb.shape[1], dim), nn.GELU(), nn.LayerNorm(dim), nn.Dropout(dropout), nn.Linear(dim, dim))
        self.enc = {"add": None, "nrms": SelfAttn(dim, heads, dim // heads, out_proj=True),
                    "ff": FastEncoder(dim, dim, heads, 1, 50)}[kind]
        self.pool = AddAttn(dim, 200)
        self.to(device)

    def set_emb(self, emb, device=None):
        self.emb = torch.as_tensor(np.ascontiguousarray(emb, np.float32)).to(device or self.emb.device)

    def score(self, hist, hist_mask, cand):
        from refmodels import safe_mask
        h, c = self.head(self.emb[hist]), self.head(self.emb[cand])
        safe = safe_mask(hist_mask)
        if self.enc is not None:
            h = self.enc(h, safe)
        u = self.pool(h, safe) * hist_mask.any(-1, keepdim=True).to(h.dtype)
        return torch.einsum("bd,bcd->bc", u, c)


class LLMEncMI(nn.Module):
    """Multi-interest reader over the same frozen article table and 2-layer head as the other readers: ``K`` learned queries each attend
    over the clicked history (candidate-independent, in the spirit of multi-interest extractors such as MIND / ComiRec) and give one interest vector;
    a candidate is scored by the soft maximum over interests, ``tau * (logsumexp_k(<u_k, c> / tau) - log K)`` with a learned ``tau``.
    ``K=1`` is a plain single-vector attention reader (the control that isolates "several interests"); readers without history score 0.
    Queries start at N(0, 1) so the interests attend differently from step 0 (N(0, 1/d) queries gave a mean cosine of 0.9998 between the
    K interests at step 0 on a toy fixture, i.e. no room to specialise; N(0, 1) gives an attention total-variation distance of ~0.17).
    ``div > 0`` adds a disagreement penalty
    (in the spirit of disagreement regularisation for multi-head attention): ``div`` times the mean squared cosine between different
    interests of the same reader."""

    def __init__(self, emb, K=4, dim=64, dropout=0.2, device="cpu", div=0.0):
        super().__init__()
        self.set_emb(emb, device)
        self.K, self.div = int(K), float(div)
        self.head = nn.Sequential(nn.Linear(self.emb.shape[1], dim), nn.GELU(), nn.LayerNorm(dim), nn.Dropout(dropout), nn.Linear(dim, dim))
        self.queries = nn.Parameter(torch.randn(self.K, dim))
        self.key = nn.Linear(dim, dim, bias=False)
        self.log_tau = nn.Parameter(torch.zeros(()))
        self.scale = dim ** 0.5
        self._aux = None
        self.to(device)

    def set_emb(self, emb, device=None):
        self.emb = torch.as_tensor(np.ascontiguousarray(emb, np.float32)).to(device or self.emb.device)

    def interests(self, hist, hist_mask, with_att=False):
        """[B, K, dim] interest vectors (zero for a reader without history); ``with_att`` also returns the attention [B, K, H]."""
        from refmodels import safe_mask
        h = self.head(self.emb[hist])
        safe = safe_mask(hist_mask)
        att = torch.einsum("kd,bhd->bkh", self.queries, self.key(h)) / self.scale
        att = torch.softmax(att.masked_fill(~safe[:, None, :], -1e9), -1)
        u = torch.einsum("bkh,bhd->bkd", att, h) * hist_mask.any(-1)[:, None, None].to(h.dtype)
        return (u, att) if with_att else u

    @staticmethod
    def _pair_cos(u, has):
        """Per reader (with history): mean squared cosine between different interests -> [B]."""
        un = F.normalize(u, dim=-1)
        cos = torch.einsum("bkd,bld->bkl", un, un) ** 2
        K = u.shape[1]
        return (cos.sum((1, 2)) - cos.diagonal(dim1=1, dim2=2).sum(1)) / (K * (K - 1)) * has.to(u.dtype)

    def score(self, hist, hist_mask, cand):
        u = self.interests(hist, hist_mask)
        s = torch.einsum("bkd,bcd->bkc", u, self.head(self.emb[cand]))
        if self.K == 1:
            return s[:, 0]
        if self.training and self.div > 0:
            has = hist_mask.any(-1)
            self._aux = self.div * self._pair_cos(u, has).sum() / has.sum().clamp(min=1)
        tau = self.log_tau.exp().clamp(min=0.05)
        return tau * (torch.logsumexp(s / tau, 1) - math.log(self.K))

    def aux_loss(self):
        """Regulariser of the last ``score`` call (0 when ``div`` is 0); the trainer adds it to the loss."""
        aux, self._aux = self._aux, None
        return aux if aux is not None else 0.0

    @torch.no_grad()
    def diversity(self, hist, hist_mask):
        """Collapse diagnostics over the readers that have a history: ``interest_cos`` = mean pairwise cosine between a reader's interest
        vectors (1 = identical; note the shared head output keeps it high even for different attention), ``attn_tv`` = mean pairwise
        total-variation distance between the interests' attention distributions (0 = every interest attends identically, 1 = disjoint)."""
        has = hist_mask.any(-1)
        if self.K < 2 or not bool(has.any()):
            return {"interest_cos": float("nan"), "attn_tv": float("nan")}
        u, att = self.interests(hist, hist_mask, with_att=True)
        u, att = F.normalize(u[has], dim=-1), att[has]
        cos = torch.einsum("bkd,bld->bkl", u, u)
        pairs = self.K * (self.K - 1)
        tv = 0.5 * (att[:, :, None, :] - att[:, None, :, :]).abs().sum(-1)                   # [B, K, K]
        return {"interest_cos": float(((cos.sum((1, 2)) - cos.diagonal(dim1=1, dim2=2).sum(1)) / pairs).mean()),
                "attn_tv": float((tv.sum((1, 2)) / pairs).mean())}


def _mi_spec(kind):
    """``mi4`` -> (4, 0.0); ``mi4d`` -> (4, 0.1) (with the disagreement regulariser); anything else -> None."""
    m = re.fullmatch(r"mi(\d+)(d?)", kind)
    return None if m is None or int(m.group(1)) < 1 else (int(m.group(1)), 0.1 if m.group(2) else 0.0)


def is_reader(kind):
    """``ca`` | ``add`` | ``nrms`` | ``ff`` | ``mi<K>`` (K learned interests, e.g. ``mi4``) | ``mi<K>d`` (same with the disagreement regulariser)."""
    return kind in ("ca", "add", "nrms", "ff") or _mi_spec(kind) is not None


def make_model(kind, emb, dim=64, dropout=0.2, device="cpu"):
    """``ca``: the V10 clone; ``add`` / ``nrms`` / ``ff``: candidate-independent readers (see ``LLMEncUser``); ``mi<K>[d]``: ``LLMEncMI``."""
    if kind == "ca":
        return LLMEncCA(emb, dim, dropout, device)
    spec = _mi_spec(kind)
    if spec is not None:
        return LLMEncMI(emb, spec[0], dim, dropout, device, spec[1])
    return LLMEncUser(emb, kind, dim, 2, dropout, device)


def pack_train(rows, max_hist=50):
    """train_core rows ``(user, hist, pos, negs)`` -> arrays (hist [N,H], mask [N,H], cands [N,1+n_neg], pos first)."""
    N = len(rows)
    H = np.zeros((N, max_hist), np.int64)
    M = np.zeros((N, max_hist), bool)
    C = np.zeros((N, 1 + len(rows[0][3])), np.int64)
    for i, (_u, hist, pos, negs) in enumerate(rows):
        h = list(hist)[-max_hist:]
        H[i, :len(h)], M[i, :len(h)] = h, True
        C[i] = [pos] + list(negs)
    return H, M, C


def train_user_model(emb, pack, val_fn, seed=0, dim=64, dropout=0.2, lr=1e-3, wd=1e-5, bs=64, max_epochs=12,
                     min_epochs=3, patience=2, min_delta=1e-3, device="cpu", deadline=None, log=print, kind="ca"):
    """``val_fn(model) -> float`` (validation nDCG@10, evaluated with the embedding table valid for the validation
    period).  ``kind``: user encoder, see ``make_model``.  Returns ``(model restored to its best epoch, info)``."""
    torch.manual_seed(seed)
    rng = np.random.default_rng(seed)
    model = make_model(kind, emb, dim, dropout, device)
    opt = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=wd)
    H, M, C = (torch.from_numpy(a).to(device) for a in pack)
    N = len(H)
    target = torch.zeros(bs, dtype=torch.long, device=device)
    best, best_state, stale, hist, t0 = None, None, 0, [], time.time()
    for epoch in range(1, max_epochs + 1):
        if deadline is not None and time.time() > deadline and best_state is not None:
            log(f"      [user-model] deadline reached after epoch {epoch - 1}")
            break
        model.train()
        perm = torch.from_numpy(rng.permutation(N)).to(device)
        tot, nb = 0.0, 0
        for s in range(0, N, bs):
            ix = perm[s:s + bs]
            logits = model.score(H[ix], M[ix], C[ix])
            loss = F.cross_entropy(logits, target[:len(ix)])
            if hasattr(model, "aux_loss"):
                loss = loss + model.aux_loss()
            if not torch.isfinite(loss):
                raise FloatingPointError(f"non-finite loss at epoch {epoch}")
            opt.zero_grad(set_to_none=True)
            loss.backward()
            opt.step()
            tot += loss.item(); nb += 1
        model.eval()
        val = val_fn(model)
        hist.append((epoch, tot / max(nb, 1), val))
        improved = best is None or val > best + min_delta
        if improved:
            best, stale = val, 0
            best_state = {k: v.detach().clone() for k, v in model.state_dict().items()}
        elif epoch >= min_epochs:
            stale += 1
        log(f"      [user-model] epoch {epoch:02d} loss={tot / max(nb, 1):.4f} val nDCG@10={val:.4f}")
        if epoch >= min_epochs and stale >= patience:
            break
    model.load_state_dict(best_state)
    model.eval()
    return model, {"best_val": float(best), "epochs": len(hist), "train_s": time.time() - t0}


def scorer(model, emb=None):
    """Slate scorer for ``rep_eval.evaluate``; ``emb`` swaps the embedding table (e.g. test-period retrieval memory)."""
    if emb is not None:
        model.set_emb(emb)

    def f(b):
        return model.score(b["hist"], b["hist_mask"], b["cand"])
    return f


In [ ]:
%%writefile stages_ext.py
"""Stage functions of the V11 suite beyond the original four (controls/graph/head/encoder/histquery):

  pooling       H8   recency / late-interaction pooling over frozen vectors, plus zero-shot k-means interests (H11)
  moredata      F    which article fields go into the text an encoder reads (title / +category / +entities / view fusion), and how
                     much history a pooling rule sees (50 / 100 / 200 clicks)
  sid           H7   Semantic IDs: RQ-KMeans profile + TIGER-style generative slate ranker
  sweep         --   encoder-scale sweep (+ lexical TF-IDF/LSA floor)
  llm_user      H5   LoRA-tuned LLM user tower, causal / bidirectional / soft attention mask
  usermodel     H10  learned candidate-aware user model (V10 ``llmenc_ca``) on top of each representation, plus Fastformer / NRMS /
                     additive readers and the multi-interest reader ``mi<K>`` (H11)
  graphrag_llm  H4b  GraphRAG with LLM-written community summaries
  augment       H6   KAR-style LLM article expansions and reader profiles
  rerank        H9   zero-shot LLM judge re-ranking the top-k

Every function takes the ``Suite`` object ``S`` (see suite.py) and the parsed ``args``.
"""
from __future__ import annotations

import os
import time

import numpy as np
import torch

import fields as FD
import llm_gen
import pooling as P
import rep_eval as R
import semgraph as G
import sid as SID
import usermodel as UM
from common import HistQueries, ScoreCache, avg_results, decoder_only, fuse, log, query_prefix, subset_evalset


# ============================================================================ shared helpers
def embed_news(meta, name, max_len, prefix, device, bs, cache_dir, dtype=torch.float32, texts=None):
    """Frozen article vectors ``[n_articles, d]`` of ``texts`` (default: the suite's own ``title. abstract``), cached on disk per
    (model, max_len, prefix, the texts themselves) so that the encoder sweep and a later run with the same model never encode twice and a
    different text variant can never be served from another variant's cache.  Returns ``(vecs, seconds)`` (0 if cached)."""
    return FD.embed_texts(meta["text"] if texts is None else texts, name, max_len, prefix, device, bs, cache_dir, dtype)


def short(name):
    return (name.split("/")[-1].replace("-en-v1.5", "").replace("Qwen3-Embedding-", "qwen3emb-").lower()
            .replace("-", "").replace(".", ""))


def table_from(S, vecs):
    E = np.zeros((S.data.n_news, vecs.shape[1]), np.float32)
    E[1:] = vecs
    return E


def load_export(S, name):
    z = np.load(os.path.join(S.work, f"emb_{name}.npz"), allow_pickle=True)
    return table_from(S, z["vecs"])


def sec_split(total, frac):
    return max(int(total * frac), 0)


# ============================================================================ H8 pooling
def stage_pooling(S, args):
    log("\n== H8 pooling rule over the frozen vectors (recency / late interaction) and H11 zero-shot multi-interest ==")
    E = S.tensor(S.E0)
    fam = "H8-pooling"
    S.select("pool_recency", fam, [{"lam": l} for l in args.pool_lams], lambda c, sp: P.recency_scorer(E, c["lam"]), "frozen")
    S.select("pool_maxsim", fam, [{}], lambda c, sp: P.late_scorer(E, "max"), "frozen")
    S.select("pool_topk", fam, [{"k": k} for k in args.pool_ks], lambda c, sp: P.late_scorer(E, "topk", c["k"]), "frozen")
    S.select("pool_lse", fam, [{"tau": t} for t in args.pool_taus], lambda c, sp: P.late_scorer(E, "lse", c["tau"]), "frozen")
    # k-means interests: K=1 is the mean-pool control, K >= the number of clicks is the lse row above, so these rows sit between the two
    for K in args.km_ks:
        S.select(f"pool_km{K}", "H11-multi-interest", [{"k": K, "tau": t} for t in args.pool_taus], lambda c, sp: P.cluster_scorer(E, c["k"], c["tau"]), "frozen")


# ============================================================================ F: more of the data
def _embedder(S, args):
    """``texts -> [len(texts), d]`` frozen vectors of the run's encoder, through the content-keyed cache (no encoding after a resume)."""
    return lambda texts: FD.embed_texts(texts, args.encoder, args.max_len, args.doc_prefix, S.device, args.enc_bs, S.cache_dir)[0]


def view_vectors(S, args):
    """Title / abstract / category-phrase vectors of every article, ``[n_news, d]`` each (the inputs of the ``views`` fusion)."""
    emb, V = _embedder(S, args), FD.view_texts(S.meta)
    return [table_from(S, x) for x in (emb(V["t"]), emb(V["a"]), FD.embed_unique(emb, V["c"]))]


def text_table(S, args, spec):
    """Article table ``[n_news, d]`` of a text variant: ``{'mode': 't'|'ta'|'cta'|'ctae'}`` or ``{'mode': 'views', 'w': [w_title, w_abstract, w_category]}``."""
    if spec["mode"] == "views":
        return FD.combine(view_vectors(S, args), spec["w"])
    return table_from(S, _embedder(S, args)(FD.build_texts(S.meta, spec["mode"])))


def stage_moredata(S, args):
    log("\n== F  more of the data: article fields in the text an encoder reads, and the length of the history a pooling rule sees ==")
    if "pool_lse" not in S.cache:
        log("  not run: the late-interaction row of the pooling stage is the reference of the -lse families (run 'pooling' first)")
        return False
    a = FD.audit(S.meta, S.data)
    log(f"  news.tsv has {a['news_columns']} columns per line (MIND-small: 8 = id, category, subcategory, title, abstract, URL, title entities, abstract entities)"
        + (": MIND-small carries no article body" if a["news_columns"] <= 8 else f": !! {a['news_columns'] - 8} EXTRA column(s) this suite does not read - check whether one is an article body"))
    log(f"  the longest text is the abstract ({a['abstract_words']:.0f} words on average, {a['abstract_empty']:.1%} empty; "
        f"title {a['title_words']:.1f} words).  {a['categories']} categories / {a['subcategories']} subcategories; {a['with_entity_labels']:.1%} of articles carry entity names "
        f"({a['entities_per_article']:.1f} on average).  Test readers have {a['test_history_mean']:.1f} clicks on average, {a['test_history_over_50']:.1%} more than the 50 the control reads (max {a['test_history_max']}).")
    S.kv["data_audit"] = a
    modes = [m for m in args.text_modes.split(",") if m]
    bad = [m for m in modes if m not in FD.MODES + ("views",)]
    if bad:
        raise ValueError(f"--text-modes: unknown {bad}; choose from {FD.MODES + ('views',)}")
    taus = [{"tau": t} for t in args.pool_taus]
    best = {"name": "pool_lse", "spec": {"mode": "ta"}, "val": S.row("pool_lse")["val_ndcg@10"]}          # the control: the suite's own text

    def consider(row, spec):
        if row["val_ndcg@10"] > best["val"]:
            best.update(name=row["name"], spec=spec, val=row["val_ndcg@10"])

    def one_text(mode):
        t0 = time.time()
        T = S.tensor(text_table(S, args, {"mode": mode}))
        log(f"  text '{mode}': {T.shape[1]}-d vectors ready ({time.time() - t0:.0f}s)")
        S.select(f"txt_{mode}", "F-text", [{"text": mode}], lambda c, sp: R.mean_pool_scorer(T), "frozen")
        consider(S.select(f"txt_{mode}_lse", "F-text-lse", [{"text": mode, **t} for t in taus], lambda c, sp: P.late_scorer(T, "lse", c["tau"]), "pool_lse"),
                 {"mode": mode})

    def view_fusion():
        t0 = time.time()
        parts = [S.tensor(x) for x in view_vectors(S, args)]
        log(f"  views: title / abstract / category vectors ready ({time.time() - t0:.0f}s)")
        held = {}

        def fused(w):                                                          # one fused table at a time: five of them would not fit next to the rest
            key = tuple(w)
            if key not in held:
                held.clear()
                held[key] = FD.combine(parts, w)
            return held[key]
        S.select("txt_views", "F-text", [{"text": "views", "w": list(w)} for w in FD.VIEW_WEIGHTS], lambda c, sp: R.mean_pool_scorer(fused(c["w"])), "frozen")
        row = S.select("txt_views_lse", "F-text-lse", [{"text": "views", "w": list(w), **t} for w in FD.VIEW_WEIGHTS for t in taus],
                       lambda c, sp: P.late_scorer(fused(c["w"]), "lse", c["tau"]), "pool_lse")
        consider(row, {"mode": "views", "w": list(row["cfg"]["w"])})

    for mode in [m for m in modes if m not in ("views", "ta")]:
        one_text(mode)
        if S.device == "cuda":
            torch.cuda.empty_cache()
    if "views" in modes:
        view_fusion()
        if S.device == "cuda":
            torch.cuda.empty_cache()
    S.kv["best_text"] = {"name": best["name"], "spec": best["spec"], "val_ndcg@10": best["val"]}
    log(f"  best text variant by validation late-interaction nDCG@10: {best['name']} ({best['val']:.4f})"
        + (" -> the suite's own title+abstract stays the text for the learned readers" if best["spec"]["mode"] == "ta" else " -> fed to the learned readers as view 'text'"))
    # a longer history: the control reads the last 50 clicks (V10); zero-shot rules can read more with no training
    E0 = S.tensor(S.E0)
    for H in args.hist_lens:
        es = {sp: R.EvalSet(rows, max_hist=H, device=S.device) for sp, rows in (("val", S.data.validation), ("test", S.data.test))}
        S.select(f"hist{H}_mean", "F-history", [{"max_hist": H}], lambda c, sp: R.mean_pool_scorer(E0), "frozen", es=es)
        S.select(f"hist{H}_lse", "F-history-lse", [{"max_hist": H, **t} for t in taus], lambda c, sp: P.late_scorer(E0, "lse", c["tau"]), "pool_lse", es=es)


# ============================================================================ H7 Semantic IDs
def stage_sid(S, args):
    log("\n== H7 Semantic IDs: RQ-KMeans tokenizer, zero-shot profile, TIGER-style generative slate ranker ==")
    K, L, fam = args.sid_k, args.sid_levels, "H7-SemanticID"
    t0 = time.time()
    codes = SID.rq_kmeans(S.E0, S.seen_tr | S.seen_va, L=L, K=K, seed=0, device=S.device)     # content only; cold articles get codes by nearest centroid
    pref = SID.prefix_ids(codes, K)
    n = S.data.n_news - 1
    log("  codebook " + ", ".join(f"L{l + 1}: {len(np.unique(pref[l][1:])):,} prefixes" for l in range(L))
        + f" for {n:,} articles ({time.time() - t0:.0f}s)")
    Et = S.tensor(S.E0)
    mp = R.mean_pool_scorer(Et)
    levels = [tuple(range(1, l + 1)) for l in range(1, L + 1)]
    S.select("sid_profile", fam, [{"levels": lv} for lv in levels],
             lambda c, sp: SID.sid_profile_scorer(codes, K, c["levels"], S.device), "frozen")
    S.select("sid_profile+meanpool", fam, [{"levels": lv, "w": w} for lv in levels for w in args.sid_ws],
             lambda c, sp: fuse([(mp, 1.0), (SID.sid_profile_scorer(codes, K, c["levels"], S.device), c["w"])]), "frozen")
    if S.time_left() < 1200:
        log("  SID-GPT skipped: wall-clock budget")
        return
    val_ids, val_es = S.subset("val", args.sid_val_n)

    def val_fn(model):
        return S.run(SID.sid_scorer(model, codes, args.sid_nmax, False, device=S.device), "val", es={"val": val_es})["mean"]["ndcg@10"]

    model, info = SID.train_sid_gpt(codes, K, S.data.train_core, n_max=args.sid_nmax, epochs=args.sid_epochs, bs=args.sid_bs,
                                    lr=args.sid_lr, seed=0, device=S.device, val_fn=val_fn, d=args.sid_d, layers=args.sid_layers,
                                    heads=args.sid_heads, max_steps=args.sid_steps or None, deadline=S.deadline(0.25), log=log)
    params = sum(p.numel() for p in model.parameters())
    log(f"  SID-GPT: {params:,} parameters, best epoch {info['best_epoch']}, {info['steps']} steps")
    codes_t = torch.from_numpy(codes).to(S.device)
    base = SID.history_free_logprob(model, codes_t, len(codes)).to(S.device)
    t1 = time.time()
    raw = SID.sid_scorer(model, codes, args.sid_nmax, False, device=S.device)
    cache = {sp: ScoreCache(S.es[sp], raw) for sp in ("val", "test")}
    log(f"  scored every validation/test slate once in {time.time() - t1:.0f}s (cached for the fusion grid)")
    lik = {sp: cache[sp].scorer(S.device) for sp in cache}
    pmi = {sp: cache[sp].scorer(S.device, shift=lambda b: base[b["cand"]]) for sp in cache}
    extra = {"params": params, "best_epoch": info["best_epoch"], "steps": info["steps"]}
    S.select("sid_gpt", fam, [{"score": "log P(codes|history)"}], lambda c, sp: lik[sp], "frozen", extra)
    S.select("sid_gpt_pmi", fam, [{"score": "log P(codes|history) - log P(codes)"}], lambda c, sp: pmi[sp], "frozen", extra)
    src = {"lik": lik, "pmi": pmi}
    S.select("sid_gpt+meanpool", fam, [{"src": s, "w": w} for s in src for w in args.sid_ws],
             lambda c, sp: fuse([(mp, 1.0), (src[c["src"]][sp], c["w"])]), "frozen", extra)
    del model
    if S.device == "cuda":
        torch.cuda.empty_cache()


# ============================================================================ encoder sweep
def stage_sweep(S, args):
    log("\n== encoder sweep: frozen zero-shot mean-pool (does a bigger / LLM-based encoder help at all?) ==")
    from sklearn.decomposition import TruncatedSVD
    from sklearn.feature_extraction.text import TfidfVectorizer
    t0 = time.time()
    X = TfidfVectorizer(max_features=60_000, stop_words="english", sublinear_tf=True, min_df=2).fit_transform(S.meta["text"])
    Z = TruncatedSVD(min(args.lsa_dim, X.shape[1] - 1), random_state=0).fit_transform(X).astype(np.float32)
    Z /= np.linalg.norm(Z, axis=1, keepdims=True) + 1e-8
    Et = S.tensor(table_from(S, Z))
    S.record("enc_tfidf_lsa", "enc-sweep", {"model": f"tfidf+svd{Z.shape[1]}", "dim": int(Z.shape[1])},
             S.run(R.mean_pool_scorer(Et), "val"), S.run(R.mean_pool_scorer(Et), "test"), "frozen", {"encode_s": time.time() - t0})
    names = [n for n in args.sweep_encoders.split(",") if n and n != args.encoder]
    for name in names:
        if S.time_left() < 1200:
            log(f"  {name}: skipped (wall-clock budget)")
            continue
        try:
            vecs, secs = embed_news(S.meta, name, args.max_len, "", S.device, args.enc_bs, S.cache_dir)
        except Exception as e:                                            # a missing / unloadable model must not stop the sweep
            log(f"  {name}: failed to load/encode ({repr(e)[:120]}) - skipped")
            continue
        Et = S.tensor(table_from(S, vecs))
        S.record(f"enc_{short(name)}", "enc-sweep", {"model": name, "dim": int(vecs.shape[1])},
                 S.run(R.mean_pool_scorer(Et), "val"), S.run(R.mean_pool_scorer(Et), "test"), "frozen", {"encode_s": secs})
        del Et


# ============================================================================ H5 LLM user tower
def stage_llm_user(S, args):
    from llm_user import UserTower, probe_masks, train_user_tower
    log("\n== H5 fine-tuned LLM user tower: history-as-text, LoRA, causal / bidirectional / soft attention mask ==")
    dec = decoder_only(args.encoder)
    modes = [m for m in args.llm_user_modes.split(",") if m] if dec else ["native"]     # "causal@1" = the causal mode with training seed 1 (noise estimate)
    K, fam = args.hist_k, "H5-LLM-user"
    hq = HistQueries(S.data, S.titles, K)
    sets = {sp: S.subset(sp, n) for sp, n in (("val", args.llm_val_n), ("test", args.llm_test_n))}
    es = {sp: sets[sp][1] for sp in sets}
    log(f"  evaluation on {len(sets['val'][0]):,} validation / {len(sets['test'][0]):,} test impressions; modes={modes}")
    E_doc = S.tensor(S.E0)
    mp = R.mean_pool_scorer(E_doc)
    pre = query_prefix(args.encoder, args.query_format, True)
    qfn = lambda h: "; ".join(S.titles[i] for i in h[-K:])
    probe_texts = hq.texts(range(min(3, len(hq.uniq))))

    def encode_split(tw, sp):
        qi = hq.needed(sp, sets[sp][0])
        Q = hq.alloc(E_doc.shape[1])
        Q[qi] = tw.encode(hq.texts(qi), bs=args.llm_enc_bs, deadline=S.deadline(0.3))
        missing = int((np.abs(Q[qi]).sum(1) == 0).sum())
        if missing:
            log(f"      WARNING: {missing:,}/{len(qi):,} {sp} queries were not encoded (wall-clock budget) - their rows score as ties")
            S.kv.setdefault("queries_missing", {})[f"{tw.label}/{sp}"] = missing
        return Q

    for label in modes:
        mode, _, sd = label.partition("@")
        seed = int(sd) if sd else 0
        if S.time_left() < 1800:
            log(f"  {label}: skipped (wall-clock budget)")
            continue
        tw = UserTower(args.encoder, mode, prefix=pre, max_len=args.q_max_len, device=S.device)
        tw.label = label
        if dec and mode != "causal":
            ok, rep = probe_masks(tw, probe_texts)
            log(f"  attention-mask probe for '{label}': {'OK' if ok else 'FAILED -> variant skipped'} {({k: round(v, 5) if isinstance(v, float) else v for k, v in rep.items()})}")
            if not ok:
                del tw
                continue

        def eval_fn(tw):
            Q = encode_split(tw, "val")
            tw.eval_payload = Q
            return S.run(hq.scorer(Q, E_doc, "val", S.device, ids=sets["val"][0]), "val", es=es)["mean"]["ndcg@10"]

        info = train_user_tower(tw, qfn, S.data.train_core, E_doc, kind="lora", steps=args.llm_user_steps, bs=args.llm_user_bs,
                                lr=args.llm_user_lr, r=args.llm_user_r, eval_fn=eval_fn, shared_negs=bool(args.llm_user_shared),
                                seed=seed, deadline=S.deadline(0.3), log=log)
        info["truncated"] = bool(info["steps"] < args.llm_user_steps)
        if info["truncated"]:
            log(f"      WARNING: '{label}' trained {info['steps']}/{args.llm_user_steps} steps (wall-clock budget) - not comparable with the other masks")
        Qv = info.pop("payload") if info.get("payload") is not None else encode_split(tw, "val")
        Qt = encode_split(tw, "test")
        info.pop("losses", None)
        sc = {"val": hq.scorer(Qv, E_doc, "val", S.device, ids=sets["val"][0]),
              "test": hq.scorer(Qt, E_doc, "test", S.device, ids=sets["test"][0])}
        S.select(f"ut_{label}", fam, [{"mode": mode, "seed": seed, "steps": args.llm_user_steps, "r": args.llm_user_r}], lambda c, sp: sc[sp],
                 "frozen", info, es=es)
        S.select(f"ut_{label}+meanpool", fam, [{"mode": mode, "seed": seed, "w": w} for w in args.ut_ws],
                 lambda c, sp: fuse([(mp, 1.0), (sc[sp], c["w"])]), "frozen", info, es=es)
        del tw
        if S.device == "cuda":
            torch.cuda.empty_cache()


# ============================================================================ H10 learned user model
def _views(S, args):
    """Representations to feed the learned user model -> {name: {"val": E, "test": E}} (RAG views depend on the period)."""
    out = {}
    wanted = [v for v in args.um_views.split(",") if v]
    for v in wanted:
        try:
            if v == "frozen":
                out[v] = {"val": S.E0, "test": S.E0}
            elif v == "head":
                E = load_export(S, S.kv["best_head"])
                out[v] = {"val": E, "test": E}
            elif v == "encoder":
                rows = [r for r in S.records if r["name"].startswith(("lora", "dora")) and r.get("val_delta") is not None]
                E = load_export(S, max(rows, key=lambda r: r["val_delta"])["name"])
                out[v] = {"val": E, "test": E}
            elif v == "text":
                bt = S.kv["best_text"]                                         # KeyError if the moredata stage has not run
                if bt["spec"]["mode"] == "ta":
                    log("  view 'text': the suite's own title+abstract won on validation, so it is the 'frozen' view - skipped")
                    continue
                T = text_table(S, args, bt["spec"])
                out[v] = {"val": T, "test": T}
            elif v == "entity_rag":
                cfg, memo = S.row("entity_rag")["cfg"], S.memo()
                out[v] = {sp: memo[sp].entity_view(cfg["beta"])[0] for sp in ("val", "test")}
            elif v == "knn_rag":
                cfg, memo = S.row("knn_rag")["cfg"], S.memo()
                out[v] = {sp: memo[sp].knn_view(cfg["beta"], cfg["k"]) for sp in ("val", "test")}
            else:
                log(f"  unknown view '{v}' ignored")
        except (KeyError, FileNotFoundError, ValueError) as e:
            log(f"  view '{v}' unavailable ({repr(e)[:80]}) - did its stage run?")
    return out


def stage_usermodel(S, args):
    log("\n== H10 learned user model on each representation: V10 llmenc_ca (candidate-aware), Fastformer / NRMS / additive readers, H11 multi-interest readers ==")
    pack = S.store.get("um_pack") or UM.pack_train(S.data.train_core)
    S.store["um_pack"] = pack
    views = _views(S, args)
    fam, ref = "H10-user-model", "um_frozen"

    def seeds_for(vname):
        """The views the new comparisons are made on (the suite's own vectors and the best text variant) get ``--um-seeds-focus`` seeds,
        the older consistency-check views (head, encoder, RAG) keep ``--um-seeds``."""
        return max(args.um_seeds_focus, args.um_seeds) if vname in ("frozen", "text") else args.um_seeds

    def fit(kind, vname, V):
        vals, tests, infos = [], [], []
        for sd in range(seeds_for(vname)):
            t0 = time.time()

            def val_fn(model):
                return S.run(UM.scorer(model), "val")["mean"]["ndcg@10"]

            model, info = UM.train_user_model(V["val"], pack, val_fn, seed=sd, max_epochs=args.um_epochs, device=S.device,
                                              deadline=S.deadline(0.2), log=lambda *a: None, kind=kind)
            vals.append(S.run(UM.scorer(model, V["val"]), "val"))
            tests.append(S.run(UM.scorer(model, V["test"]), "test"))
            if isinstance(model, UM.LLMEncMI) and model.K > 1:                  # are the K interests different at all?
                b = next(iter(S.es["val"]))
                info.update(model.diversity(b["hist"], b["hist_mask"]))
            infos.append(info)
            log(f"    {kind}/{vname} seed {sd}: {info['epochs']} epochs, val nDCG@10={info['best_val']:.4f}"
                + (f", interests: cosine {info['interest_cos']:.2f}, attention TV {info['attn_tv']:.2f}" if "interest_cos" in info else "") + f" ({time.time() - t0:.0f}s)")
        return vals, tests, infos

    for vname, V in views.items():
        if S.time_left() < 900:
            log(f"  {vname}: skipped (wall-clock budget)")
            continue
        vals, tests, infos = fit("ca", vname, V)
        per_seed = [t["mean"]["ndcg@10"] for t in tests]
        row = S.record(f"um_{vname}", fam, {"view": vname, "seeds": seeds_for(vname), "reader": "ca"}, avg_results(vals), avg_results(tests),
                       ref if ref in S.cache else f"um_{vname}",
                       {"epochs": [i["epochs"] for i in infos], "ndcg10_per_seed": per_seed, **({"ndcg10_std": float(np.std(per_seed))} if len(per_seed) > 1 else {})})
        if vname == "frozen" and args.um_repro:
            auc_ref, nd_ref = (float(x) for x in args.um_repro.split(","))
            ok = abs(row["auc"] - auc_ref) < 0.01 and abs(row["ndcg@10"] - nd_ref) < 0.01
            log(f"  reproduction check vs V10 llmenc_ca (AUC {auc_ref}, nDCG@10 {nd_ref}): "
                f"{'MATCH (within 0.01)' if ok else 'DIFFERS - single V10 seed, our seeds average; inspect if the gap is > 0.01'}")
        if S.device == "cuda":
            torch.cuda.empty_cache()
    for item in [x for x in args.um_extra.split(",") if x]:                           # "reader:view", e.g. ff:frozen
        kind, _, vname = item.partition(":")
        V = views.get(vname)
        if not UM.is_reader(kind) or kind == "ca" or V is None:
            log(f"  {item}: skipped (reader must be add|nrms|ff|mi<K> and the view must have been built)")
            continue
        if S.time_left() < 900:
            log(f"  {item}: skipped (wall-clock budget)")
            continue
        vals, tests, infos = fit(kind, vname, V)
        name = f"um_{kind}_{vname}"
        extra = {"epochs": [i["epochs"] for i in infos], "ndcg10_per_seed": [t["mean"]["ndcg@10"] for t in tests]}
        if len(tests) > 1:
            extra["ndcg10_std"] = float(np.std(extra["ndcg10_per_seed"]))
        if any("interest_cos" in i for i in infos):
            extra["interest_cos"] = [i.get("interest_cos") for i in infos]
            extra["attn_tv"] = [i.get("attn_tv") for i in infos]
        S.record(name, fam, {"view": vname, "seeds": seeds_for(vname), "reader": kind}, avg_results(vals), avg_results(tests),
                 ref if ref in S.cache else name, extra)
        if S.device == "cuda":
            torch.cuda.empty_cache()


# ============================================================================ generation-based stages
COMM_PROMPT = ("Below are headlines from one cluster of related news articles.\n{titles}\n\n"
               "In one sentence (at most 40 words), describe the common topic of this cluster and what kind of reader would follow it.")
EXPAND_PROMPT = ("News headline: {title}\nSummary: {abstract}\n\n"
                 "In one or two sentences (at most 50 words), state the main topic of this article and what kind of reader would be interested in it.")
PROFILE_PROMPT = ("A reader recently clicked these news headlines (oldest first):\n{titles}\n\n"
                  "In two short sentences (at most 60 words), describe this reader's interests and what other news they would want to read next.")
JUDGE_PROMPT = ("A news reader recently read these articles (oldest first):\n{hist}\n\n"
                "Candidate article: \"{title}\" (category: {cat})\n\n"
                "Would this reader be interested in clicking the candidate article? Answer with exactly one word: Yes or No.")


def stage_graphrag_llm(S, args):
    log("\n== H4b GraphRAG with LLM-written community summaries ==")
    gen, enc = S.generator(), S.doc_encoder()
    comm, memo = S.communities(), S.memo()
    T, fam = S.titles, "H4b-GraphRAG-LLM"
    summ = {}
    for sp in ("val", "test"):
        labels, cent, method = comm[sp]
        mem_ids = memo[sp].mem_ids
        K = len(cent)
        size = np.bincount(labels[mem_ids], minlength=K)
        order = np.argsort(-size)[:args.gr_max_comm]
        prompts = []
        for k in order:
            members = mem_ids[labels[mem_ids] == k]
            top = members[np.argsort(-(S.E0[members] @ cent[k]))[:args.gr_titles]]
            prompts.append(COMM_PROMPT.format(titles="\n".join(f"- {T[i]}" for i in top)))
        out = gen.generate(prompts, max_new_tokens=64, bs=args.gen_bs, deadline=S.deadline(0.1), desc=f"community summaries [{sp}]")
        cent_llm = cent.copy()
        ok = [(k, t) for k, t in zip(order, out) if t]
        if ok:
            V = enc.encode([t for _, t in ok], bs=64)
            cent_llm[[k for k, _ in ok]] = V
        summ[sp] = cent_llm
        log(f"  [{sp}] {len(ok)}/{K} communities summarised by the LLM ({method}); example: {ok[0][1][:110] if ok else '-'!r}")
    S.select("community_llm", fam, [{"lam": l} for l in args.lams],
             lambda c, sp: G.community_scorer(S.E0, comm[sp][0], summ[sp], c["lam"], S.device), "frozen")

    def ctx_view(c, sp):
        E = S.E0.copy()
        E[1:] = G._norm(S.E0[1:] + c["beta"] * summ[sp][np.clip(comm[sp][0][1:], 0, None)])
        return E
    S.select("community_llm_ctx", fam, [{"beta": b} for b in args.betas],
             lambda c, sp: R.mean_pool_scorer(S.tensor(ctx_view(c, sp))), "frozen")
    del enc
    if S.device == "cuda":
        torch.cuda.empty_cache()


def _gen_subsets(S, args, gen, items_of, per_impr, req, kind, minutes, frac_val=0.3):
    """Choose how many validation/test impressions fit the generation allowance (probe throughput on a few items)."""
    allow = min(minutes * 60.0, 0.45 * S.time_left())
    spi = gen.seconds_per_item(kind, 1.0)
    out = {}
    for sp, share in (("test", 1.0 - frac_val), ("val", frac_val)):
        perm = S.perm(sp)
        lists = [items_of(sp, i) for i in perm[:req[sp]]]
        cap = allow * share / max(spi, 1e-3)
        n, n_items = llm_gen.plan_prefix(lists, cap, per_impr)
        out[sp] = (np.sort(perm[:n]), n_items)
    log(f"  allowance {allow / 60:.0f} min at {spi:.2f}s/item -> validation {len(out['val'][0]):,} / test {len(out['test'][0]):,} impressions "
        f"(requested {req['val']:,}/{req['test']:,})")
    return out


def stage_augment(S, args):
    log("\n== H6 LLM-augmented articles and readers (KAR-style expansions + LLM-written profiles) ==")
    gen, enc_d, enc_q = S.generator(), S.doc_encoder(), S.query_encoder()
    T, A, fam = S.titles, [""] + S.meta["abstracts"], "H6-LLM-aug"
    hk, K = args.gen_hist_k, args.hist_k
    rows = {"val": S.data.validation, "test": S.data.test}
    items_of = lambda sp, i: list(rows[sp][i][3]) + list(rows[sp][i][2][-hk:])
    # probe throughput on a handful of expansions (kept in the cache) before sizing the subsets
    probe = list(dict.fromkeys(items_of("test", S.perm("test")[0])))[:32]
    gen.generate([EXPAND_PROMPT.format(title=T[i], abstract=A[i][:300]) for i in probe], max_new_tokens=args.gen_new_tokens,
                 bs=args.gen_bs, desc="probe")
    plan = _gen_subsets(S, args, gen, items_of, 1.3, {"val": args.gen_val_n, "test": args.gen_test_n}, "gen", args.augment_minutes)
    need = sorted({i for sp in plan for r in plan[sp][0] for i in items_of(sp, r)})
    log(f"  expanding {len(need):,} distinct articles")
    out = gen.generate([EXPAND_PROMPT.format(title=T[i], abstract=A[i][:300]) for i in need], max_new_tokens=args.gen_new_tokens,
                       bs=args.gen_bs, deadline=S.deadline(0.3), desc="article expansions")
    exp = {i: t for i, t in zip(need, out) if t}
    ok = np.array(sorted(exp))
    Eg = np.zeros_like(S.E0)
    Eg[ok] = enc_d.encode([T[i] + ". " + exp[i] for i in ok], bs=64)
    log(f"  {len(ok):,}/{len(need):,} expansions embedded; example: {exp[ok[0]][:120]!r}" if len(ok) else "  no expansion produced")
    # keep only impressions whose every needed article was expanded
    ids, es = {}, {}
    for sp in plan:
        keep = [r for r in plan[sp][0] if all(i in exp for i in items_of(sp, r))]
        ids[sp] = np.array(keep, np.int64)
        es[sp] = subset_evalset(rows[sp], ids[sp], S.device)
    log(f"  fully covered impressions: validation {len(ids['val']):,}, test {len(ids['test']):,}")
    if len(ids["test"]) < 50 or len(ids["val"]) < 20:
        log("  too few covered impressions - stage aborted")
        return
    E0t = S.tensor(S.E0)
    mp0 = R.mean_pool_scorer(E0t)
    aug = lambda a: S.tensor(G._norm(S.E0 + a * Eg))
    S.select("kar_item", fam, [{"alpha": a} for a in args.kar_alphas], lambda c, sp: R.mean_pool_scorer(aug(c["alpha"])), "frozen", es=es)
    best_alpha = S.row("kar_item")["cfg"]["alpha"]
    # reader profiles written by the LLM from the last K headlines
    hq = HistQueries(S.data, T, K)
    need_q = np.unique(np.concatenate([hq.needed(sp, ids[sp]) for sp in ids]))
    prof = gen.generate([PROFILE_PROMPT.format(titles="\n".join(f"- {T[i]}" for i in hq.uniq[q])) for q in need_q],
                        max_new_tokens=args.gen_new_tokens + 16, bs=args.gen_bs, deadline=S.deadline(0.3), desc="reader profiles")
    got = [(q, t) for q, t in zip(need_q, prof) if t]
    Q = hq.alloc(S.E0.shape[1])
    if got:
        Q[[q for q, _ in got]] = enc_q.encode([t for _, t in got], bs=64)
        log(f"  {len(got):,}/{len(need_q):,} reader profiles embedded; example: {got[0][1][:130]!r}")
    Eb = aug(best_alpha)
    mp_b = R.mean_pool_scorer(Eb)
    prof_sc = lambda E, sp: hq.scorer(Q, E, sp, S.device, ids=ids[sp])
    S.select("kar_user", fam, [{}], lambda c, sp: prof_sc(E0t, sp), "frozen", es=es)
    S.select("kar_user+meanpool", fam, [{"w": w} for w in args.kar_ws], lambda c, sp: fuse([(mp0, 1.0), (prof_sc(E0t, sp), c["w"])]),
             "frozen", es=es)
    S.select("kar_item+user", fam, [{"alpha": best_alpha, "w": w} for w in args.kar_ws],
             lambda c, sp: fuse([(mp_b, 1.0), (prof_sc(Eb, sp), c["w"])]), "frozen", es=es)
    del enc_d, enc_q
    if S.device == "cuda":
        torch.cuda.empty_cache()


def stage_rerank(S, args):
    log("\n== H9 zero-shot LLM judge re-ranking the top-k of the frozen embedding (pointwise Yes/No log-odds) ==")
    gen = S.generator()
    T, cats, fam, k, hk = S.titles, [""] + S.meta["cats"], "H9-LLM-rerank", args.rerank_k, args.rerank_hist
    first = R.mean_pool_scorer(S.tensor(S.E0))
    rows = {"val": S.data.validation, "test": S.data.test}

    def build(sp, n):
        ids, es = S.subset(sp, n)
        cache = ScoreCache(es, first)
        top, prompts = {}, []
        for pos, orig in enumerate(ids):
            s = cache.rows[pos]
            tk = np.argsort(-s, kind="stable")[:k]
            row = rows[sp][orig]
            htxt = "\n".join(f"- {T[i]}" for i in list(row[2])[-hk:]) or "- (nothing yet)"
            top[int(orig)] = tk
            prompts += [JUDGE_PROMPT.format(hist=htxt, title=T[row[3][j]], cat=cats[row[3][j]]) for j in tk]
        return ids, cache, top, prompts

    t_ids, t_cache, t_top, t_prompts = build("test", args.rerank_test_n)
    gen.yes_no(t_prompts[:64], bs=args.gen_bs, desc="probe")
    spp = gen.seconds_per_item("yn", 0.05)
    allow = min(args.rerank_minutes * 60.0, 0.5 * S.time_left())
    prompts_ok = int(allow / max(spp, 1e-3))
    n_test = int(min(args.rerank_test_n, 0.7 * prompts_ok / k))
    n_val = int(min(args.rerank_val_n, 0.3 * prompts_ok / k))
    log(f"  {spp * 1000:.0f} ms/prompt -> allowance {allow / 60:.0f} min: re-rank {n_val:,} validation / {n_test:,} test impressions (top-{k})")
    if n_test < 50 or n_val < 20:
        log("  allowance too small - stage skipped")
        return
    sets = {}
    for sp, n in (("val", n_val), ("test", n_test)):
        ids, cache, top, prompts = build(sp, n)
        lo = gen.yes_no(prompts, bs=args.gen_bs, deadline=S.deadline(0.45), desc=f"judge [{sp}]")
        off = np.concatenate([[0], np.cumsum([len(top[int(o)]) for o in ids])])
        keep = [i for i in range(len(ids)) if not np.isnan(lo[off[i]:off[i + 1]]).any()]      # impressions fully judged
        sets[sp] = {"ids": ids[keep], "first": {int(ids[i]): cache.rows[i] for i in keep},
                    "top": {int(ids[i]): top[int(ids[i])] for i in keep}, "lo": {int(ids[i]): lo[off[i]:off[i + 1]] for i in keep}}
        log(f"  [{sp}] {len(keep):,}/{len(ids):,} impressions fully judged")
    es = {sp: subset_evalset(rows[sp], sets[sp]["ids"], S.device) for sp in sets}
    zs = lambda x: (x - x.mean()) / (x.std() + 1e-6)

    def make(c, sp):
        d, w = sets[sp], c["w"]

        def f(b):
            out = np.full((len(b["rows"]), b["cand"].shape[1]), -1e4, np.float32)
            for j, p in enumerate(b["rows"]):
                o = int(d["ids"][p])
                s1, tk, lo = d["first"][o], d["top"][o], d["lo"][o]
                base = s1 - 1e3                                    # everything outside the re-ranked group stays below it
                base[tk] = (zs(lo) + 1e-3 * zs(s1[tk])) if w == "llm" else (zs(s1[tk]) + w * zs(lo))
                out[j, :len(base)] = base
            return torch.from_numpy(out).to(S.device)
        return f
    S.select("rerank_llm", fam, [{"w": "llm", "k": k, "model": args.gen_model}], make, "frozen", es=es)
    S.select("rerank_fused", fam, [{"w": w, "k": k, "model": args.gen_model} for w in args.rerank_ws], make, "frozen", es=es)


In [ ]:
%%writefile stages_ref.py
"""Reference baselines as suite stages.

Headline rows (family ``ref-baseline``) -- NRMS, NAML, Fastformer trained with the usual published recipe on V10's split:
``ref_nrms``, ``ref_naml``, ``ref_ff``.
Ablation ladder (family ``ref-ladder``) -- NRMS moved step by step from V10's recipe to the published one, to attribute V10's gap to the
literature: ``ref_l0`` (V10 recipe re-implemented; must land near V10's own NRMS, otherwise the harness is suspect), ``ref_l1`` (+ title-only
regex tokens, padding mask, negatives re-drawn each epoch), ``ref_l2`` (+ published width / heads / lr, random word vectors); the next rung
(+ GloVe) is ``nrms_ref``.

Each model is its own stage, so a crash or an exhausted budget loses at most one model and a re-run resumes at the next one.

Notebook v13 runs only Fastformer (``ref_ff`` with several seeds, then ``ref_ff_refit``); the NRMS / NAML / ladder stages stay available
(``V11_REF_ALL=1``) and their measured numbers are in RESULTS_v7.md.
"""
from __future__ import annotations

import time
from dataclasses import replace

import numpy as np
import torch

import common
import rep_eval as R
import refdata as RD
import refmodels as RM
from common import avg_results, log

SPECS = {
    "nrms": dict(name="nrms_ref", family="ref-baseline", preset="nrms", pop=True),
    "naml": dict(name="naml_ref", family="ref-baseline", preset="naml", pop=True),
    "ff": dict(name="fastformer_ref", family="ref-baseline", preset="fastformer", pop=True),
    "nrms_refit": dict(name="nrms_ref_refit", family="ref-baseline", preset="nrms", pop=False, refit="nrms_ref"),     # train_core + validation, epochs fixed by nrms_ref
    "ff_refit": dict(name="fastformer_ref_refit", family="ref-baseline", preset="fastformer", pop=False, refit="fastformer_ref"),   # same for Fastformer
    "nrms1": dict(name="nrms_ref@1", family="ref-baseline", preset="nrms", pop=False, seed0=1),       # same recipe, other seed: the training-noise floor
    "l0": dict(name="ladder0_v10recipe", family="ref-ladder", preset="v10", pop=False),
    "l1": dict(name="ladder1_data", family="ref-ladder", preset="data", pop=False),
    "l2": dict(name="ladder2_recipe", family="ref-ladder", preset="recipe", pop=False),
}
LADDER = ("ladder0_v10recipe", "ladder1_data", "ladder2_recipe", "nrms_ref")


def resources(S, args):
    """Token tables, GloVe and training sets, built once per process (``S.store``)."""
    if "ref" in S.store:
        return S.store["ref"]
    t0 = time.time()
    text = RD.RefText(S.meta, title_len=args.ref_title_len, body_len=args.ref_body_len, use_glove=bool(args.ref_glove),
                      glove_dim=_dims(args)[0], glove_source=args.ref_glove_path or None, log=log)
    core = RD.read_train_core(args.mind_train, S.meta["ids"], S.data)
    full = RD.TrainSet(core)
    RD.check_train_alignment(full, S.data)                              # always on the whole set: it is the check that the article ids line up
    dyn = full if args.ref_train_frac >= 1 else RD.TrainSet(RD.subsample(core, args.ref_train_frac))
    val_imps = [(r[1], list(r[2]), list(r[3]), list(r[4]), "") for r in S.data.validation]                    # (user, history, candidates, labels, day)
    static = RD.StaticTrainSet(RD.subsample(S.data.train_core, args.ref_train_frac))
    log(f"  reference data: {text.stats} | {dyn.n_samples:,} dynamic-negative samples from {dyn.n_imp:,} impressions "
        f"(V10 static: {static.n_samples:,}) in {time.time() - t0:.0f}s")
    S.store["ref"] = {"text": text, "dyn": dyn, "static": static, "core": RD.subsample(core, args.ref_train_frac),            # the refit sees the same fraction (QUICK)
                      "val_imps": RD.subsample(val_imps, args.ref_train_frac)}
    return S.store["ref"]


def _dims(args):
    """``--ref-dims emb,heads,head_dim,naml_filters,ff_layers`` (published-style sizes 300,16,16,400,1; the self-test shrinks them)."""
    d = [int(x) for x in args.ref_dims.split(",")]
    if len(d) != 5:
        raise ValueError("--ref-dims needs 5 integers: emb,heads,head_dim,naml_filters,ff_layers")
    return d


def _cfg(spec, args, text):
    cfg = RM.preset(spec["preset"])
    if spec["preset"] not in ("v10", "data"):                          # the V10 rungs keep V10's 64-d / 2-head sizes by definition
        emb, heads, hd, filt, layers = _dims(args)
        cfg = replace(cfg, emb_dim=emb, heads=heads, head_dim=hd, naml_filters=filt, ff_layers=layers)
    if args.ref_max_epochs:
        cfg = replace(cfg, max_epochs=min(cfg.max_epochs, args.ref_max_epochs), min_epochs=min(cfg.min_epochs, args.ref_max_epochs))
    if args.ref_bs:
        cfg = replace(cfg, bs=args.ref_bs)
    if cfg.init == "glove" and text.glove_mat is None:
        cfg = replace(cfg, init="random")                              # no GloVe source worked: same model, random word vectors
    return cfg


def _fit_with_oom_fallback(S, cfg, trainset, val_fn, seed, deadline, text, **kw):
    """Train; on a CUDA out-of-memory error rebuild the network and retry with half / quarter the batch size (reported in the row)."""
    last = None
    for bs in dict.fromkeys([cfg.bs, max(cfg.bs // 2, 8), max(cfg.bs // 4, 8)]):
        net = None
        try:
            net = RM.build_net(cfg, S.device, seed, text, S.data)
            net, info = RM.train_ref(net, trainset, val_fn, replace(cfg, bs=bs), seed=seed, device=S.device, deadline=deadline, log=log, **kw)
            info["bs_used"] = bs
            return net, info
        except torch.cuda.OutOfMemoryError as e:
            last = e
            log(f"    out of memory at batch size {bs}: retrying with a smaller batch")
            del net
            torch.cuda.empty_cache()
    raise last


def stage_refit(S, args, spec, res, cfg, name):
    """Returns False when it could not run.  Refit on train_core + validation for the epoch count that ``nrms_ref`` selected on validation, then test once.  The validation columns
    of the row repeat the epoch-selection run (the refit model has seen those days), so only the test columns are meaningful."""
    base_name = spec["refit"] + ("_noglove" if res["text"].glove_mat is None else "")
    try:
        base = S.row(base_name)
    except KeyError:
        log(f"  {name}: skipped ({base_name} has not run: its validated epoch count is needed)")
        return False
    epochs = max(int(round(float(np.mean(base["best_epochs"])))), 1)       # several seeds: the mean of their validated epoch counts
    trainset = RD.TrainSet(res["core"] + res["val_imps"])
    t0 = time.time()
    net, info = _fit_with_oom_fallback(S, cfg, trainset, None, 0, S.deadline(0.45, cap_s=args.ref_model_minutes * 60), res["text"], fixed_epochs=epochs)
    sc = RM.make_scorer(net)
    test = S.run(sc, "test")
    log(f"    refit on {trainset.n_samples:,} samples for {info['epochs']} of {epochs} epochs ({time.time() - t0:.0f}s)")
    S.record(name, spec["family"], {**cfg.__dict__, "refit_epochs": epochs, "refit_on": "train_core+validation"}, S.cache[base_name]["val"], test, "frozen",
             {"params": info["params"], "train_s": info["train_s"], "init": base.get("init"), "refit_of": base_name, "val_columns": "epoch-selection run (not held out)",
              "partial": info["epochs"] < epochs, "bs_used": info["bs_used"]})


def stage_ref(S, args, which):
    spec = SPECS[which]
    res = resources(S, args)
    text = res["text"]
    cfg = _cfg(spec, args, text)
    name = spec["name"]
    if SPECS[which]["preset"] in ("nrms", "naml", "fastformer") and cfg.init == "random":
        name = name.replace("@1", "") + "_noglove" + ("@1" if "@1" in spec["name"] else "")
    if which == "l2" and text.glove_mat is None:
        log("  ladder2_recipe is identical to nrms_ref_noglove here (no GloVe): skipped")
        return False
    if S.time_left() < args.ref_min_minutes * 60:
        log(f"  {name}: skipped (wall-clock budget)")
        return False
    log(f"\n== reference baseline {name}: {cfg.kind} text={cfg.text} dim={cfg.emb_dim}/{cfg.heads}x{cfg.head_dim} init={cfg.init} "
        f"dyn_neg={cfg.dyn_neg} lr={cfg.lr} bs={cfg.bs} ==")
    if spec.get("refit"):
        return stage_refit(S, args, spec, res, cfg, name)
    trainset = res["dyn"] if cfg.dyn_neg else res["static"]
    twin = spec["pop"] and S.want_pop("frozen")
    vals, tests, pv, pt, infos = [], [], [], [], []
    for sd in range(spec.get("seed0", 0), spec.get("seed0", 0) + args.ref_seeds):
        t0 = time.time()
        val_fn = lambda n: S.run(RM.make_scorer(n), "val")["mean"]["ndcg@10"]
        net, info = _fit_with_oom_fallback(S, cfg, trainset, val_fn, sd, S.deadline(0.45 / args.ref_seeds, cap_s=args.ref_model_minutes * 60 / args.ref_seeds), text)
        sc = RM.make_scorer(net)
        vals.append(S.run(sc, "val"))
        tests.append(S.run(sc, "test"))
        if twin:
            fz = common.fuse([(sc, 1.0), (common.pop_scorer, 1.0)])
            pv.append(S.run(fz, "val"))
            pt.append(S.run(fz, "test"))
        infos.append(info)
        log(f"    seed {sd}: {info['epochs']} epochs (best {info['best_epoch']}), {info['params']:,} parameters, val nDCG@10={info['best_val']:.4f} "
            f"({time.time() - t0:.0f}s)")
        del net, sc
        if S.device == "cuda":
            torch.cuda.empty_cache()
    extra = {"seeds": args.ref_seeds, "epochs": [i["epochs"] for i in infos], "best_epochs": [i["best_epoch"] for i in infos],
             "params": infos[0]["params"], "train_s": sum(i["train_s"] for i in infos), "init": text.glove_label if cfg.init == "glove" else cfg.init,
             "ndcg10_per_seed": [t["mean"]["ndcg@10"] for t in tests],
             "partial": any(h["partial"] for i in infos for h in i["history"]), "bs_used": infos[0]["bs_used"]}
    if args.ref_seeds > 1:
        extra["ndcg10_std"] = float(np.std(extra["ndcg10_per_seed"]))
    row = S.record(name, spec["family"], {**{k: v for k, v in cfg.__dict__.items()}}, avg_results(vals), avg_results(tests), "frozen", extra)
    if twin:
        S.record(f"{name}+pop", f"{spec['family']}+pop", {"base": name}, avg_results(pv), avg_results(pt), "frozen+pop", extra)
    if which == "l0" and args.ref_repro:
        auc_ref, nd_ref = (float(x) for x in args.ref_repro.split(","))
        ok = abs(row["auc"] - auc_ref) < 0.015 and abs(row["ndcg@10"] - nd_ref) < 0.015
        log(f"  harness check vs V10's own NRMS (AUC {auc_ref}, nDCG@10 {nd_ref}): "
            f"{'MATCH (within 0.015)' if ok else 'DIFFERS - a single V10 seed is noisy, but a gap > 0.015 means this harness or V10 trains differently: read the epoch log'}")


def ladder_table(S, repro=None, path=None):
    """Markdown: V10's recipe -> published recipe, one change set per rung, paired Δ nDCG@10 vs the previous rung (test split)."""
    order = [n for n in LADDER if n in S.cache]
    if "nrms_ref" not in S.cache and "nrms_ref_noglove" in S.cache:
        order.append("nrms_ref_noglove")
    rows = [(n, S.cache[n]["test"]) for n in order]
    if len(rows) < 2:
        return None
    what = {"ladder0_v10recipe": "V10 recipe (64-d, 2 heads, N(0,1) word vectors, 20 tokens of title+abstract, rare words -> PAD, no title mask, 4 negatives drawn once, lr 1e-3)",
            "ladder1_data": "+ title-only regex tokens (30), UNK instead of PAD, padding mask, negatives re-drawn every epoch",
            "ladder2_recipe": "+ published NRMS size (300-d input, 16 heads x 16), lr 1e-4, no output projection / attention dropout, N(0,0.1) word vectors",
            "nrms_ref": "+ GloVe-300 word vectors", "nrms_ref_noglove": "(GloVe unavailable: same model with random word vectors)"}
    lines = ["| rung | change | test AUC | test nDCG@10 | Δ nDCG@10 vs previous rung [95% CI] | Δ cold AUC [95% CI] |", "|---|---|---|---|---|---|"]
    if repro:
        a, n = (float(x) for x in repro.split(","))
        lines.append(f"| V10 (reported, its own code) | NRMS as run in V10 | {a:.4f} | {n:.4f} | - | - |")
    prev = None
    for n, t in rows:
        d = R.compare(t, prev, "ndcg@10") if prev is not None else None
        c = R.compare_slice(t, prev, "cold") if prev is not None else None
        fmt = lambda x: "-" if x is None else f"{x[0]:+.4f} [{x[1]:+.4f}, {x[2]:+.4f}]"
        lines.append(f"| {n} | {what.get(n, '')} | {t['mean']['auc']:.4f} | {t['mean']['ndcg@10']:.4f} | {fmt(d)} | {fmt(c)} |")
        prev = t
    md = "\n".join(lines)
    log("\n================ ABLATION LADDER: V10's NRMS recipe -> published recipe (same split, same evaluator) ================\n" + md)
    if path:
        with open(path, "w", encoding="utf-8") as f:
            f.write("# NRMS ablation ladder (MINDsmall_dev test)\n\n" + md + "\n")
    return md


STAGES = {"ref_nrms": lambda S, a: stage_ref(S, a, "nrms"), "ref_naml": lambda S, a: stage_ref(S, a, "naml"),
          "ref_ff": lambda S, a: stage_ref(S, a, "ff"), "ref_nrms_s1": lambda S, a: stage_ref(S, a, "nrms1"),
          "ref_nrms_refit": lambda S, a: stage_ref(S, a, "nrms_refit"), "ref_ff_refit": lambda S, a: stage_ref(S, a, "ff_refit"),
          "ref_l0": lambda S, a: stage_ref(S, a, "l0"),
          "ref_l1": lambda S, a: stage_ref(S, a, "l1"), "ref_l2": lambda S, a: stage_ref(S, a, "l2")}


In [ ]:
%%writefile suite.py
"""V11 decision suite: which LLM-era direction is worth a thesis on news recommendation?

Every variant is evaluated on the exact V10 protocol (chronological validation split, MINDsmall_dev as the one-shot
test).  Hyper-parameters are picked on validation, the test split is touched once per selected configuration, and every
comparison carries a paired 95% CI plus a cold-article slice (clicked articles the training rows never saw).

Hypotheses (see README):
  H1  history-as-text, instruction-aware query                         (LLM as user encoder, zero-shot)
  H2  co-click contrastive adaptation: heads, LoRA/DoRA, MRL, hard neg  (LLM2Rec-style)
  H3  entity / kNN retrieval-augmented article representations          (RAG)
  H4  GraphRAG-lite (Louvain communities, similar-user retrieval)  +  H4b with LLM-written community summaries
  H5  LoRA-tuned LLM user tower, causal / bidirectional / soft attention mask
  H6  LLM-augmented articles and readers (KAR-style expansions, profiles)
  H7  Semantic IDs: RQ-KMeans profile + TIGER-style generative slate ranker
  H8  pooling rule: recency / late interaction
  H9  zero-shot LLM judge re-ranking the top-k
  H10 learned candidate-aware user model on top of each representation (downstream consistency check)
  H11 multi-interest: zero-shot k-means interests (pooling stage) and a learned K-interest reader (user-model stage), K=1 as its control
  F   more of the data: category / entity names in the text an encoder reads, view fusion, a longer history (moredata stage)
  +   encoder sweep (TF-IDF/LSA, BGE small/base/large, Qwen3-Embedding)
"""
from __future__ import annotations

import argparse
import hashlib
import json
import os
import pickle
import time
import traceback

import numpy as np
import torch

import adapt
import common
import rep_eval as R
import semgraph as G
import stages_ext as X
import stages_ref as SR
from common import HistQueries, avg_results, fuse, log, query_prefix  # noqa: F401  (re-exported for callers)
from data import MindData

EQUIV_REFERENCE = None  # callable(texts)->np.ndarray of reference embeddings (e.g. Sentence-Transformers), set by the notebook

DEFAULT_STAGES = "controls,pooling,moredata,graph,head,histquery,sid,encoder,usermodel,sweep"
REFERENCE_FAMILIES = {"ref-baseline", "ref-ladder"}                             # trained baselines: comparators, not candidate directions
TECHNIQUE_FAMILIES_EXCLUDED = {"control", "H10-user-model", "enc-sweep"} | REFERENCE_FAMILIES      # not "techniques over the frozen embedding"


def _slim(r):
    """Keep what paired comparisons need, in compact dtypes (a full run caches ~100 variants)."""
    out = {"arr": {k: np.asarray(v, np.float32) for k, v in r["arr"].items()}, "ids": np.asarray(r["ids"], np.int64),
           "pos_news": np.asarray(r["pos_news"], np.int32), "pos_auc": np.asarray(r["pos_auc"], np.float32),
           "pos_row": np.asarray(r["pos_row"], np.int32), "mean": dict(r["mean"])}
    for k in ("cold", "cold_frac"):
        if k in r:
            out[k] = r[k]
    return out


class Suite:
    def __init__(self, data, meta, E0, device, work, seen_tr, seen_va, args=None):
        self.data, self.meta, self.E0, self.device, self.work = data, meta, E0, device, work
        self.seen_tr, self.seen_va, self.args = seen_tr, seen_va, args
        self.titles = [""] + meta["titles"]
        self.es = {"val": R.EvalSet(data.validation, device=device), "test": R.EvalSet(data.test, device=device)}
        self.records, self.cache = [], {}
        self.store, self.kv = {}, {}                      # in-memory artefacts / JSON-able facts that survive a resume
        self.cache_dir = (getattr(args, "cache_dir", None) or os.path.join(work, "cache"))
        self.stage, self._perm, self._sub = "", {}, {}

    # ---- plumbing
    def tensor(self, E):
        return torch.from_numpy(np.ascontiguousarray(E, dtype=np.float32)).to(self.device)

    def run(self, scorer, split, es=None):
        return R.evaluate(scorer, (es or self.es)[split], self.seen_tr)

    def val_ndcg(self, E):
        return self.run(R.mean_pool_scorer(self.tensor(E)), "val")["mean"]["ndcg@10"]

    def time_left(self):
        return common.time_left()

    def deadline(self, frac=1.0, cap_s=None):
        """Absolute time by which a long loop should stop: ``frac`` of the remaining budget (None = unlimited)."""
        left = common.time_left()
        if left == float("inf"):
            return None if cap_s is None else time.time() + cap_s
        t = left * frac
        return time.time() + max(min(t, cap_s) if cap_s is not None else t, 0.0)

    def perm(self, split):
        if split not in self._perm:
            self._perm[split] = np.random.default_rng(0).permutation(len(self.es[split].rows))
        return self._perm[split]

    def subset(self, split, n):
        """Nested random subset of a split -> ``(original row ids, EvalSet)``; ``n <= 0`` or ``>= N`` means the full split."""
        N = len(self.es[split].rows)
        if n is None or n <= 0 or n >= N:
            return np.arange(N), self.es[split]
        if (split, n) not in self._sub:
            ids = np.sort(self.perm(split)[:n])
            self._sub[(split, n)] = (ids, common.subset_evalset(self.data.validation if split == "val" else self.data.test, ids, self.device))
        return self._sub[(split, n)]

    def row(self, name):
        for r in reversed(self.records):
            if r["name"] == name:
                return r
        raise KeyError(name)

    # ---- shared heavy objects, built lazily so a resumed run can skip the stage that first needed them
    def memo(self):
        if "memo" not in self.store:
            A = G.entity_matrix(self.meta["ents"], self.data.n_news)
            mem = {"val": self.seen_tr, "test": self.seen_tr | self.seen_va}
            self.store["entity_A"] = A
            self.store["memo"] = {sp: G.Memory(self.E0, A, mem[sp], self.device) for sp in ("val", "test")}
        return self.store["memo"]

    def communities(self):
        if "comm" not in self.store:
            out = {}
            for sp, m in self.memo().items():
                t0 = time.time()
                out[sp] = m.communities(k=10, resolution=self.args.louvain_res, seed=0)
                log(f"  communities[{sp}]: {out[sp][1].shape[0]} via {out[sp][2]} ({time.time() - t0:.0f}s)")
            self.store["comm"] = out
        return self.store["comm"]

    def generator(self):
        import llm_gen
        if "gen" not in self.store:
            path = os.path.join(self.work, "gen_cache.jsonl")
            half = self.device == "cuda" and not self.args.gen_fp32
            g = llm_gen.Generator(self.args.gen_model, self.device, torch.float16 if half else torch.float32, path)
            if half and not g.self_test():                                    # fp16 overflow: fall back to fp32 (slower, the planner shrinks the subsets)
                log("  generator: fp16 output is broken (overflow?) -> reloading in fp32")
                g.free()
                g = llm_gen.Generator(self.args.gen_model, self.device, torch.float32, path + ".fp32")
            self.store["gen"] = g
        return self.store["gen"]

    def doc_encoder(self):
        return adapt.HFEncoder(self.args.encoder, max_len=self.args.max_len, device=self.device, prefix=self.args.doc_prefix)

    def query_encoder(self):
        return adapt.HFEncoder(self.args.encoder, max_len=self.args.q_max_len, device=self.device,
                               prefix=query_prefix(self.args.encoder, self.args.query_format, True))

    def close(self):
        g = self.store.pop("gen", None)
        if g is not None:
            g.free()

    # ---- results
    def record(self, name, family, cfg, val, test, ref_name, extra=None):
        """Store a row; Δ vs the reference is computed on the test split with paired CIs (common impressions only)."""
        ref = self.cache.get(ref_name)
        val, test = _slim(val), _slim(test)           # same float32 rounding as the cached reference: identical variants give exactly 0
        row = {"name": name, "family": family, "stage": self.stage, "cfg": cfg, "ref": ref_name,
               "n_val": int(len(val["ids"])), "n_test": int(len(test["ids"])),
               "val_ndcg@10": val["mean"]["ndcg@10"], "val_auc": val["mean"]["auc"], **test["mean"],
               "cold_frac": test.get("cold_frac"), **(extra or {})}
        if ref is not None and name != ref_name:
            rv, rt = ref["val"], ref["test"]
            row["val_delta"] = R.compare(val, rv, "ndcg@10")[0]
            row["d_ndcg@10"] = R.compare(test, rt, "ndcg@10")
            row["d_auc"] = R.compare(test, rt, "auc")
            row["d_cold_auc"] = R.compare_slice(test, rt, "cold")
            row["d_warm_auc"] = R.compare_slice(test, rt, "warm")
            row["verdict"] = R.verdict(row["d_ndcg@10"][1], row["d_ndcg@10"][2])
            _, ib = R._align(test, rt)
            row["ref_ndcg@10_same"] = float(np.mean(rt["arr"]["ndcg@10"][ib]))
            row["ref_auc_same"] = float(np.nanmean(rt["arr"]["auc"][ib]))
        self.records.append(row)
        self.cache[name] = {"val": val, "test": test}
        d = row.get("d_ndcg@10")
        log(f"  {name:34s} val nDCG@10={row['val_ndcg@10']:.4f} | test AUC={row['auc']:.4f} nDCG@10={row['ndcg@10']:.4f}"
            + (f" (n={row['n_test']:,})" if row["n_test"] != len(self.es["test"].rows) else "")
            + (f" | Δ={d[0]:+.4f} [{d[1]:+.4f},{d[2]:+.4f}] {row['verdict']}" if d else "")
            + (f" | cold AUC={row.get('cold_auc', float('nan')):.4f}" if "cold_auc" in row else ""))
        return row

    def pop_twin(self, name, family, cfg, make, es, extra=None, ref_pop="frozen+pop"):
        """Same scorer + V10's online popularity (equal-weight z-score sum, exactly V10's ``bge_zs_pop`` recipe), reference ``frozen+pop``
        (or the popularity twin of whichever row the variant is compared with, e.g. ``pool_lse+pop``)."""
        sc = lambda sp: common.fuse([(make(cfg, sp), 1.0), (common.pop_scorer, 1.0)])
        return self.record(f"{name}+pop", f"{family}+pop", {**cfg, "base": name}, self.run(sc("val"), "val", es=es),
                           self.run(sc("test"), "test", es=es), ref_pop, extra)

    def want_pop(self, ref_name):
        """A popularity twin is reported when the row it would be compared with has one (``frozen`` always does, ``pool_lse`` after the pooling stage)."""
        return bool(getattr(self.args, "with_pop", 0)) and f"{ref_name}+pop" in self.cache

    def select(self, name, family, configs, make, ref_name, extra=None, es=None, pop=None):
        """Generic grid: choose the config on validation, report it on test once (plus its popularity twin)."""
        best = None
        for cfg in configs:
            v = self.run(make(cfg, "val"), "val", es=es)
            if len(configs) > 1:
                log(f"    {name} {cfg}: val nDCG@10={v['mean']['ndcg@10']:.4f}")
            if best is None or v["mean"]["ndcg@10"] > best[0]["mean"]["ndcg@10"]:
                best = (v, cfg)
        v, cfg = best
        t = self.run(make(cfg, "test"), "test", es=es)
        row = self.record(name, family, cfg, v, t, ref_name, extra)
        if self.want_pop(ref_name) if pop is None else pop:
            self.pop_twin(name, family, cfg, make, es, extra, ref_pop=f"{ref_name}+pop" if f"{ref_name}+pop" in self.cache else "frozen+pop")
        return row

    def export(self, name, E, model_name):
        path = os.path.join(self.work, f"emb_{name}.npz")
        np.savez(path, ids=np.array(self.meta["ids"]), vecs=np.asarray(E[1:], np.float32), model_name=np.array(model_name))
        return path

    # ---- resume
    SIG_IGNORED = {"stages", "budget_hours", "min_stage_s", "no_resume", "work", "cache_dir", "device", "mind_train", "mind_dev"}

    def signature(self):
        """A saved state is reused only if the data AND every result-relevant argument are identical (so a QUICK run can never leak
        into a full run); the stage list, budget, device and paths do not matter."""
        a = {k: v for k, v in sorted(vars(self.args).items()) if k not in self.SIG_IGNORED and not k.startswith("ref_")}
        return {"args": hashlib.sha1(json.dumps(a, sort_keys=True, default=str).encode()).hexdigest(), "n_news": int(self.data.n_news),
                "n_val": len(self.data.validation), "n_test": len(self.data.test)}

    def save_state(self, done):
        path = os.path.join(self.work, "state.pkl")
        with open(path + ".tmp", "wb") as f:
            pickle.dump({"sig": self.signature(), "records": self.records, "cache": self.cache, "done": sorted(done), "kv": self.kv}, f, protocol=4)
        os.replace(path + ".tmp", path)

    def load_state(self):
        path = os.path.join(self.work, "state.pkl")
        if not os.path.exists(path):
            return set()
        try:
            with open(path, "rb") as f:
                st = pickle.load(f)
        except Exception as e:                                            # a truncated file must not block a fresh run
            log(f"state.pkl unreadable ({repr(e)[:80]}) - starting fresh")
            return set()
        if st["sig"] != self.signature():
            log("state.pkl belongs to a different configuration - starting fresh")
            return set()
        self.records, self.cache, self.kv = st["records"], st["cache"], st["kv"]
        return set(st["done"])


# ------------------------------------------------------------------- original stages
def stage_controls(S, args):
    log("\n== controls ==")
    E = S.tensor(S.E0)
    v, t = S.run(R.mean_pool_scorer(E), "val"), S.run(R.mean_pool_scorer(E), "test")
    S.record("frozen", "control", {}, v, t, "frozen")
    if args.repro:
        auc_ref, nd_ref = (float(x) for x in args.repro.split(","))
        ok = abs(t["mean"]["auc"] - auc_ref) < 0.003 and abs(t["mean"]["ndcg@10"] - nd_ref) < 0.003
        log(f"  reproduction check vs V10 bge_zeroshot (AUC {auc_ref}, nDCG@10 {nd_ref}): "
            f"{'MATCH' if ok else 'MISMATCH - check input text / truncation / embedding file'}")
    if t.get("cold_frac") is not None:
        log(f"  clicked articles that are cold (unseen in training rows): {t['cold_frac']:.1%}")
    mp = R.mean_pool_scorer(E)
    pv, pt = S.run(common.pop_scorer, "val"), S.run(common.pop_scorer, "test")
    rp = S.record("popularity", "control", {"note": "V10 online smoothed log-CTR, content-free"}, pv, pt, "frozen")
    fz = common.fuse([(mp, 1.0), (common.pop_scorer, 1.0)])
    rf = S.record("frozen+pop", "control", {"note": "V10 bge_zs_pop: z(BGE mean-pool) + z(popularity)"}, S.run(fz, "val"), S.run(fz, "test"), "frozen")
    if args.repro_pop:
        a1, n1, a2, n2 = (float(x) for x in args.repro_pop.split(","))
        ok = (abs(rp["auc"] - a1) < 0.003 and abs(rp["ndcg@10"] - n1) < 0.003 and abs(rf["auc"] - a2) < 0.003 and abs(rf["ndcg@10"] - n2) < 0.003)
        log(f"  reproduction check vs V10 popularity (AUC {a1}, nDCG@10 {n1}) and bge_zs_pop (AUC {a2}, nDCG@10 {n2}): "
            f"{'MATCH' if ok else 'MISMATCH - popularity plumbing differs from V10'}")
    Er = np.random.default_rng(0).standard_normal(S.E0.shape).astype(np.float32)
    Er[0] = 0
    Er[1:] /= np.linalg.norm(Er[1:], axis=1, keepdims=True)
    Et = S.tensor(Er)
    S.record("random_vec", "control", {"note": "random unit vectors: only 'already read' identity information"},
             S.run(R.mean_pool_scorer(Et), "val"), S.run(R.mean_pool_scorer(Et), "test"), "frozen")


def stage_graph(S, args):
    log("\n== H3/H4 retrieval-augmented representations (content-only, causal memory) ==")
    memo = S.memo()
    A = S.store["entity_A"]
    n_with = int((np.asarray(A.sum(1)).ravel() > 0).sum())
    log(f"  entity coverage: {n_with:,}/{S.data.n_news - 1:,} articles carry Wikidata entities")
    if n_with:
        S.select("entity_rag", "H3-RAG", [{"beta": b} for b in args.betas],
                 lambda c, sp: R.mean_pool_scorer(S.tensor(memo[sp].entity_view(c["beta"])[0])), "frozen")
    nn = {sp: memo[sp].neighbours(np.arange(1, S.data.n_news), max(args.knn_k)) for sp in ("val", "test")}

    def knn_view(c, sp):
        nbr, sim = nn[sp]
        k = c["k"]
        w = np.clip(sim[:, :k], 0, None)
        w = w / (w.sum(1, keepdims=True) + 1e-8)
        E = S.E0.copy()
        for s0 in range(0, len(nbr), 8192):                       # chunked: [rows,k,d] would be ~GBs at k=20
            sl = slice(s0, s0 + 8192)
            agg = (S.E0[nbr[sl, :k]] * w[sl, :, None]).sum(1)
            E[1 + s0:1 + s0 + len(agg)] = G._norm(S.E0[1 + s0:1 + s0 + len(agg)] + c["beta"] * G._norm(agg))
        return E
    S.select("knn_rag", "H3-RAG", [{"k": k, "beta": b} for k in args.knn_k for b in args.betas],
             lambda c, sp: R.mean_pool_scorer(S.tensor(knn_view(c, sp))), "frozen")
    comm = S.communities()
    S.select("community_graphrag", "H4-GraphRAG", [{"lam": l} for l in args.lams],
             lambda c, sp: G.community_scorer(S.E0, comm[sp][0], comm[sp][1], c["lam"], S.device), "frozen")
    um = G.UserMemory(S.data, S.E0, S.device)
    S.select("user_rag", "H4-GraphRAG", [{"gamma": g} for g in args.gammas],
             lambda c, sp: G.user_rag_scorer(S.E0, um, c["gamma"], S.device), "frozen")


def _seed_runs(S, name, family, cfg, train_fn, ref, seeds, trunc=None):
    """Train a stochastic adapter over seeds -> one averaged row; Matryoshka truncation reported."""
    vals, tests, Es, infos = [], [], [], []
    for sd in range(seeds):
        E, info = train_fn(sd)
        Et = S.tensor(E)
        vals.append(S.run(R.mean_pool_scorer(Et), "val"))
        tests.append(S.run(R.mean_pool_scorer(Et), "test"))
        Es.append(E); infos.append(info)
    extra = {"seeds": seeds, "best_epochs": [i["best_epoch"] for i in infos],
             "ndcg10_per_seed": [t["mean"]["ndcg@10"] for t in tests]}
    if seeds > 1:
        extra["ndcg10_std"] = float(np.std(extra["ndcg10_per_seed"]))
    for m in (trunc or []):
        Em = Es[0][:, :m].copy()
        Em = Em / (np.linalg.norm(Em, axis=1, keepdims=True) + 1e-8)
        extra[f"ndcg10@{m}d"] = S.run(R.mean_pool_scorer(S.tensor(Em)), "test")["mean"]["ndcg@10"]
    S.record(name, family, cfg, avg_results(vals), avg_results(tests), ref, extra)
    if S.want_pop("frozen"):
        pv, pt = [], []
        for E in Es:
            fz = common.fuse([(R.mean_pool_scorer(S.tensor(E)), 1.0), (common.pop_scorer, 1.0)])
            pv.append(S.run(fz, "val")), pt.append(S.run(fz, "test"))
        S.record(f"{name}+pop", f"{family}+pop", {**cfg, "base": name}, avg_results(pv), avg_results(pt), "frozen+pop", extra)
    return Es[0]


def stage_head(S, args):
    log("\n== H2 contrastive co-click adaptation: residual heads on frozen vectors ==")
    pairs = adapt.build_pairs(S.data, max_pairs=args.pairs)
    log(f"  {len(pairs[0]):,} (anchor, clicked, hard-negative) pairs from train_core")
    d = S.E0.shape[1]
    trunc = [m for m in (64, 128) if m < d]
    ref_t = S.cache["frozen"]["test"]
    for m in trunc:  # how good is plain truncation of the frozen vector? (the thing MRL should fix)
        Em = S.E0[:, :m] / (np.linalg.norm(S.E0[:, :m], axis=1, keepdims=True) + 1e-8)
        log(f"  frozen truncated to {m}d: test nDCG@10={S.run(R.mean_pool_scorer(S.tensor(Em)), 'test')['mean']['ndcg@10']:.4f}"
            f" (full {ref_t['mean']['ndcg@10']:.4f})")
    best = None
    for kind in ("lin", "mlp"):
        for tag, mrl, hn in (("", False, False), ("+hn", False, True), ("+hn+mrl", True, True)):
            name = f"head_{kind}{tag}"
            fn = lambda sd, kind=kind, mrl=mrl, hn=hn: adapt.train_head(
                S.E0, pairs, kind, mrl, hn, epochs=args.epochs_head, seed=sd, device=S.device,
                val_fn=S.val_ndcg, deadline=S.deadline(0.2), log=lambda *a: None)
            E = _seed_runs(S, name, "H2-adapt", {"kind": kind, "mrl": mrl, "hardneg": hn}, fn, "frozen", args.seeds, trunc)
            r = S.records[-1]
            if best is None or r["val_ndcg@10"] > best[0]:
                best = (r["val_ndcg@10"], name, E)
    S.kv["best_head"] = best[1]
    log(f"  best head on validation: {best[1]}  -> exported {S.export(best[1], best[2], args.encoder)}")


def stage_encoder(S, args):
    log("\n== H2 contrastive co-click adaptation: LoRA / DoRA with the encoder in the loop ==")
    texts = [""] + S.meta["text"]
    val_ids = np.unique(np.concatenate([np.asarray(r[2], np.int64).ravel() for r in S.data.validation]
                                        + [np.asarray(r[3], np.int64) for r in S.data.validation]))
    pairs = adapt.build_pairs(S.data, max_pairs=args.pairs_enc)
    log(f"  {len(pairs[0]):,} pairs, {len(val_ids):,} validation articles re-encoded per epoch")
    enc0 = adapt.HFEncoder(args.encoder, max_len=args.max_len, device=S.device, prefix=args.doc_prefix)
    Ef = np.zeros_like(S.E0)
    Ef[1:] = enc0.encode(S.meta["text"], bs=args.enc_bs)
    Et = S.tensor(Ef)
    S.record("frozen_hf", "control", {"max_len": args.max_len}, S.run(R.mean_pool_scorer(Et), "val"),
             S.run(R.mean_pool_scorer(Et), "test"), "frozen_hf")
    del enc0
    variants = (("lora", True), ("dora", True), ("lora", False))
    for i, (mode, mrl) in enumerate(variants):
        if S.time_left() < 1800:
            log(f"  {mode}: skipped (wall-clock budget)")
            continue
        name = f"{mode}{'+mrl' if mrl else ''}+hn"
        enc = adapt.HFEncoder(args.encoder, max_len=args.max_len, device=S.device, prefix=args.doc_prefix)
        t0 = time.time()
        E, info = adapt.train_encoder(enc, texts, pairs, Ef, mode=mode, mrl=mrl, hardneg=True,
                                      epochs=args.epochs_enc, bs=args.bs_enc, r=args.lora_r, val_ids=val_ids,
                                      val_fn=S.val_ndcg, max_steps=args.enc_steps, deadline=S.deadline(0.45 / (len(variants) - i)), log=log)
        Et = S.tensor(E)
        S.record(name, "H2-adapt", {"mode": mode, "mrl": mrl, "r": args.lora_r}, S.run(R.mean_pool_scorer(Et), "val"),
                 S.run(R.mean_pool_scorer(Et), "test"), "frozen_hf",
                 {"trainable": info["trainable"], "steps": info["steps"], "train_s": time.time() - t0,
                  "truncated": bool(info["steps"] < min(args.enc_steps, args.epochs_enc * (len(pairs[0]) // args.bs_enc)))})
        S.export(name, E, args.encoder)
        if S.want_pop("frozen"):
            fz = common.fuse([(R.mean_pool_scorer(S.tensor(E)), 1.0), (common.pop_scorer, 1.0)])
            S.record(f"{name}+pop", "H2-adapt+pop", {"mode": mode, "mrl": mrl, "r": args.lora_r, "base": name}, S.run(fz, "val"), S.run(fz, "test"), "frozen+pop")
        del enc
        if S.device == "cuda":
            torch.cuda.empty_cache()


def stage_histquery(S, args):
    log("\n== H1 history-as-text, instruction-aware query ==")
    K = args.hist_k
    hq = HistQueries(S.data, S.titles, K)
    n_val, n_test = (args.llm_val_n, args.llm_test_n) if args.hq_subset else (0, 0)
    sets = {sp: S.subset(sp, n) for sp, n in (("val", n_val), ("test", n_test))}
    es = {sp: sets[sp][1] for sp in sets}
    need = np.union1d(hq.needed("val", sets["val"][0]), hq.needed("test", sets["test"][0]))
    log(f"  {len(need):,} unique histories (last {K} headlines) to encode; evaluated on "
        f"{len(sets['val'][0]):,} validation / {len(sets['test'][0]):,} test impressions")
    Et = S.tensor(S.E0)
    mp = R.mean_pool_scorer(Et)
    for instruct in ((True, False) if args.hq_plain else (True,)):
        if S.time_left() < 900:
            log("  skipped (wall-clock budget)")
            break
        pre = query_prefix(args.encoder, args.query_format, instruct)
        enc = adapt.HFEncoder(args.encoder, max_len=args.q_max_len, device=S.device, prefix=pre)
        Q = hq.alloc(S.E0.shape[1])
        t0 = time.time()
        Q[need] = enc.encode(hq.texts(need), bs=args.enc_bs)
        log(f"  encoded queries ({'instruct' if instruct else 'plain'}) in {time.time() - t0:.0f}s, prefix={pre[:40]!r}")
        qs = lambda sp: hq.scorer(Q, Et, sp, S.device, ids=sets[sp][0])
        tag = "instr" if instruct else "plain"
        S.select(f"histq_{tag}", "H1-text-user", [{"w": 0.0}], lambda c, sp: qs(sp), "frozen", es=es)
        S.select(f"histq_{tag}+meanpool", "H1-text-user", [{"w": w} for w in (0.5, 1.0, 2.0)],
                 lambda c, sp: fuse([(qs(sp), 1.0), (mp, c["w"])]), "frozen", es=es)
        del enc
        if S.device == "cuda":
            torch.cuda.empty_cache()


STAGES = {"controls": stage_controls, "pooling": X.stage_pooling, "moredata": X.stage_moredata, "graph": stage_graph, "head": stage_head,
          "histquery": stage_histquery, "sid": X.stage_sid, "encoder": stage_encoder, "llm_user": X.stage_llm_user,
          "usermodel": X.stage_usermodel, "sweep": X.stage_sweep, "graphrag_llm": X.stage_graphrag_llm,
          "augment": X.stage_augment, "rerank": X.stage_rerank, **SR.STAGES}


# --------------------------------------------------------------------- output
def fmt_ci(t, digits=4):
    return f"{t[0]:+.{digits}f} [{t[1]:+.{digits}f}, {t[2]:+.{digits}f}]" if t else "-"


def write_table(S, work):
    cols = ["variant", "family", "n test", "val nDCG@10", "test AUC", "MRR", "nDCG@5", "nDCG@10", "ref", "ref nDCG@10",
            "Δ nDCG@10 [95% CI]", "cold AUC", "Δ cold AUC [95% CI]", "warm AUC", "verdict"]
    lines = ["| " + " | ".join(cols) + " |", "|" + "---|" * len(cols)]
    for r in S.records:
        ref_nd = r.get("ref_ndcg@10_same", r["ndcg@10"] if r["name"] == r["ref"] else float("nan"))
        lines.append("| " + " | ".join([
            r["name"], r["family"], f"{r['n_test']:,}", f"{r['val_ndcg@10']:.4f}", f"{r['auc']:.4f}", f"{r['mrr']:.4f}",
            f"{r['ndcg@5']:.4f}", f"{r['ndcg@10']:.4f}", r["ref"], f"{ref_nd:.4f}", fmt_ci(r.get("d_ndcg@10")),
            f"{r.get('cold_auc', float('nan')):.4f}", fmt_ci(r.get("d_cold_auc")),
            f"{r.get('warm_auc', float('nan')):.4f}", r.get("verdict", "control")]) + " |")
    md = "\n".join(lines)
    with open(os.path.join(work, "results.md"), "w", encoding="utf-8") as f:
        f.write("# V11 decision suite (MINDsmall_dev test; Δ and 'ref nDCG@10' are computed on the same impressions as the row)\n\n" + md + "\n")
    with open(os.path.join(work, "results.json"), "w", encoding="utf-8") as f:
        json.dump(S.records, f, indent=1, default=lambda o: o.tolist() if hasattr(o, "tolist") else str(o))
    return md


def _val_key(r):
    return r.get("val_delta", r["val_ndcg@10"])


def summarize(S, pop=False):
    """One representative per family, chosen on VALIDATION (never on test), then compared with its control.  The verdict is also
    Bonferroni-adjusted for the number of families, because picking 'the best of m directions' is a multiple-comparison decision.
    ``pop=False``: content-only variants vs ``frozen``;  ``pop=True``: the same variants fused with popularity vs ``frozen+pop`` (= V10 ``bge_zs_pop``)."""
    from statistics import NormalDist
    log("\n================ DECISION SUMMARY WITH POPULARITY (control = frozen+pop = V10 bge_zs_pop) ================" if pop
        else "\n================ DECISION SUMMARY (content only; control = frozen) ================")
    by_h = {}
    for r in S.records:
        if r["family"] != "control" and "verdict" in r and r["family"].endswith("+pop") == pop and r["family"].replace("+pop", "") not in REFERENCE_FAMILIES:
            by_h.setdefault(r["family"], []).append(r)
    if not by_h:
        return []
    z_adj = NormalDist().inv_cdf(1 - 0.05 / (2 * max(len(by_h), 1)))
    reps = []
    for fam, rs in by_h.items():
        best = max(rs, key=_val_key)                                # selection on validation only
        d, c = best["d_ndcg@10"], best.get("d_cold_auc")
        se = (d[2] - d[1]) / (2 * 1.96)
        adj = R.verdict(d[0] - z_adj * se, d[0] + z_adj * se)
        reps.append(best)
        kept = " [adapter = frozen: no epoch beat epoch 0 on validation]" if best.get("best_epochs") and not any(best["best_epochs"]) else ""
        log(f"{fam:18s} rep={best['name']:24s} (best of {len(rs):2d} by val, n={best['n_test']:,}, vs {best['ref']}) Δ nDCG@10 {fmt_ci(d)} -> {best['verdict']}"
            f" | Bonferroni(m={len(by_h)}): {adj}" + (f" | Δ cold AUC {fmt_ci(c)}" if c and not np.isnan(c[0]) else "") + kept)
    log("Rule: pursue a direction only if its Bonferroni-adjusted verdict is BETTER; prefer the ones that also win on the "
        "cold slice (where LLM embeddings should matter) and whose gain is practically meaningful (>~0.005 nDCG@10).\n"
        "Unadjusted 'BETTER' on one of many rows can be chance: with all effects null, about 1 in 20 rows would still pass.")
    return sorted(reps, key=_val_key, reverse=True)


def head_to_head(S, reps, top=5):
    """Direction-vs-direction comparison on the test split (paired over common impressions), Bonferroni over the pairs.
    Restricted to the ``top`` technique families ranked on VALIDATION (user-model / sweep rows have other controls)."""
    from itertools import combinations
    from statistics import NormalDist
    reps = [r for r in reps if r["family"] not in TECHNIQUE_FAMILIES_EXCLUDED][:top]
    pairs = list(combinations(reps, 2))
    out = []
    if not pairs:
        return out
    z_adj = NormalDist().inv_cdf(1 - 0.05 / (2 * len(pairs)))
    log(f"\n-- HEAD-TO-HEAD between the top-{len(reps)} directions by validation (test, paired on common impressions). "
        f"CI shown is unadjusted 95%; the verdict is Bonferroni-adjusted over {len(pairs)} pairs --")
    for a, b in pairs:
        ra, rb = S.cache[a["name"]]["test"], S.cache[b["name"]]["test"]
        d, c = R.compare(ra, rb, "ndcg@10"), R.compare_slice(ra, rb, "cold")
        se = (d[2] - d[1]) / (2 * 1.96)
        lo, hi = d[0] - z_adj * se, d[0] + z_adj * se
        word = f"{a['family']} > {b['family']}" if lo > 0 else (f"{b['family']} > {a['family']}" if hi < 0 else "not distinguishable")
        out.append({"a": a["name"], "b": b["name"], "d_ndcg@10": d, "d_cold_auc": c, "verdict": word})
        log(f"  {a['family']}:{a['name']} vs {b['family']}:{b['name']}  Δ nDCG@10 {fmt_ci(d)} | Δ cold AUC {fmt_ci(c)} -> {word}")
    return out


def save_arrays(S):
    """Per-impression metrics of every variant (test split) so any further paired analysis needs no re-run."""
    out = {"impr_ids": np.array([r[0] for r in S.data.test])}
    for name, c in S.cache.items():
        t = c["test"]
        for k in R.METRICS:
            out[f"{name}__{k}"] = t["arr"][k].astype(np.float32)
        out[f"{name}__pos_auc"] = t["pos_auc"].astype(np.float32)
        out[f"{name}__ids"] = t["ids"].astype(np.int32)                  # positions of the evaluated impressions (subsets differ)
    full = next((c["test"] for c in S.cache.values() if len(c["test"]["ids"]) == len(S.data.test)), None)
    if full is None:
        return None
    out["pos_news"], out["pos_row"], out["cold"] = full["pos_news"], full["pos_row"], full["cold"]
    path = os.path.join(S.work, "test_arrays.npz")
    np.savez_compressed(path, **out)
    return path


def cross_compare(suites, tags, pairs, path=None):
    """Direct paired comparison between named rows of (possibly different) runs on their common impressions:
    ``pairs = [((tag_a, name_a), (tag_b, name_b)), ...]`` -> Δ nDCG@10 and Δ cold AUC of A - B with 95% CIs."""
    by_tag = dict(zip(tags, suites))
    lines = ["| A | B | common impressions | Δ nDCG@10 (A-B) [95% CI] | Δ cold AUC (A-B) [95% CI] | verdict |", "|---|---|---|---|---|---|"]
    out = []
    for (ta, na), (tb, nb) in pairs:
        try:
            a, b = by_tag[ta].cache[na]["test"], by_tag[tb].cache[nb]["test"]
        except KeyError as e:
            log(f"  cross_compare: {ta}:{na} vs {tb}:{nb} skipped (missing {e})")
            continue
        d, c = R.compare(a, b, "ndcg@10"), R.compare_slice(a, b, "cold")
        n = int(len(np.intersect1d(a["ids"], b["ids"])))
        out.append({"a": f"{ta}:{na}", "b": f"{tb}:{nb}", "n": n, "d_ndcg@10": d, "d_cold_auc": c})
        lines.append(f"| {ta}:{na} | {tb}:{nb} | {n:,} | {fmt_ci(d)} | {fmt_ci(c)} | {R.verdict(d[1], d[2])} |")
    md = "\n".join(lines)
    log("\n================ DIRECT COMPARISONS ACROSS RUNS (paired, common impressions) ================\n" + md)
    if path:
        with open(path, "w", encoding="utf-8") as f:
            f.write("# Direct paired comparisons between rows (A - B)\n\n" + md + "\n")
    return out


def plot_forest(rows, path, top=32):
    """Forest plot of the paired Δ nDCG@10 (95% CI) of the best rows vs the reference; returns the path (None if matplotlib is missing)."""
    try:
        import matplotlib
        matplotlib.use("Agg")
        import matplotlib.pyplot as plt
    except Exception:
        return None
    rows = rows[:top]
    fig, ax = plt.subplots(figsize=(9, 0.3 * len(rows) + 1.6))
    for i, x in enumerate(rows):
        d = x["d"]
        col = "#1f77b4" if d[1] > 0 else ("#c0392b" if d[2] < 0 else "#7f8c8d")
        ax.plot([d[1], d[2]], [-i, -i], lw=2.2, color=col)
        ax.plot([d[0]], [-i], "o", ms=4, color=col)
    ax.axvline(0, color="k", lw=0.8)
    ax.set_yticks([-i for i in range(len(rows))])
    ax.set_yticklabels([f"{x['run']}:{x['name']}  [{x['family']}]" + (f" n={x['n']:,}" if x["n"] < 70000 else "") for x in rows], fontsize=7)
    ax.set_xlabel("Δ nDCG@10 vs reference (paired 95% CI)")
    ax.set_title("blue: CI above 0   red: CI below 0   grey: not distinguishable", fontsize=8)
    fig.tight_layout()
    fig.savefig(path, dpi=150)
    plt.close(fig)
    return path


def leaderboard(suites, tags, ref, path=None, top=None, plot=None, pop=False):
    """One table over several runs (e.g. BGE-small and Qwen3-Embedding), each row compared on the common impressions with a single
    global reference ``ref = (tag, row name)`` -- by default the BGE-small frozen embedding (the V10-comparable control)."""
    by_tag = dict(zip(tags, suites))
    ref_res = by_tag[ref[0]].cache[ref[1]]["test"]
    rows = []
    for tag, S in by_tag.items():
        for r in S.records:
            if r["family"] == "H10-user-model":
                continue                                              # learned user models have their own control (um_frozen)
            keep_controls = ("popularity", "frozen+pop") if pop else ("frozen",)
            if r["family"] == "control":
                if r["name"] not in keep_controls:
                    continue
            elif r["family"].endswith("+pop") != pop:
                continue                                              # content-only and popularity-fused variants are ranked separately
            t = S.cache[r["name"]]["test"]
            d = R.compare(t, ref_res, "ndcg@10")
            if np.isnan(d[0]):
                continue
            rows.append({"run": tag, "name": r["name"], "family": r["family"], "n": r["n_test"], "d": d,
                         "dc": R.compare_slice(t, ref_res, "cold"), "val": _val_key(r), "auc": r["auc"], "ndcg": r["ndcg@10"]})
    rows.sort(key=lambda x: -x["d"][0])
    rows = rows[:top] if top else rows
    lines = [f"| rank | run | variant | family | n | Δ nDCG@10 vs {ref[0]}:{ref[1]} [95% CI] | Δ cold AUC [95% CI] | verdict |", "|---|---|---|---|---|---|---|---|"]
    for i, x in enumerate(rows, 1):
        lines.append(f"| {i} | {x['run']} | {x['name']} | {x['family']} | {x['n']:,} | {fmt_ci(x['d'])} | {fmt_ci(x['dc'])} | {R.verdict(x['d'][1], x['d'][2])} |")
    md = "\n".join(lines)
    if path:
        with open(path, "w", encoding="utf-8") as f:
            f.write(f"# Leaderboard over runs {tags}: zero-shot representation / re-ranking variants and trained reference baselines vs one global reference, paired on the common "
                    f"impressions (rows evaluated on subsets use the same impressions in the reference). Learned user-model rows (H10) are excluded: see each run's results.md (control um_frozen).\n"
                    + ("NOTE: '+pop' rows add V10's online CTR, counted from earlier impressions *of the test day itself* (streaming feedback): not comparable with offline papers.\n" if pop else "")
                    + f"\n{md}\n")
    log("\n================ LEADERBOARD (all runs, one global reference) ================\n" + md)
    if plot:
        log(f"forest plot -> {plot_forest(rows, plot)}")
    return rows


# ------------------------------------------------------------------------ main
def build_parser():
    ap = argparse.ArgumentParser()
    ap.add_argument("--mind-train", required=True)
    ap.add_argument("--mind-dev", required=True)
    ap.add_argument("--work", default="v11_out")
    ap.add_argument("--cache-dir", default=None, help="where frozen article embeddings are cached (share it between runs)")
    ap.add_argument("--encoder", default="BAAI/bge-small-en-v1.5")
    ap.add_argument("--news-emb", default=None, help="npz from V10 llm_embed.py (exact V10 control)")
    ap.add_argument("--doc-prefix", default="")
    ap.add_argument("--query-format", default="auto")
    ap.add_argument("--max-len", type=int, default=256)
    ap.add_argument("--q-max-len", type=int, default=384)
    ap.add_argument("--stages", default=DEFAULT_STAGES)
    ap.add_argument("--seeds", type=int, default=3)
    ap.add_argument("--device", default="auto")
    ap.add_argument("--repro", default=None, help='published V10 control "auc,ndcg10" (bge_zeroshot: 0.6241,0.3909)')
    ap.add_argument("--budget-hours", type=float, default=0.0, help="shared wall-clock budget; long loops stop gracefully (0 = unlimited)")
    ap.add_argument("--min-stage-s", type=float, default=300.0, help="do not start a stage with less than this left")
    ap.add_argument("--no-resume", action="store_true")
    ap.add_argument("--with-pop", type=int, default=1, help="also report every zero-shot variant fused with V10's online popularity (rows '+pop')")
    ap.add_argument("--repro-pop", default=None, help='V10 "popularity_auc,popularity_ndcg10,bge_zs_pop_auc,bge_zs_pop_ndcg10" (0.6533,0.4026,0.6769,0.4330)')
    ap.add_argument("--pairs", type=int, default=200_000)
    ap.add_argument("--pairs-enc", type=int, default=100_000)
    ap.add_argument("--epochs-head", type=int, default=3)
    ap.add_argument("--epochs-enc", type=int, default=1)
    ap.add_argument("--bs-enc", type=int, default=32)
    ap.add_argument("--enc-steps", type=int, default=1500)
    ap.add_argument("--enc-bs", type=int, default=128)
    ap.add_argument("--lora-r", type=int, default=16)
    ap.add_argument("--hist-k", type=int, default=10)
    ap.add_argument("--hq-plain", type=int, default=1, help="also run the history query without the instruction")
    ap.add_argument("--hq-subset", type=int, default=0, help="evaluate H1 on the same nested subsets as H5")
    ap.add_argument("--louvain-res", type=float, default=1.0)
    ap.add_argument("--betas", type=float, nargs="+", default=[0.25, 0.5, 1.0])
    ap.add_argument("--knn-k", type=int, nargs="+", default=[5, 20])
    ap.add_argument("--lams", type=float, nargs="+", default=[0.25, 0.5, 1.0, 2.0])
    ap.add_argument("--gammas", type=float, nargs="+", default=[0.25, 0.5, 1.0])
    ap.add_argument("--pool-lams", type=float, nargs="+", default=[0.02, 0.05, 0.1, 0.2])
    ap.add_argument("--pool-taus", type=float, nargs="+", default=[0.02, 0.03, 0.05, 0.07, 0.1, 0.15, 0.2, 0.3, 0.5])
    ap.add_argument("--pool-ks", type=int, nargs="+", default=[2, 3, 4, 5, 7, 10])
    ap.add_argument("--km-ks", type=int, nargs="*", default=[2, 3, 5], help="zero-shot multi-interest: number of k-means interests per reader (rows pool_km<K>)")
    # more of the data
    ap.add_argument("--text-modes", default="t,cta,ctae,views", help="article text variants to embed besides the suite's own title+abstract: t, cta, ctae, views")
    ap.add_argument("--hist-lens", type=int, nargs="*", default=[100, 200], help="history lengths (clicks) for the zero-shot rows; the control reads 50")
    # semantic ids
    ap.add_argument("--sid-k", type=int, default=256)
    ap.add_argument("--sid-levels", type=int, default=3)
    ap.add_argument("--sid-epochs", type=int, default=3)
    ap.add_argument("--sid-bs", type=int, default=128)
    ap.add_argument("--sid-lr", type=float, default=1e-3)
    ap.add_argument("--sid-nmax", type=int, default=30)
    ap.add_argument("--sid-d", type=int, default=256)
    ap.add_argument("--sid-layers", type=int, default=4)
    ap.add_argument("--sid-heads", type=int, default=4)
    ap.add_argument("--sid-steps", type=int, default=0)
    ap.add_argument("--sid-val-n", type=int, default=3000)
    ap.add_argument("--sid-ws", type=float, nargs="+", default=[0.25, 0.5, 1.0, 2.0])
    # encoder sweep
    ap.add_argument("--sweep-encoders", default="BAAI/bge-base-en-v1.5,BAAI/bge-large-en-v1.5,BAAI/bge-m3,thenlper/gte-base,"
                                                 "sentence-transformers/all-mpnet-base-v2,sentence-transformers/all-MiniLM-L6-v2,Qwen/Qwen3-Embedding-0.6B")
    ap.add_argument("--lsa-dim", type=int, default=256)
    # LLM user tower
    ap.add_argument("--llm-user-modes", default="causal,bidir,soft,causal@1")
    ap.add_argument("--llm-user-steps", type=int, default=800)
    ap.add_argument("--llm-user-bs", type=int, default=16)
    ap.add_argument("--llm-user-lr", type=float, default=1e-4)
    ap.add_argument("--llm-user-r", type=int, default=16)
    ap.add_argument("--llm-user-shared", type=int, default=1, help="share all positives and impression negatives of the batch as negatives")
    ap.add_argument("--llm-val-n", type=int, default=5000)
    ap.add_argument("--llm-test-n", type=int, default=30000)
    ap.add_argument("--llm-enc-bs", type=int, default=64)
    ap.add_argument("--ut-ws", type=float, nargs="+", default=[0.5, 1.0, 2.0])
    # learned user model
    ap.add_argument("--um-views", default="frozen,head,encoder,entity_rag,knn_rag,text")
    ap.add_argument("--um-seeds", type=int, default=2)
    ap.add_argument("--um-seeds-focus", type=int, default=3, help="seeds for the views the new comparisons are made on ('frozen' and 'text'); trained rows differ by 0.007-0.012 between seeds")
    ap.add_argument("--um-epochs", type=int, default=12)
    ap.add_argument("--um-repro", default=None, help='V10 llmenc_ca "auc,ndcg10" (0.6494,0.3997)')
    ap.add_argument("--um-extra", default="ff:frozen,ff:knn_rag,mi1:frozen,mi4:frozen,mi4d:frozen,mi4:text",
                    help="other readers on frozen vectors, 'reader:view' (reader: add|nrms|ff|mi<K>, K learned interests); '' = none")
    # reference baselines (NRMS / NAML / Fastformer with the published recipe) and the ablation ladder
    ap.add_argument("--ref-glove", type=int, default=1, help="initialise word vectors from GloVe-300 (downloads it; falls back to random and says so)")
    ap.add_argument("--ref-glove-path", default="", help="GloVe txt/zip to use instead of downloading")
    ap.add_argument("--ref-dims", default="300,16,16,400,1", help="emb,heads,head_dim,naml_filters,fastformer_layers of the published-recipe models")
    ap.add_argument("--ref-title-len", type=int, default=30)
    ap.add_argument("--ref-body-len", type=int, default=50)
    ap.add_argument("--ref-train-frac", type=float, default=1.0, help="fraction of training impressions (QUICK only)")
    ap.add_argument("--ref-max-epochs", type=int, default=0, help="cap on epochs (0 = the recipe's own: 10, V10 recipe 12)")
    ap.add_argument("--ref-bs", type=int, default=0, help="batch size (0 = the recipe's own: 64)")
    ap.add_argument("--ref-seeds", type=int, default=1)
    ap.add_argument("--ref-model-minutes", type=float, default=35.0, help="wall-clock cap per reference model")
    ap.add_argument("--ref-min-minutes", type=float, default=10.0, help="do not start a reference model with less than this left")
    ap.add_argument("--ref-repro", default=None, help='V10 NRMS "auc,ndcg10" for the harness check of the V10-recipe rung (0.6181,0.3685)')
    # generation-based stages
    ap.add_argument("--gen-model", default="Qwen/Qwen3-1.7B")
    ap.add_argument("--gen-fp32", type=int, default=0, help="load the generator in fp32 (use if fp16 produces garbage)")
    ap.add_argument("--gen-bs", type=int, default=48)
    ap.add_argument("--gen-new-tokens", type=int, default=56)
    ap.add_argument("--gen-hist-k", type=int, default=5)
    ap.add_argument("--gen-val-n", type=int, default=800)
    ap.add_argument("--gen-test-n", type=int, default=2500)
    ap.add_argument("--augment-minutes", type=float, default=35.0)
    ap.add_argument("--kar-alphas", type=float, nargs="+", default=[0.25, 0.5, 1.0])
    ap.add_argument("--kar-ws", type=float, nargs="+", default=[0.5, 1.0, 2.0])
    ap.add_argument("--gr-max-comm", type=int, default=300)
    ap.add_argument("--gr-titles", type=int, default=8)
    ap.add_argument("--rerank-k", type=int, default=10)
    ap.add_argument("--rerank-hist", type=int, default=8)
    ap.add_argument("--rerank-val-n", type=int, default=600)
    ap.add_argument("--rerank-test-n", type=int, default=2000)
    ap.add_argument("--rerank-minutes", type=float, default=35.0)
    ap.add_argument("--rerank-ws", type=float, nargs="+", default=[0.5, 1.0, 2.0])
    return ap


def guard_embeddings(vecs, meta, reference, tol=0.99):
    """Catch a wrong pooling / EOS / prefix convention up front by comparing 64 vectors with a reference implementation."""
    ref = reference(meta["text"][:64])
    if ref is None:
        log("E0 guard: reference encoder unavailable - pooling/EOS/prefix NOT cross-checked")
        return
    ref = np.asarray(ref, np.float32)
    cos = (vecs[:64] * ref).sum(1) / (np.linalg.norm(vecs[:64], axis=1) * np.linalg.norm(ref, axis=1) + 1e-8)
    log(f"E0 guard: HFEncoder vs reference embeddings, min cos={cos.min():.4f} mean={cos.mean():.4f}")
    if cos.min() < tol:
        raise RuntimeError("HFEncoder disagrees with the reference encoder (pooling/EOS/prefix) - fix before running")


def main(argv=None):
    args = build_parser().parse_args(argv)
    common.CLOCK.start(args.budget_hours)
    os.makedirs(args.work, exist_ok=True)
    device = "cuda" if args.device == "auto" and torch.cuda.is_available() else ("cpu" if args.device == "auto" else args.device)
    log(f"device={device} encoder={args.encoder} stages={args.stages} budget={'unlimited' if not args.budget_hours else f'{args.budget_hours:.1f}h'}")
    meta = G.load_news_meta(args.mind_train, args.mind_dev)
    cache_dir = args.cache_dir or os.path.join(args.work, "cache")
    if args.news_emb:
        z = np.load(args.news_emb, allow_pickle=True)
        emb = {str(i): v for i, v in zip(z["ids"], z["vecs"])}
        log(f"E0 <- {args.news_emb} ({len(emb):,} vectors)")
    else:
        vecs, secs = X.embed_news(meta, args.encoder, args.max_len, args.doc_prefix, device, args.enc_bs, cache_dir)
        if EQUIV_REFERENCE is not None:
            guard_embeddings(vecs, meta, EQUIV_REFERENCE)
        log(f"E0 {'loaded from cache' if not secs else f'encoded with {args.encoder} in {secs:.0f}s'} ({vecs.shape[0]:,} x {vecs.shape[1]})")
        emb = dict(zip(meta["ids"], vecs))
    data = MindData.from_mind(args.mind_train, args.mind_dev, llm_embeddings=emb)
    assert data.n_news - 1 == len(meta["ids"]), "article order mismatch between loader and metadata"
    seen_tr, seen_va = G.observed_masks(data)
    S = Suite(data, meta, data.llm_emb, device, args.work, seen_tr, seen_va, args)
    done = set() if args.no_resume else S.load_state()
    ref_sig = hashlib.sha1(json.dumps({k: v for k, v in sorted(vars(args).items()) if k.startswith("ref_")}, default=str).encode()).hexdigest()
    stale = {s_ for s_ in done if s_.startswith("ref_")} if S.kv.get("ref_sig", ref_sig) != ref_sig else set()
    if stale:                                                                          # reference-model settings changed: redo only those stages
        log(f"reference-baseline settings changed since the saved state -> re-running {sorted(stale)}")
        done -= stale
        S.records = [r for r in S.records if r.get("stage") not in stale]
        S.cache = {k: v for k, v in S.cache.items() if k in {r["name"] for r in S.records}}
    S.kv["ref_sig"] = ref_sig
    if done:
        log(f"resuming: stages already finished -> {sorted(done)} ({len(S.records)} rows restored)")
    wanted = [s for s in args.stages.split(",") if s]
    if "controls" not in wanted:
        wanted.insert(0, "controls")
    unknown = [s for s in wanted if s not in STAGES]
    if unknown:
        raise SystemExit(f"unknown stages {unknown}; choose from {sorted(STAGES)}")
    for s in wanted:
        if s in done:
            log(f"-- stage '{s}' already finished (resume), skipped")
            continue
        if s != "controls" and common.out_of_time(args.min_stage_s):               # controls are the reference: always run
            log(f"-- stage '{s}' SKIPPED: wall-clock budget exhausted ({common.CLOCK.elapsed() / 3600:.1f}h used)")
            continue
        S.records = [r for r in S.records if r.get("stage") != s]                  # a retried stage must not duplicate its rows
        t_stage = time.time()
        S.stage = s
        try:
            if STAGES[s](S, args) is False:                                        # the stage declined to run (budget, missing prerequisite): retry on resume
                log(f"-- stage '{s}' did not run (reason above); not marked finished")
            else:
                done.add(s)
                S.kv.setdefault("timings", {})[s] = round(time.time() - t_stage)
                log(f"-- stage '{s}' finished in {time.time() - t_stage:.0f}s (total {common.CLOCK.elapsed() / 3600:.2f}h)")
        except Exception:  # one failing stage must not discard the others
            log(f"\n!! stage '{s}' failed:\n{traceback.format_exc()}")
        if device == "cuda":
            torch.cuda.empty_cache()
        write_table(S, args.work)
        S.save_state(done)
    S.close()
    tm = S.kv.get("timings", {})
    if tm:
        log("STAGE TIMINGS (s): " + ", ".join(f"{k}={v}" for k, v in tm.items()) + f" | sum={sum(tm.values())}s")
        with open(os.path.join(args.work, "timings.json"), "w", encoding="utf-8") as f:
            json.dump(tm, f, indent=1)
    md = write_table(S, args.work)
    log("\n" + md)
    S.reps = summarize(S)
    S.h2h = head_to_head(S, S.reps)
    S.reps_pop = summarize(S, pop=True)
    S.h2h_pop = head_to_head(S, S.reps_pop)
    for fn, obj in (("head_to_head.json", S.h2h), ("head_to_head_pop.json", S.h2h_pop)):
        with open(os.path.join(args.work, fn), "w", encoding="utf-8") as f:
            json.dump(obj, f, indent=1, default=lambda o: o.tolist() if hasattr(o, "tolist") else str(o))
    try:
        SR.ladder_table(S, args.ref_repro, os.path.join(args.work, "ladder.md"))
    except Exception:
        log(f"ladder table failed:\n{traceback.format_exc()}")
    log(f"per-impression arrays -> {save_arrays(S)}")
    return S


if __name__ == "__main__":
    main()


### 2. Cấu hình

In [ ]:
import os, glob, sys, gc, time
QUICK = bool(int(os.environ.get("V11_QUICK", "0")))              # True: bản rút gọn để bắt lỗi môi trường trên dữ liệu thật
BUDGET_H = float(os.environ.get("V11_BUDGET_H", "9.5"))          # ngân sách đồng hồ chung cho cả notebook (giờ)
ENCODER_A = os.environ.get("V11_ENCODER", "BAAI/bge-small-en-v1.5")        # run A: encoder nhỏ, đúng control của V10
ENCODER_B = os.environ.get("V11_ENCODER_B", "Qwen/Qwen3-Embedding-0.6B")   # run B: LLM embedding (decoder-only)
GEN_MODEL = os.environ.get("V11_GEN_MODEL", "Qwen/Qwen3-1.7B")             # LLM sinh văn bản / giám khảo
FOCUS = bool(int(os.environ.get("V11_FOCUS", "0")))              # 1: chỉ phần mới (moredata, multi-interest, Fastformer): bỏ run B và các stage cũ, ước ~2-2,5 giờ
RUN_B = bool(int(os.environ.get("V11_RUN_B", "0" if FOCUS else "1")))
RUN_REF = bool(int(os.environ.get("V11_REF", "1")))              # baseline Fastformer chuẩn (2 seed + refit, ước ~85 phút)
REF_ALL = bool(int(os.environ.get("V11_REF_ALL", "0")))          # 1: chạy thêm NRMS / NAML / refit NRMS / thang ablation / nrms_ref@1 như v7 (+~1,4 giờ)
GLOVE_PATH = os.environ.get("V11_GLOVE_PATH", "")                # file GloVe-300 (txt/zip) nếu đã Add Input; để trống -> tự tìm trong /kaggle/input rồi tải từ Hugging Face
USE_REFERENCE = bool(int(os.environ.get("V11_REFERENCE", "1")))   # so HFEncoder với Sentence-Transformers trước khi chạy (đặt 0 để bỏ qua)
SWEEP = os.environ.get("V11_SWEEP", "BAAI/bge-base-en-v1.5" if QUICK else "BAAI/bge-base-en-v1.5,BAAI/bge-large-en-v1.5,Qwen/Qwen3-Embedding-0.6B")
MIND_TRAIN = os.environ.get("V11_MIND_TRAIN")                    # để None -> tự dò trong /kaggle/input
MIND_DEV = os.environ.get("V11_MIND_DEV")
DEVICE = os.environ.get("V11_DEVICE", "auto")
WORK = ("/kaggle/working/v11_out" if os.path.isdir("/kaggle/working") else "v11_out") + ("_quick" if QUICK else "")   # QUICK never mixes with a full run
CACHE = "/kaggle/temp/v11_cache" if os.path.isdir("/kaggle/temp") else os.path.join(WORK, "cache")   # embedding bài báo: dùng chung giữa các run
sys.path.insert(0, os.getcwd())
print(dict(QUICK=QUICK, FOCUS=FOCUS, BUDGET_H=BUDGET_H, ENCODER_A=ENCODER_A, ENCODER_B=ENCODER_B, GEN_MODEL=GEN_MODEL, RUN_B=RUN_B, RUN_REF=RUN_REF, REF_ALL=REF_ALL, WORK=WORK))


### 3. Tìm MIND (cần cặp train + dev của CÙNG một dataset)

In [ ]:
if not MIND_TRAIN:
    hits = sorted({os.path.dirname(p) for p in glob.glob("/kaggle/input/**/news.tsv", recursive=True)})
    trs = [d for d in hits if "train" in d.lower()]
    dvs = [d for d in hits if "dev" in d.lower() or "valid" in d.lower()]
    for t in trs:
        sib = [d for d in dvs if os.path.dirname(d) == os.path.dirname(t)]
        if sib:
            MIND_TRAIN, MIND_DEV = t, sib[0]
            break
assert MIND_TRAIN and MIND_DEV and MIND_TRAIN != MIND_DEV, (
    "Hãy Add Input một dataset MIND có CẢ MINDsmall_train và MINDsmall_dev (ví dụ thinhhuynh3108/mindsmall).")
count = lambda p: sum(1 for _ in open(p, encoding="utf-8"))
print("train:", MIND_TRAIN, "| behaviors:", f"{count(MIND_TRAIN + '/behaviors.tsv'):,}")
print("dev  :", MIND_DEV, "| behaviors:", f"{count(MIND_DEV + '/behaviors.tsv'):,}  (MIND-small: ~157k / ~73k)")


### 4. Cấu hình các run

* **Run A** = BGE-small (đúng control V10): H1–H4, H7, H8, H11 (k-means interest), F (`moredata`), H2 (LoRA/DoRA), H10 (+ reader multi-interest), quét encoder. Chạy làm **nhiều lần gọi** vào cùng thư mục, mỗi lần **resume** từ `state.pkl`:
  *A-core* (rẻ, quan trọng, có cả phần mới) → **baseline (R)** → *run B* → *A-phần còn lại* (LoRA/DoRA, user model, sweep). Thứ tự chỉ để ưu tiên ngân sách thời gian.
* **Run R** = Fastformer chuẩn (2 seed) + refit, cùng thư mục `bge` để các hàng nằm chung bảng với control (cùng impression ⇒ CI ghép cặp). `V11_REF_ALL=1` thêm NRMS / NAML / thang ablation.
* **Run B** = Qwen3-Embedding-0.6B + LLM sinh văn bản: stage rẻ và phần mới chạy trước (pooling, moredata với `t`,`cta`, user model), rồi H5 (tower), H9 judge, H6 augment, H4b — các stage đắt nhất để sau.


In [ ]:
import torch
import suite, common
common.CLOCK.reset()                                   # chạy lại cell trong cùng kernel: đồng hồ ngân sách bắt đầu lại

def reference_encoder(name, max_len):
    """Tham chiếu Sentence-Transformers (chỉ 64 bài) để bắt lỗi pooling/EOS/prefix. Trả None nếu không dùng được."""
    if not USE_REFERENCE:
        return lambda texts: None
    def ref(texts):
        try:
            from sentence_transformers import SentenceTransformer
            st = SentenceTransformer(name, device="cuda" if torch.cuda.is_available() else "cpu")
            st.max_seq_length = max_len
            out = st.encode(list(texts), normalize_embeddings=True)
            del st
            gc.collect(); torch.cuda.empty_cache()
            return out
        except Exception as e:
            print("reference encoder unavailable, skipping the pooling guard:", repr(e)[:120])
            return None
    return ref

COMMON = ["--mind-train", MIND_TRAIN, "--mind-dev", MIND_DEV, "--cache-dir", CACHE, "--device", DEVICE, "--budget-hours", str(BUDGET_H)]
repro = ["--repro", "0.6241,0.3909", "--um-repro", "0.6494,0.3997", "--repro-pop", "0.6533,0.4026,0.6769,0.4330"] if "bge-small-en-v1.5" in ENCODER_A else []
# Fastformer chuẩn: 2 seed (trung bình) để baseline có phương sai thấp hơn một lần chạy; 70 phút = trần 35 phút mỗi seed (v7 đo: 29 phút/seed trên T4)
REF_ARGS = ["--ref-glove-path", GLOVE_PATH, "--ref-seeds", "2", "--ref-model-minutes", "70"]
if REF_ALL:
    REF_ARGS += ["--ref-repro", "0.6181,0.3685"]       # NRMS của V10 (kaggle_run.ipynb): harness check của rung đầu thang ablation
if QUICK:
    REF_ARGS += ["--ref-train-frac", "0.1", "--ref-max-epochs", "2", "--ref-seeds", "1"]

A_BASE = ["--encoder", ENCODER_A, "--work", WORK + "/bge", "--max-len", "256", "--q-max-len", "384", "--hist-k", "10",
          "--sweep-encoders", SWEEP, "--um-seeds-focus", "3", *repro, *REF_ARGS]
B_BASE = ["--encoder", ENCODER_B, "--work", WORK + "/qwen3", "--max-len", "256", "--q-max-len", "384", "--hist-k", "10", "--seeds", "1",
          "--hq-plain", "0", "--hq-subset", "1", "--gen-model", GEN_MODEL, "--um-views", "frozen,head,entity_rag,text", "--um-seeds-focus", "2",
          "--um-extra", "ff:frozen,mi1:frozen,mi4:frozen,mi4d:frozen", "--text-modes", "t,cta", "--hist-lens", "100"]
A_QUICK = ["--seeds", "1", "--pairs", "20000", "--pairs-enc", "3000", "--enc-steps", "60", "--epochs-head", "1", "--betas", "0.5",
           "--knn-k", "10", "--lams", "1.0", "--gammas", "0.5", "--bs-enc", "16", "--hq-plain", "0", "--sid-epochs", "1", "--sid-steps", "300",
           "--sid-val-n", "500", "--um-epochs", "2", "--um-seeds", "1", "--um-seeds-focus", "1", "--um-views", "frozen,head,text",
           "--um-extra", "ff:frozen,mi1:frozen,mi4:frozen,mi4d:frozen,mi4:text", "--text-modes", "t,cta,views", "--hist-lens", "100", "--km-ks", "2",
           "--llm-user-steps", "60", "--llm-user-bs", "16", "--llm-val-n", "400", "--llm-test-n", "1000"]
B_QUICK = ["--seeds", "1", "--pairs", "20000", "--epochs-head", "1", "--betas", "0.5", "--knn-k", "10", "--lams", "1.0", "--gammas", "0.5",
           "--llm-user-modes", "causal,soft", "--llm-user-steps", "60", "--llm-user-bs", "8", "--llm-val-n", "400", "--llm-test-n", "1000",
           "--gen-val-n", "60", "--gen-test-n", "150", "--augment-minutes", "4", "--rerank-val-n", "40", "--rerank-test-n", "100",
           "--rerank-minutes", "4", "--gr-max-comm", "30", "--um-epochs", "2", "--um-seeds", "1", "--um-seeds-focus", "1", "--um-views", "frozen,text",
           "--um-extra", "ff:frozen,mi4:frozen", "--text-modes", "t", "--km-ks", "2"]
A1 = "controls,pooling,moredata,graph,head,histquery,sid"
A2 = A1 + ",encoder,llm_user,usermodel,sweep"
REF_STAGES = "ref_ff,ref_ff_refit" + (",ref_nrms,ref_naml,ref_nrms_refit,ref_l0,ref_l1,ref_l2,ref_nrms_s1" if REF_ALL else "")      # priority order: the headline model, its refit, then the optional v7 baselines
B_STAGES = "controls,pooling,moredata,graph,head,usermodel,histquery,llm_user,rerank,augment,graphrag_llm"                            # cheap and new first, the 2,5-hour tower and the generation stages after
if QUICK:                                              # smoke test: every stage family runs once, at toy sizes (Louvain only inside graphrag_llm)
    A1 = "controls,pooling,moredata,head,histquery,sid"
    A2 = A1 + ",encoder,llm_user,usermodel"
    REF_STAGES = "ref_ff,ref_ff_refit" + (",ref_nrms,ref_naml,ref_nrms_refit,ref_l0,ref_l1" if REF_ALL else "")
    B_STAGES = "controls,pooling,moredata,usermodel,histquery,llm_user,rerank,augment,graphrag_llm"
if FOCUS:                                              # only what is new: moredata, multi-interest, the Fastformer reference
    A1, A2 = "controls,pooling,moredata", "controls,pooling,moredata,usermodel"
    B_STAGES = "controls,pooling,moredata,usermodel"
A_ARGS, B_ARGS = [*COMMON, *A_BASE, *(A_QUICK if QUICK else [])], [*COMMON, *B_BASE, *(B_QUICK if QUICK else [])]
if FOCUS:                                              # the user-model views that exist without the old stages
    A_ARGS += ["--um-views", "frozen,text", "--um-extra", "ff:frozen,mi1:frozen,mi4:frozen,mi4d:frozen,mi4:text"]
    B_ARGS += ["--um-views", "frozen,text", "--um-extra", "ff:frozen,mi1:frozen,mi4:frozen,mi4d:frozen"]


### 5. Run A — phần lõi (H1, H2-head, H3, H4, H7, H8, H11 k-means interest, F `moredata`)

In [ ]:
suite.EQUIV_REFERENCE = reference_encoder(ENCODER_A, 256)
S = suite.main([*A_ARGS, "--stages", A1])
gc.collect(); torch.cuda.empty_cache()


### 6. Run R — baseline Fastformer theo công thức chuẩn: 2 seed + refit (cùng thư mục `bge`; `V11_REF_ALL=1` thêm NRMS / NAML / thang ablation)

Mỗi mô hình là một stage; nếu GloVe không tải được, mô hình vẫn chạy với vector ngẫu nhiên và mang hậu tố `_noglove`.

In [ ]:
if RUN_REF:
    try:
        suite.EQUIV_REFERENCE = None                    # embedding đã kiểm ở run A
        S = suite.main([*A_ARGS, "--stages", REF_STAGES])
    except Exception as e:
        import traceback; traceback.print_exc()
        print("!! RUN R FAILED:", repr(e)[:300], "-> continuing with the remaining runs")
    gc.collect(); torch.cuda.empty_cache()


### 7. Run B — LLM embedding (pooling, moredata, user model rẻ chạy trước) rồi LLM sinh văn bản (H5 tower causal/bidir/soft, H9 judge, H6 augment, H4b GraphRAG-LLM, …)

In [ ]:
SB = None
if RUN_B:
    try:
        suite.EQUIV_REFERENCE = reference_encoder(ENCODER_B, 256)
        SB = suite.main([*B_ARGS, "--stages", B_STAGES])
    except Exception as e:                       # e.g. model download failed: keep the A results and still run the rest
        import traceback; traceback.print_exc()
        print("!! RUN B FAILED:", repr(e)[:300], "-> continuing with the remaining runs")
    gc.collect(); torch.cuda.empty_cache()


### 8. Run A — phần còn lại (LoRA/DoRA, user model học được + reader Fastformer / multi-interest, quét encoder); resume từ state.pkl

In [ ]:
suite.EQUIV_REFERENCE = reference_encoder(ENCODER_A, 256)
try:
    S = suite.main([*A_ARGS, "--stages", A2])
except Exception as e:
    import traceback; traceback.print_exc()
    print("!! RUN A (second part) FAILED:", repr(e)[:300], "-> the results of the first part are still in", WORK + "/bge")
gc.collect(); torch.cuda.empty_cache()


### 9. Leaderboard chung (control toàn cục = BGE-small frozen, đúng V10, có cả baseline) và forest plot; bảng thứ hai là các biến thể gộp với popularity (control = `frozen+pop` = V10 `bge_zs_pop`)

In [ ]:
runs, tags = [S] + ([SB] if SB is not None else []), ["bge-small"] + (["qwen3-0.6b"] if SB is not None else [])
rows = suite.leaderboard(runs, tags, ("bge-small", "frozen"), WORK + "/leaderboard.md", plot=WORK + "/forest.png")
rows_pop = suite.leaderboard(runs, tags, ("bge-small", "frozen+pop"), WORK + "/leaderboard_pop.md", plot=WORK + "/forest_pop.png", pop=True)
Q, B = "qwen3-0.6b", "bge-small"
pairs = [# --- H8 / H11: where do k-means interests sit between the mean-pool control and late interaction?
         ((B, "pool_lse"), (B, "frozen")), ((B, "pool_km2"), (B, "pool_lse")), ((B, "pool_km3"), (B, "pool_lse")), ((B, "pool_km5"), (B, "pool_lse")),
         ((B, "pool_km3"), (B, "frozen")), ((Q, "pool_km3"), (Q, "pool_lse")),
         # --- F: what do the extra fields add, under each pooling rule? (txt_t = title only: the same input as the Fastformer reference)
         ((B, "txt_t"), (B, "frozen")), ((B, "txt_cta"), (B, "frozen")), ((B, "txt_ctae"), (B, "txt_cta")), ((B, "txt_views"), (B, "txt_cta")),
         ((B, "txt_t_lse"), (B, "pool_lse")), ((B, "txt_cta_lse"), (B, "pool_lse")), ((B, "txt_ctae_lse"), (B, "txt_cta_lse")), ((B, "txt_views_lse"), (B, "txt_cta_lse")),
         ((B, "hist100_lse"), (B, "pool_lse")), ((B, "hist200_lse"), (B, "pool_lse")), ((B, "hist100_mean"), (B, "frozen")),
         ((Q, "txt_t"), (Q, "frozen")), ((Q, "txt_cta"), (Q, "frozen")), ((Q, "txt_t_lse"), (Q, "pool_lse")), ((Q, "txt_cta_lse"), (Q, "pool_lse")),
         ((Q, "pool_lse"), (Q, "frozen")),
         # --- the trained Fastformer reference (2 seeds; the refit saw the validation days too, like the published numbers)
         ((B, "fastformer_ref_refit"), (B, "fastformer_ref")), ((B, "frozen"), (B, "fastformer_ref")), ((B, "pool_lse"), (B, "fastformer_ref")),
         ((B, "txt_t_lse"), (B, "fastformer_ref")), ((B, "txt_cta_lse"), (B, "fastformer_ref")),
         ((B, "pool_lse"), (B, "fastformer_ref_refit")), ((B, "txt_t_lse"), (B, "fastformer_ref_refit")), ((B, "txt_cta_lse"), (B, "fastformer_ref_refit")),
         ((B, "fastformer_ref+pop"), (B, "frozen+pop")), ((B, "pool_lse+pop"), (B, "fastformer_ref+pop")), ((B, "txt_cta_lse+pop"), (B, "pool_lse+pop")),
         ((B, "um_frozen"), (B, "fastformer_ref")), ((B, "um_ff_frozen"), (B, "fastformer_ref")),
         # --- H11 learned readers: single vector (mi1) vs several interests (mi4, mi4d) vs the V10 reader vs the best zero-shot rule
         ((B, "um_mi1_frozen"), (B, "um_frozen")), ((B, "um_mi4_frozen"), (B, "um_mi1_frozen")), ((B, "um_mi4d_frozen"), (B, "um_mi4_frozen")),
         ((B, "um_mi4_frozen"), (B, "um_frozen")), ((B, "um_mi4_frozen"), (B, "um_ff_frozen")), ((B, "um_mi4_frozen"), (B, "pool_lse")),
         ((B, "um_mi4_frozen"), (B, "fastformer_ref")), ((B, "um_mi4_frozen"), (B, "pool_km3")),
         ((B, "um_text"), (B, "um_frozen")), ((B, "um_mi4_text"), (B, "um_mi4_frozen")), ((B, "um_mi4_text"), (B, "pool_lse")),
         ((B, "pool_lse"), (B, "um_frozen")), ((B, "um_ff_frozen"), (B, "um_frozen")), ((B, "um_ff_knn_rag"), (B, "um_knn_rag")),
         ((B, "um_knn_rag"), (B, "pool_lse")),
         ((Q, "um_mi1_frozen"), (Q, "um_frozen")), ((Q, "um_mi4_frozen"), (Q, "um_mi1_frozen")), ((Q, "um_mi4d_frozen"), (Q, "um_mi4_frozen")),
         ((Q, "um_mi4_frozen"), (Q, "pool_lse")), ((Q, "um_ff_frozen"), (Q, "um_frozen")), ((Q, "um_text"), (Q, "um_frozen")),
         # --- H5 tower and the rest, carried over from v7
         ((Q, "ut_causal+meanpool"), (B, "pool_lse")), ((Q, "ut_causal"), (B, "ut_native")), ((Q, "ut_causal+meanpool"), (B, "ut_native+meanpool")),
         ((Q, "ut_bidir"), (Q, "ut_causal")), ((Q, "ut_soft"), (Q, "ut_causal")), ((Q, "ut_bidir"), (Q, "ut_soft")),
         ((Q, "ut_causal@1"), (Q, "ut_causal")),                                   # same recipe, different seed: the training-noise floor of the tower
         ((Q, "ut_causal+meanpool"), (B, "um_knn_rag")), ((Q, "ut_causal+meanpool"), (B, "fastformer_ref")), ((Q, "ut_causal+meanpool"), (B, "fastformer_ref_refit")),
         ((Q, "ut_causal+meanpool+pop"), (B, "pool_lse+pop")), ((Q, "ut_causal+meanpool+pop"), (B, "frozen+pop")), ((B, "pool_lse+pop"), (B, "frozen+pop")),
         ((Q, "ut_causal+meanpool+pop"), (B, "fastformer_ref+pop")),
         # --- only with V11_REF_ALL=1 (rows missing otherwise are skipped): the v7 baselines
         ((B, "nrms_ref"), (B, "ladder0_v10recipe")), ((B, "nrms_ref@1"), (B, "nrms_ref")), ((B, "nrms_ref_refit"), (B, "nrms_ref")),
         ((B, "naml_ref"), (B, "nrms_ref")), ((B, "fastformer_ref"), (B, "nrms_ref")), ((B, "pool_lse"), (B, "nrms_ref_refit")), ((B, "pool_lse"), (B, "naml_ref"))]
direct = suite.cross_compare(runs, tags, pairs, WORK + "/direct_comparisons.md")


### 10. Kết quả đầy đủ và file gộp để gửi lại

In [ ]:
for sub in ("bge", "qwen3"):
    p = os.path.join(WORK, sub, "results.md")
    if os.path.exists(p):
        print("=" * 30, sub, "=" * 30)
        print(open(p, encoding="utf-8").read())
lad = os.path.join(WORK, "bge", "ladder.md")
if os.path.exists(lad):
    print(open(lad, encoding="utf-8").read())
# seed noise of every trained row (a gap between two trained rows smaller than this is not a finding) and the multi-interest collapse diagnostics
import json
noise = ["# Seed-to-seed spread of the trained rows (test nDCG@10 of each seed; the paired CIs in the other tables do NOT include it)", "",
         "| run | row | seeds | nDCG@10 per seed | std | mean cosine between interests | attention TV between interests |", "|---|---|---|---|---|---|---|"]
for sub in ("bge", "qwen3"):
    p = os.path.join(WORK, sub, "results.json")
    if os.path.exists(p):
        for r in json.load(open(p, encoding="utf-8")):
            if r.get("ndcg10_per_seed") and len(r["ndcg10_per_seed"]) > 1:
                f = lambda v: "-" if not v else ", ".join("-" if x is None else f"{x:.2f}" for x in v)
                noise.append(f"| {sub} | {r['name']} | {len(r['ndcg10_per_seed'])} | {', '.join(f'{x:.4f}' for x in r['ndcg10_per_seed'])} | {r.get('ndcg10_std', float('nan')):.4f} | {f(r.get('interest_cos'))} | {f(r.get('attn_tv'))} |")
open(os.path.join(WORK, "noise.md"), "w", encoding="utf-8").write("\n".join(noise) + "\n")
print("\n".join(noise))
try:
    audit = S.kv.get("data_audit")
    if audit:
        json.dump(audit, open(os.path.join(WORK, "bge", "data_audit.json"), "w"), indent=1)
        print("data audit:", audit)
except Exception as e:
    print("no data audit:", repr(e)[:80])
print("tổng thời gian: %.2f h" % (common.CLOCK.elapsed() / 3600))
print("files:", sorted(os.listdir(WORK)))
# one file with every table needed to read the run (attach it, plus the log if something failed)
parts = [("leaderboard.md", WORK), ("leaderboard_pop.md", WORK), ("direct_comparisons.md", WORK), ("noise.md", WORK), ("data_audit.json", WORK + "/bge"), ("ladder.md", WORK + "/bge"),
         ("results.md", WORK + "/bge"), ("results.md", WORK + "/qwen3"), ("timings.json", WORK + "/bge"), ("timings.json", WORK + "/qwen3")]
with open(os.path.join(WORK, "SUMMARY_v13.md"), "w", encoding="utf-8") as out:
    for fn, d in parts:
        p = os.path.join(d, fn)
        if os.path.exists(p):
            out.write(f"\n\n## {os.path.relpath(p, WORK)}\n\n" + open(p, encoding="utf-8").read())
print("-> gửi lại file:", os.path.join(WORK, "SUMMARY_v13.md"), f"({os.path.getsize(os.path.join(WORK, 'SUMMARY_v13.md')) / 1024:.0f} KB)")


### Cách đọc kết quả → hướng luận văn

**Bước 0 — kiểm tra bộ đo trước khi tin bất cứ hàng nào:** `frozen` khớp V10 (MATCH), `popularity` và `frozen+pop` khớp V10; `um_frozen` trong 0,01 của `llmenc_ca` của V10; `fastformer_ref` nằm gần số v7 đã đo (AUC ≈ 0,6455 / nDCG@10 ≈ 0,3926 với 1 seed, sai khác giữa hai seed cỡ 0,007) và dưới số bài báo (Fastformer ≈ 0,66 / 0,41; con số lấy từ kết quả tìm kiếm, hãy đối chiếu PDF trước khi trích dẫn);
`fastformer_ref_refit` mới là số cùng điều kiện huấn luyện với bài báo (toàn bộ ngày train). Nếu `fastformer_ref` thấp hơn nhiều, đọc nhật ký epoch và xem GloVe có được dùng không (hậu tố `_noglove`).

| Kết quả (CI loại trừ 0 trên test, sau Bonferroni nếu là đại diện hướng) | Ý nghĩa |
|---|---|
| `txt_t` ≈ `frozen`, `txt_t_lse` ≈ `pool_lse` | abstract không thêm gì khi chỉ có title đã đủ (hoặc ngược lại: `frozen` > `txt_t` nghĩa là abstract giúp — đọc dấu của Δ) |
| `txt_cta*` > `frozen` / `pool_lse` | **category giúp**: tín hiệu chủ đề tường minh mà vector title+abstract chưa nén hết; kiểm tra thêm ở cột cold AUC (bài mới có category nhưng chưa có click) |
| `txt_ctae*` ≈ `txt_cta*` | tên entity đã nằm trong title/abstract nên không thêm thông tin |
| `txt_views*` > `txt_cta*` | tách title / abstract / category thành các view có trọng số tốt hơn nhét chung một chuỗi |
| `hist100_*`, `hist200_*` ≈ `frozen` / `pool_lse` | click cũ hơn 50 không thêm gì (nhiều người đọc có ít hơn 50 click; xem `data_audit`) |
| `pool_km{K}` > `pool_lse` | gom click thành vài interest tốt hơn coi mỗi click là một interest; nếu < `pool_lse`, soft-max trên từng click đã là multi-interest tốt nhất |
| `um_mi4_*` > `um_mi1_*` (CI loại trừ 0 **và** hơn nhiễu seed ở `noise.md`) | **nhiều interest hơn một vector** trong reader học được |
| `um_mi4_*` ≈ `um_mi1_*` và `attention TV` ≈ 0 / `interest cosine` ≈ 1 | các interest đã sập về một: kết luận chưa đủ, thử `um_mi4d_*` (có regulariser) |
| `um_mi4_*` ≈ `um_mi1_*` và các interest khác nhau rõ | kết luận âm sạch: nhiều interest không giúp trên MIND-small với vector đóng băng |
| `um_mi4_*` hoặc `pool_km*` **thắng `fastformer_ref_refit`** | claim "vượt baseline huấn luyện chuẩn, cùng điều kiện" (nhưng nhớ: Fastformer chỉ đọc title) |
| `*_lse` / `um_*` chỉ thắng `fastformer_ref` (không refit) | chưa đủ: baseline thiệt thòi vì không được huấn luyện trên các ngày validation |
| `fastformer_ref`: hai seed chênh X | chênh nhỏ hơn X giữa hai mô hình **huấn luyện** không phải phát hiện (CI ghép cặp chỉ tính nhiễu mẫu test) |
| Thắng ở bảng *content only* nhưng **không** thắng ở bảng *with popularity* | kỹ thuật chỉ bù chỗ popularity đã làm — hợp cho **cold-start**, không hợp cho xếp hạng chung |
| H1–H10 | đọc như v7: `RESULTS_v7.md` ghi số đo và kết luận (late-interaction pooling là kết quả sạch nhất; RAG/GraphRAG/Semantic ID/judge/augment ≤ 0,005 hoặc âm; mask của tower chưa kết luận được vì nhiễu seed) |
| Không hướng mới nào | **Kết quả âm có giá trị**: bản v7 đã gần trần của vector đóng băng trên MIND-small ⇒ MIND-large / EB-NeRD (bộ có body thật) |

Lưu ý: dòng chạy trên **tập con** (cột `n test` nhỏ hơn 73k) so sánh với control **trên đúng các impression đó** (cột `ref nDCG@10`); con số tuyệt đối của chúng không so trực tiếp với dòng chạy toàn bộ.
Hàng `um_*` trung bình 3 seed (view `frozen`, `text`) hoặc 2 seed (các view khác); `fastformer_ref` trung bình 2 seed. Baseline huấn luyện trên `train_core` (≈ 80,7% số impression của MINDsmall_train; 10% ngày cuối giữ làm validation) và chọn epoch trên validation;
`fastformer_ref_refit` huấn luyện lại trên cả validation với số epoch trung bình đã chọn (test vẫn chỉ chạm một lần; cột validation của hàng này lặp lại lượt chọn epoch, chỉ cột test có ý nghĩa).